# AttnRes VLM Ablation (Colab / CUDA)

Self-contained exploratory study of Full and Block Attention Residuals on a tiny modular VLM.

**Run all** on a Colab GPU runtime. Persistent artifacts live under the current account's `MyDrive/SharedColab/attnres_vlm_ablation`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
SHARED_COLAB_ROOT = DRIVE_ROOT / "SharedColab"
PROJECT_ROOT = SHARED_COLAB_ROOT / "attnres_vlm_ablation"

SHARED_COLAB_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

for name in [
    "source",
    "configs",
    "checkpoints",
    "runs",
    "logs",
    "metrics",
    "summaries",
    "plots",
    "tables",
    "cache",
    "manifests",
]:
    (PROJECT_ROOT / name).mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# Install dependencies (CUDA PyTorch is provided by Colab).
%pip install -q PyYAML numpy pandas matplotlib seaborn tqdm wandb


## Weights & Biases login


In [ ]:
import os
import wandb

# Prefer an existing env var / Colab secret. Do not hardcode API keys in the notebook.
try:
    from google.colab import userdata

    if not os.environ.get("WANDB_API_KEY"):
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

WANDB_ENTITY = "atin5551-uc-davis"
WANDB_PROJECT = "attnres-next-scale-vlm"
WANDB_ENABLED = True
WANDB_MODE = "auto"  # auto | online | offline | disabled
WANDB_RESUME = "allow"  # allow | must | never

wandb.login()
print("W&B project:", WANDB_PROJECT)
print("W&B entity:", WANDB_ENTITY)


## Configuration


In [ ]:
RUN_MODE = "quick"  # "smoke", "quick", or "full"

RESUME = True
FORCE_RESTART = False

RUN_PRIMARY_FULL_GRID = True
RUN_BLOCK_GRID = True

SEEDS = [0, 1, 2]

PRIMARY_VARIANTS = [
    "baseline",
    "encoder_full",
    "decoder_full",
    "both_full",
]

BLOCK_VARIANTS = [
    "encoder_block",
    "decoder_block",
    "both_block",
]

# Training / model knobs
BATCH_SIZE = 64
NUM_WORKERS = 2
CHECKPOINT_INTERVAL = 100
EVALUATION_INTERVAL = 1
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
MIXED_PRECISION = True
AMP_DTYPE = "auto"  # "auto" | "float16" | "bfloat16"

VISION_D_MODEL = 128
VISION_N_LAYERS = 6
VISION_N_HEADS = 4
VISION_D_FF = 512
DECODER_D_MODEL = 128
DECODER_N_LAYERS = 6
DECODER_N_HEADS = 4
DECODER_D_FF = 512
NUM_BLOCKS = 3
DROPOUT = 0.0

print("RUN_MODE:", RUN_MODE)
print("PRIMARY_VARIANTS:", PRIMARY_VARIANTS)
print("BLOCK_VARIANTS:", BLOCK_VARIANTS)
print("SEEDS:", SEEDS)
print("WANDB_PROJECT:", WANDB_PROJECT)


## Bootstrap canonical source into Google Drive


In [ ]:
import hashlib
import json
import sys
from pathlib import Path

SOURCE_FILES = json.loads('{"src/__init__.py": "\\"\\"\\"AttnResGPT next-scale experiment package.\\"\\"\\"\\n", "src/models/__init__.py": "\\"\\"\\"Model definitions for baseline GPT and AttnRes GPT.\\"\\"\\"\\n", "src/metrics/__init__.py": "\\"\\"\\"Metrics and diagnostic helpers.\\"\\"\\"\\n", "src/utils/__init__.py": "\\"\\"\\"Configuration, logging, and runtime utilities.\\"\\"\\"\\n", "src/vlm/__init__.py": "\\"\\"\\"Package marker.\\"\\"\\"\\n\\n", "src/models/baseline.py": "from __future__ import annotations\\n\\nimport math\\nfrom typing import Any, Optional\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom src.utils.config import ModelConfig\\n\\n\\nclass RMSNorm(nn.Module):\\n    def __init__(self, dim: int, eps: float = 1e-5) -> None:\\n        super().__init__()\\n        self.weight = nn.Parameter(torch.ones(dim))\\n        self.eps = eps\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        norm = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)\\n        return x * norm * self.weight\\n\\n\\nclass CausalSelfAttention(nn.Module):\\n    def __init__(\\n        self,\\n        d_model: int,\\n        n_heads: int,\\n        dropout: float,\\n        *,\\n        bias: bool = True,\\n        max_seq_len: int = 2048,\\n    ) -> None:\\n        super().__init__()\\n        if d_model % n_heads != 0:\\n            raise ValueError(\\"d_model must be divisible by n_heads\\")\\n        self.d_model = d_model\\n        self.n_heads = n_heads\\n        self.head_dim = d_model // n_heads\\n        self.dropout = dropout\\n        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=bias)\\n        self.out_proj = nn.Linear(d_model, d_model, bias=bias)\\n        self.attn_dropout = nn.Dropout(dropout)\\n        self.resid_dropout = nn.Dropout(dropout)\\n        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))\\n        self.register_buffer(\\"causal_mask\\", mask, persistent=False)\\n\\n    def forward(\\n        self,\\n        x: torch.Tensor,\\n        *,\\n        return_attention: bool = False,\\n    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:\\n        batch_size, seq_len, _ = x.shape\\n        qkv = self.qkv_proj(x)\\n        q, k, v = qkv.chunk(3, dim=-1)\\n\\n        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)\\n        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)\\n        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)\\n\\n        if return_attention or not hasattr(F, \\"scaled_dot_product_attention\\"):\\n            scale = 1.0 / math.sqrt(self.head_dim)\\n            scores = torch.matmul(q, k.transpose(-2, -1)) * scale\\n            causal = self.causal_mask[:seq_len, :seq_len]\\n            scores = scores.masked_fill(~causal, torch.finfo(scores.dtype).min)\\n            attn = torch.softmax(scores, dim=-1)\\n            attn = self.attn_dropout(attn)\\n            output = torch.matmul(attn, v)\\n            attn_summary = attn.mean(dim=1)\\n        else:\\n            output = F.scaled_dot_product_attention(\\n                q,\\n                k,\\n                v,\\n                dropout_p=self.dropout if self.training else 0.0,\\n                is_causal=True,\\n            )\\n            attn_summary = None\\n\\n        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)\\n        output = self.resid_dropout(self.out_proj(output))\\n        return output, attn_summary\\n\\n\\nclass BidirectionalSelfAttention(nn.Module):\\n    \\"\\"\\"Non-causal self-attention for vision encoders.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        d_model: int,\\n        n_heads: int,\\n        dropout: float,\\n        *,\\n        bias: bool = True,\\n    ) -> None:\\n        super().__init__()\\n        if d_model % n_heads != 0:\\n            raise ValueError(\\"d_model must be divisible by n_heads\\")\\n        self.d_model = d_model\\n        self.n_heads = n_heads\\n        self.head_dim = d_model // n_heads\\n        self.dropout = dropout\\n        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=bias)\\n        self.out_proj = nn.Linear(d_model, d_model, bias=bias)\\n        self.attn_dropout = nn.Dropout(dropout)\\n        self.resid_dropout = nn.Dropout(dropout)\\n\\n    def forward(\\n        self,\\n        x: torch.Tensor,\\n        *,\\n        return_attention: bool = False,\\n    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:\\n        batch_size, seq_len, _ = x.shape\\n        qkv = self.qkv_proj(x)\\n        q, k, v = qkv.chunk(3, dim=-1)\\n\\n        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)\\n        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)\\n        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)\\n\\n        if return_attention or not hasattr(F, \\"scaled_dot_product_attention\\"):\\n            scale = 1.0 / math.sqrt(self.head_dim)\\n            scores = torch.matmul(q, k.transpose(-2, -1)) * scale\\n            attn = torch.softmax(scores, dim=-1)\\n            attn = self.attn_dropout(attn)\\n            output = torch.matmul(attn, v)\\n            attn_summary = attn.mean(dim=1)\\n        else:\\n            output = F.scaled_dot_product_attention(\\n                q,\\n                k,\\n                v,\\n                dropout_p=self.dropout if self.training else 0.0,\\n                is_causal=False,\\n            )\\n            attn_summary = None\\n\\n        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)\\n        output = self.resid_dropout(self.out_proj(output))\\n        return output, attn_summary\\n\\n\\nclass FeedForward(nn.Module):\\n    def __init__(self, d_model: int, d_ff: int, dropout: float, *, bias: bool = True) -> None:\\n        super().__init__()\\n        self.fc_in = nn.Linear(d_model, d_ff, bias=bias)\\n        self.activation = nn.GELU(approximate=\\"tanh\\")\\n        self.fc_out = nn.Linear(d_ff, d_model, bias=bias)\\n        self.dropout = nn.Dropout(dropout)\\n\\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\\n        x = self.fc_in(x)\\n        x = self.activation(x)\\n        x = self.dropout(x)\\n        x = self.fc_out(x)\\n        return self.dropout(x)\\n\\n\\nclass BaselineTransformerBlock(nn.Module):\\n    def __init__(self, config: ModelConfig, layer_index: int) -> None:\\n        super().__init__()\\n        self.layer_index = layer_index\\n        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.attn = CausalSelfAttention(\\n            config.d_model,\\n            config.n_heads,\\n            config.dropout,\\n            bias=config.bias,\\n            max_seq_len=config.max_seq_len,\\n        )\\n        self.mlp_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.mlp = FeedForward(config.d_model, config.d_ff, config.dropout, bias=config.bias)\\n\\n    def forward(self, x: torch.Tensor, *, return_aux: bool = False) -> tuple[torch.Tensor, dict[str, Any]]:\\n        attn_out, _ = self.attn(self.attn_norm(x))\\n        x = x + attn_out\\n        mlp_out = self.mlp(self.mlp_norm(x))\\n        x = x + mlp_out\\n\\n        return x, {}\\n\\n\\nclass GPTBaseline(nn.Module):\\n    def __init__(self, config: ModelConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)\\n        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)\\n        self.dropout = nn.Dropout(config.dropout)\\n        self.blocks = nn.ModuleList(\\n            [BaselineTransformerBlock(config, layer_index=index) for index in range(config.n_layers)]\\n        )\\n        self.final_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)\\n\\n        self.apply(self._init_weights)\\n        if config.tie_weights:\\n            self.lm_head.weight = self.token_embedding.weight\\n\\n    @property\\n    def num_sublayers(self) -> int:\\n        return 2 * self.config.n_layers\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Embedding):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n\\n    def forward(\\n        self,\\n        input_ids: torch.Tensor | None,\\n        *,\\n        return_aux: bool = False,\\n        prefix_embeddings: torch.Tensor | None = None,\\n    ) -> tuple[torch.Tensor, dict[str, Any]]:\\n        if input_ids is None and prefix_embeddings is None:\\n            raise ValueError(\\"Provide input_ids, prefix_embeddings, or both\\")\\n        text_len = 0 if input_ids is None else input_ids.size(1)\\n        prefix_len = 0 if prefix_embeddings is None else prefix_embeddings.size(1)\\n        seq_len = prefix_len + text_len\\n        if seq_len > self.config.max_seq_len:\\n            raise ValueError(\\"Input sequence is longer than model.max_seq_len\\")\\n        device = prefix_embeddings.device if prefix_embeddings is not None else input_ids.device\\n        positions = torch.arange(seq_len, device=device)\\n        position_embeddings = self.position_embedding(positions)[None, :, :]\\n        token_embeddings = self.token_embedding(input_ids) if input_ids is not None else None\\n        if prefix_embeddings is None:\\n            x = token_embeddings\\n        elif token_embeddings is None:\\n            x = prefix_embeddings\\n        else:\\n            x = torch.cat([prefix_embeddings, token_embeddings], dim=1)\\n        x = x + position_embeddings\\n        x = self.dropout(x)\\n\\n        for block in self.blocks:\\n            x, _ = block(x, return_aux=return_aux)\\n\\n        x = self.final_norm(x)\\n        logits = self.lm_head(x)\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention_rows\\": [],\\n                \\"depth_source_indices\\": [],\\n            }\\n        return logits, aux\\n", "src/models/attnres.py": "from __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\n\\nfrom src.metrics.depth_metrics import contribution_breakdown\\nfrom src.models.baseline import CausalSelfAttention, FeedForward, GPTBaseline, RMSNorm\\nfrom src.utils.config import ModelConfig\\n\\n\\nclass DepthAttentionResidual(nn.Module):\\n    \\"\\"\\"Depth-wise softmax aggregation over previous states.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        d_model: int,\\n        *,\\n        temperature: float = 1.0,\\n        window_size: int | None = None,\\n        rmsnorm_keys: bool = True,\\n        zero_init_query: bool = True,\\n        include_embedding: bool = True,\\n        keep_embedding_in_window: bool = True,\\n        eps: float = 1e-5,\\n    ) -> None:\\n        super().__init__()\\n        self.temperature = temperature\\n        self.window_size = window_size\\n        self.include_embedding = include_embedding\\n        self.keep_embedding_in_window = keep_embedding_in_window\\n        self.query = nn.Parameter(torch.empty(d_model))\\n        self.key_norm = RMSNorm(d_model, eps=eps) if rmsnorm_keys else nn.Identity()\\n        self.capture_weights = False\\n        self.last_weights: torch.Tensor | None = None\\n        self.last_source_indices: list[int] = []\\n        if zero_init_query:\\n            nn.init.zeros_(self.query)\\n        else:\\n            nn.init.normal_(self.query, mean=0.0, std=0.02)\\n\\n    def _select_history(self, history: list[torch.Tensor]) -> tuple[list[torch.Tensor], list[int]]:\\n        if not history:\\n            raise ValueError(\\"history must contain at least one tensor\\")\\n        indices = list(range(len(history)))\\n        selected_history = history\\n        if not self.include_embedding and indices:\\n            selected_history = selected_history[1:]\\n            indices = indices[1:]\\n        if self.window_size is None or len(selected_history) <= self.window_size + int(self.keep_embedding_in_window):\\n            return selected_history, indices\\n        if self.keep_embedding_in_window and indices:\\n            return [selected_history[0], *selected_history[-self.window_size :]], [indices[0], *indices[-self.window_size :]]\\n        return selected_history[-self.window_size :], indices[-self.window_size :]\\n\\n    def forward(self, history: list[torch.Tensor], *, return_stats: bool = False) -> tuple[torch.Tensor, dict[str, Any]]:\\n        selected_history, selected_indices = self._select_history(history)\\n        values = torch.stack(selected_history, dim=0)\\n        keys = self.key_norm(values)\\n        logits = torch.einsum(\\"d,sbtd->sbt\\", self.query, keys)\\n        logits = logits / max(self.temperature, 1e-6)\\n        weights = torch.softmax(logits, dim=0)\\n        mixed = torch.einsum(\\"sbt,sbtd->btd\\", weights, values)\\n\\n        if self.capture_weights:\\n            self.last_weights = weights.detach().cpu()\\n            self.last_source_indices = list(selected_indices)\\n\\n        stats: dict[str, Any] = {}\\n        if return_stats:\\n            mean_weights = weights.detach().mean(dim=(1, 2)).cpu()\\n            entropy = -(weights.detach() * weights.detach().clamp_min(1e-8).log()).sum(dim=0).mean().cpu()\\n            stats = {\\n                \\"source_indices\\": selected_indices,\\n                \\"mean_weights\\": mean_weights,\\n                \\"entropy\\": float(entropy.item()),\\n            }\\n        return mixed, stats\\n\\n\\nclass AttnResTransformerBlock(nn.Module):\\n    def __init__(self, config: ModelConfig, layer_index: int) -> None:\\n        super().__init__()\\n        self.layer_index = layer_index\\n        attnres = config.attnres\\n        self.pre_attn_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=attnres.window_size,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.pre_mlp_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=attnres.window_size,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.attn = CausalSelfAttention(\\n            config.d_model,\\n            config.n_heads,\\n            config.dropout,\\n            bias=config.bias,\\n            max_seq_len=config.max_seq_len,\\n        )\\n        self.mlp_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.mlp = FeedForward(config.d_model, config.d_ff, config.dropout, bias=config.bias)\\n\\n    def forward(self, history: list[torch.Tensor], *, return_aux: bool = False) -> tuple[list[torch.Tensor], dict[str, Any]]:\\n        attn_input, attn_stats = self.pre_attn_res(history, return_stats=return_aux)\\n        attn_out, _ = self.attn(self.attn_norm(attn_input))\\n        history.append(attn_out)\\n\\n        mlp_input, mlp_stats = self.pre_mlp_res(history, return_stats=return_aux)\\n        mlp_out = self.mlp(self.mlp_norm(mlp_input))\\n        history.append(mlp_out)\\n\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention\\": [\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_attn\\", **attn_stats},\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_mlp\\", **mlp_stats},\\n                ],\\n            }\\n        return history, aux\\n\\n\\nclass GPTAttnRes(nn.Module):\\n    def __init__(self, config: ModelConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)\\n        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)\\n        self.dropout = nn.Dropout(config.dropout)\\n        self.blocks = nn.ModuleList(\\n            [AttnResTransformerBlock(config, layer_index=index) for index in range(config.n_layers)]\\n        )\\n        self.final_residual = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=config.attnres.temperature,\\n            window_size=config.attnres.window_size,\\n            rmsnorm_keys=config.attnres.rmsnorm_keys,\\n            zero_init_query=config.attnres.zero_init_queries,\\n            include_embedding=config.attnres.include_embedding,\\n            keep_embedding_in_window=config.attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.final_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)\\n\\n        self.apply(self._init_weights)\\n        if config.tie_weights:\\n            self.lm_head.weight = self.token_embedding.weight\\n\\n    @property\\n    def num_sublayers(self) -> int:\\n        return 2 * self.config.n_layers\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Embedding):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        for module in self.modules():\\n            if isinstance(module, DepthAttentionResidual):\\n                module.capture_weights = enabled\\n                if not enabled:\\n                    module.last_weights = None\\n                    module.last_source_indices = []\\n\\n    def iter_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        return [module for module in self.modules() if isinstance(module, DepthAttentionResidual)]\\n\\n    def forward(\\n        self,\\n        input_ids: torch.Tensor | None,\\n        *,\\n        return_aux: bool = False,\\n        prefix_embeddings: torch.Tensor | None = None,\\n    ) -> tuple[torch.Tensor, dict[str, Any]]:\\n        if input_ids is None and prefix_embeddings is None:\\n            raise ValueError(\\"Provide input_ids, prefix_embeddings, or both\\")\\n        batch_size = prefix_embeddings.size(0) if prefix_embeddings is not None else input_ids.size(0)\\n        text_len = 0 if input_ids is None else input_ids.size(1)\\n        prefix_len = 0 if prefix_embeddings is None else prefix_embeddings.size(1)\\n        seq_len = prefix_len + text_len\\n        if seq_len > self.config.max_seq_len:\\n            raise ValueError(\\"Input sequence is longer than model.max_seq_len\\")\\n        device = prefix_embeddings.device if prefix_embeddings is not None else input_ids.device\\n        positions = torch.arange(seq_len, device=device)\\n        position_embeddings = self.position_embedding(positions)[None, :, :]\\n        token_embeddings = self.token_embedding(input_ids) if input_ids is not None else None\\n        if prefix_embeddings is None:\\n            x0 = token_embeddings\\n        elif token_embeddings is None:\\n            x0 = prefix_embeddings\\n        else:\\n            x0 = torch.cat([prefix_embeddings, token_embeddings], dim=1)\\n        x0 = x0 + position_embeddings\\n        x0 = self.dropout(x0)\\n        history: list[torch.Tensor] = [x0]\\n\\n        depth_rows: list[torch.Tensor] = []\\n        depth_source_indices: list[list[int]] = []\\n\\n        for block in self.blocks:\\n            history, block_aux = block(history, return_aux=return_aux)\\n            if return_aux:\\n                for row in block_aux[\\"depth_attention\\"]:\\n                    depth_rows.append(row[\\"mean_weights\\"])\\n                    depth_source_indices.append(row[\\"source_indices\\"])\\n\\n        if self.config.attnres.final_readout:\\n            x, final_stats = self.final_residual(history, return_stats=return_aux)\\n            if return_aux:\\n                depth_rows.append(final_stats[\\"mean_weights\\"])\\n                depth_source_indices.append(final_stats[\\"source_indices\\"])\\n        else:\\n            x = history[-1]\\n\\n        x = self.final_norm(x)\\n        logits = self.lm_head(x)\\n\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention_rows\\": depth_rows,\\n                \\"depth_source_indices\\": depth_source_indices,\\n                **contribution_breakdown(depth_rows, depth_source_indices),\\n            }\\n        return logits, aux\\n\\n\\ndef _block_sizes(n_layers: int, num_blocks: int) -> list[int]:\\n    \\"\\"\\"Partition transformer layers into ``num_blocks`` contiguous blocks.\\n\\n    Layers are split as evenly as possible: the first ``n_layers % num_blocks``\\n    blocks get one extra layer, so sizes differ by at most 1. This is a deliberate\\n    departure from the paper\'s \\"remainder in the last block\\" wording, which can\\n    produce a single oversized final block (e.g. 12 layers / 8 blocks as\\n    ``[1,1,1,1,1,1,1,5]``). Counting note: ``num_blocks`` partitions transformer\\n    layers, and each transformer layer is two sublayers (attention + MLP), so the\\n    paper\'s sublayer-level ``block_size`` equals ``2 * (n_layers // num_blocks)``\\n    when the split is even.\\n    \\"\\"\\"\\n    if num_blocks < 1:\\n        raise ValueError(f\\"num_blocks must be >= 1, got {num_blocks}\\")\\n    if n_layers < num_blocks:\\n        raise ValueError(f\\"n_layers ({n_layers}) must be >= num_blocks ({num_blocks})\\")\\n    base, remainder = divmod(n_layers, num_blocks)\\n    return [base + (1 if index < remainder else 0) for index in range(num_blocks)]\\n\\n\\nclass BlockAttnResTransformerBlock(nn.Module):\\n    \\"\\"\\"One transformer layer for Block AttnRes.\\n\\n    Inside a block the residual stream (``partial``) accumulates additively, while\\n    the depth-attention mixers attend only over block-level sources (completed\\n    block summaries plus the current partial). Block resets are handled by the\\n    parent model at block boundaries.\\n    \\"\\"\\"\\n\\n    def __init__(self, config: ModelConfig, layer_index: int) -> None:\\n        super().__init__()\\n        self.layer_index = layer_index\\n        attnres = config.attnres\\n        self.pre_attn_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=None,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.pre_mlp_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=None,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.attn = CausalSelfAttention(\\n            config.d_model,\\n            config.n_heads,\\n            config.dropout,\\n            bias=config.bias,\\n            max_seq_len=config.max_seq_len,\\n        )\\n        self.mlp_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.mlp = FeedForward(config.d_model, config.d_ff, config.dropout, bias=config.bias)\\n\\n    def forward(\\n        self,\\n        blocks: list[torch.Tensor],\\n        partial: torch.Tensor | None,\\n        *,\\n        return_aux: bool = False,\\n    ) -> tuple[torch.Tensor, dict[str, Any]]:\\n        sources = blocks if partial is None else [*blocks, partial]\\n        attn_input, attn_stats = self.pre_attn_res(sources, return_stats=return_aux)\\n        attn_out, _ = self.attn(self.attn_norm(attn_input))\\n        partial = attn_out if partial is None else partial + attn_out\\n\\n        mlp_input, mlp_stats = self.pre_mlp_res([*blocks, partial], return_stats=return_aux)\\n        mlp_out = self.mlp(self.mlp_norm(mlp_input))\\n        partial = partial + mlp_out\\n\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention\\": [\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_attn\\", **attn_stats},\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_mlp\\", **mlp_stats},\\n                ],\\n            }\\n        return partial, aux\\n\\n\\nclass GPTBlockAttnRes(nn.Module):\\n    def __init__(self, config: ModelConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        if config.attnres.num_blocks is None:\\n            raise ValueError(\\"Block AttnRes requires model.attnres.num_blocks to be set\\")\\n        self.block_sizes = _block_sizes(config.n_layers, config.attnres.num_blocks)\\n        boundaries: set[int] = set()\\n        accumulated = 0\\n        for size in self.block_sizes:\\n            accumulated += size\\n            boundaries.add(accumulated - 1)\\n        self.block_boundaries = boundaries\\n\\n        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)\\n        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)\\n        self.dropout = nn.Dropout(config.dropout)\\n        self.blocks = nn.ModuleList(\\n            [BlockAttnResTransformerBlock(config, layer_index=index) for index in range(config.n_layers)]\\n        )\\n        self.final_residual = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=config.attnres.temperature,\\n            window_size=None,\\n            rmsnorm_keys=config.attnres.rmsnorm_keys,\\n            zero_init_query=config.attnres.zero_init_queries,\\n            include_embedding=config.attnres.include_embedding,\\n            keep_embedding_in_window=config.attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.final_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)\\n\\n        self.apply(self._init_weights)\\n        if config.tie_weights:\\n            self.lm_head.weight = self.token_embedding.weight\\n\\n    @property\\n    def num_sublayers(self) -> int:\\n        return 2 * self.config.n_layers\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Embedding):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        for module in self.modules():\\n            if isinstance(module, DepthAttentionResidual):\\n                module.capture_weights = enabled\\n                if not enabled:\\n                    module.last_weights = None\\n                    module.last_source_indices = []\\n\\n    def iter_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        return [module for module in self.modules() if isinstance(module, DepthAttentionResidual)]\\n\\n    def forward(\\n        self,\\n        input_ids: torch.Tensor | None,\\n        *,\\n        return_aux: bool = False,\\n        prefix_embeddings: torch.Tensor | None = None,\\n    ) -> tuple[torch.Tensor, dict[str, Any]]:\\n        if input_ids is None and prefix_embeddings is None:\\n            raise ValueError(\\"Provide input_ids, prefix_embeddings, or both\\")\\n        text_len = 0 if input_ids is None else input_ids.size(1)\\n        prefix_len = 0 if prefix_embeddings is None else prefix_embeddings.size(1)\\n        seq_len = prefix_len + text_len\\n        if seq_len > self.config.max_seq_len:\\n            raise ValueError(\\"Input sequence is longer than model.max_seq_len\\")\\n        device = prefix_embeddings.device if prefix_embeddings is not None else input_ids.device\\n        positions = torch.arange(seq_len, device=device)\\n        position_embeddings = self.position_embedding(positions)[None, :, :]\\n        token_embeddings = self.token_embedding(input_ids) if input_ids is not None else None\\n        if prefix_embeddings is None:\\n            x0 = token_embeddings\\n        elif token_embeddings is None:\\n            x0 = prefix_embeddings\\n        else:\\n            x0 = torch.cat([prefix_embeddings, token_embeddings], dim=1)\\n        x0 = x0 + position_embeddings\\n        x0 = self.dropout(x0)\\n\\n        blocks: list[torch.Tensor] = [x0]\\n        partial: torch.Tensor | None = None\\n\\n        depth_rows: list[torch.Tensor] = []\\n        depth_source_indices: list[list[int]] = []\\n\\n        for layer_index, block in enumerate(self.blocks):\\n            partial, block_aux = block(blocks, partial, return_aux=return_aux)\\n            if return_aux:\\n                for row in block_aux[\\"depth_attention\\"]:\\n                    depth_rows.append(row[\\"mean_weights\\"])\\n                    depth_source_indices.append(row[\\"source_indices\\"])\\n            if layer_index in self.block_boundaries:\\n                blocks.append(partial)\\n                partial = None\\n\\n        if self.config.attnres.final_readout:\\n            x, final_stats = self.final_residual(blocks, return_stats=return_aux)\\n            if return_aux:\\n                depth_rows.append(final_stats[\\"mean_weights\\"])\\n                depth_source_indices.append(final_stats[\\"source_indices\\"])\\n        else:\\n            x = blocks[-1]\\n\\n        x = self.final_norm(x)\\n        logits = self.lm_head(x)\\n\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention_rows\\": depth_rows,\\n                \\"depth_source_indices\\": depth_source_indices,\\n                **contribution_breakdown(depth_rows, depth_source_indices),\\n            }\\n        return logits, aux\\n\\n\\ndef build_model(config: ModelConfig) -> nn.Module:\\n    if config.architecture == \\"baseline\\":\\n        return GPTBaseline(config)\\n    if config.architecture == \\"block_attnres\\" or (\\n        config.architecture == \\"attnres\\" and config.attnres.mode == \\"block\\"\\n    ):\\n        return GPTBlockAttnRes(config)\\n    if config.architecture == \\"attnres\\":\\n        return GPTAttnRes(config)\\n    raise ValueError(f\\"Unsupported architecture: {config.architecture}\\")\\n", "src/models/vision_attnres.py": "from __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Literal\\n\\nimport torch\\nimport torch.nn as nn\\n\\nfrom src.metrics.depth_metrics import contribution_breakdown\\nfrom src.models.attnres import DepthAttentionResidual, _block_sizes\\nfrom src.models.baseline import BidirectionalSelfAttention, FeedForward, RMSNorm\\nfrom src.utils.config import AttnResConfig\\n\\n\\n@dataclass\\nclass VisionConfig:\\n    image_size: int = 64\\n    patch_size: int = 8\\n    in_channels: int = 3\\n    d_model: int = 128\\n    n_layers: int = 6\\n    n_heads: int = 4\\n    d_ff: int = 512\\n    dropout: float = 0.0\\n    bias: bool = True\\n    norm_eps: float = 1e-5\\n    init_std: float = 0.02\\n    residual: Literal[\\"baseline\\", \\"attnres\\", \\"block_attnres\\"] = \\"baseline\\"\\n    attnres: AttnResConfig = field(default_factory=AttnResConfig)\\n\\n    def __post_init__(self) -> None:\\n        if self.image_size % self.patch_size != 0:\\n            raise ValueError(\\"image_size must be divisible by patch_size\\")\\n        if self.d_model % self.n_heads != 0:\\n            raise ValueError(\\"d_model must be divisible by n_heads\\")\\n        if self.residual == \\"block_attnres\\":\\n            self.attnres.enabled = True\\n            self.attnres.mode = \\"block\\"\\n            if self.attnres.num_blocks is None:\\n                self.attnres.num_blocks = max(1, self.n_layers // 2)\\n        elif self.residual == \\"attnres\\":\\n            self.attnres.enabled = True\\n            self.attnres.mode = \\"full\\"\\n\\n    @property\\n    def num_patches(self) -> int:\\n        side = self.image_size // self.patch_size\\n        return side * side\\n\\n\\nclass VisionPatchEmbed(nn.Module):\\n    def __init__(self, config: VisionConfig) -> None:\\n        super().__init__()\\n        self.proj = nn.Conv2d(\\n            config.in_channels,\\n            config.d_model,\\n            kernel_size=config.patch_size,\\n            stride=config.patch_size,\\n        )\\n        self.num_patches = config.num_patches\\n        self.position_embedding = nn.Parameter(torch.zeros(1, self.num_patches, config.d_model))\\n        self.dropout = nn.Dropout(config.dropout)\\n        nn.init.normal_(self.position_embedding, mean=0.0, std=config.init_std)\\n\\n    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:\\n        # [B, C, H, W] -> [B, D, H\', W\'] -> [B, N, D]\\n        x = self.proj(pixel_values)\\n        x = x.flatten(2).transpose(1, 2)\\n        x = x + self.position_embedding\\n        return self.dropout(x)\\n\\n\\nclass BaselineVisionBlock(nn.Module):\\n    def __init__(self, config: VisionConfig, layer_index: int) -> None:\\n        super().__init__()\\n        self.layer_index = layer_index\\n        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.attn = BidirectionalSelfAttention(\\n            config.d_model,\\n            config.n_heads,\\n            config.dropout,\\n            bias=config.bias,\\n        )\\n        self.mlp_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.mlp = FeedForward(config.d_model, config.d_ff, config.dropout, bias=config.bias)\\n\\n    def forward(self, x: torch.Tensor, *, return_aux: bool = False) -> tuple[torch.Tensor, dict[str, Any]]:\\n        attn_delta, _ = self.attn(self.attn_norm(x))\\n        x = x + attn_delta\\n        mlp_delta = self.mlp(self.mlp_norm(x))\\n        x = x + mlp_delta\\n        return x, {}\\n\\n\\nclass AttnResVisionBlock(nn.Module):\\n    def __init__(self, config: VisionConfig, layer_index: int) -> None:\\n        super().__init__()\\n        self.layer_index = layer_index\\n        attnres = config.attnres\\n        self.pre_attn_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=attnres.window_size,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.pre_mlp_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=attnres.window_size,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.attn = BidirectionalSelfAttention(\\n            config.d_model,\\n            config.n_heads,\\n            config.dropout,\\n            bias=config.bias,\\n        )\\n        self.mlp_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.mlp = FeedForward(config.d_model, config.d_ff, config.dropout, bias=config.bias)\\n\\n    def forward(\\n        self,\\n        history: list[torch.Tensor],\\n        *,\\n        return_aux: bool = False,\\n    ) -> tuple[list[torch.Tensor], dict[str, Any]]:\\n        attn_input, attn_stats = self.pre_attn_res(history, return_stats=return_aux)\\n        attn_delta, _ = self.attn(self.attn_norm(attn_input))\\n        history.append(attn_delta)\\n\\n        mlp_input, mlp_stats = self.pre_mlp_res(history, return_stats=return_aux)\\n        mlp_delta = self.mlp(self.mlp_norm(mlp_input))\\n        history.append(mlp_delta)\\n\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention\\": [\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_attn\\", **attn_stats},\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_mlp\\", **mlp_stats},\\n                ],\\n            }\\n        return history, aux\\n\\n\\nclass BlockAttnResVisionBlock(nn.Module):\\n    def __init__(self, config: VisionConfig, layer_index: int) -> None:\\n        super().__init__()\\n        self.layer_index = layer_index\\n        attnres = config.attnres\\n        self.pre_attn_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=None,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.pre_mlp_res = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=attnres.temperature,\\n            window_size=None,\\n            rmsnorm_keys=attnres.rmsnorm_keys,\\n            zero_init_query=attnres.zero_init_queries,\\n            include_embedding=attnres.include_embedding,\\n            keep_embedding_in_window=attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.attn_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.attn = BidirectionalSelfAttention(\\n            config.d_model,\\n            config.n_heads,\\n            config.dropout,\\n            bias=config.bias,\\n        )\\n        self.mlp_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.mlp = FeedForward(config.d_model, config.d_ff, config.dropout, bias=config.bias)\\n\\n    def forward(\\n        self,\\n        blocks: list[torch.Tensor],\\n        partial: torch.Tensor | None,\\n        *,\\n        return_aux: bool = False,\\n    ) -> tuple[torch.Tensor, dict[str, Any]]:\\n        sources = blocks if partial is None else [*blocks, partial]\\n        attn_input, attn_stats = self.pre_attn_res(sources, return_stats=return_aux)\\n        attn_delta, _ = self.attn(self.attn_norm(attn_input))\\n        partial = attn_delta if partial is None else partial + attn_delta\\n\\n        mlp_input, mlp_stats = self.pre_mlp_res([*blocks, partial], return_stats=return_aux)\\n        mlp_delta = self.mlp(self.mlp_norm(mlp_input))\\n        partial = partial + mlp_delta\\n\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention\\": [\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_attn\\", **attn_stats},\\n                    {\\"name\\": f\\"block_{self.layer_index:02d}_pre_mlp\\", **mlp_stats},\\n                ],\\n            }\\n        return partial, aux\\n\\n\\nclass BaselineVisionTransformer(nn.Module):\\n    def __init__(self, config: VisionConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        self.patch_embed = VisionPatchEmbed(config)\\n        self.blocks = nn.ModuleList(\\n            [BaselineVisionBlock(config, layer_index=index) for index in range(config.n_layers)]\\n        )\\n        self.final_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.apply(self._init_weights)\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Conv2d):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        return None\\n\\n    def iter_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        return []\\n\\n    def forward(self, pixel_values: torch.Tensor, *, return_aux: bool = False) -> tuple[torch.Tensor, dict[str, Any]]:\\n        x = self.patch_embed(pixel_values)\\n        for block in self.blocks:\\n            x, _ = block(x, return_aux=return_aux)\\n        x = self.final_norm(x)\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\"depth_attention_rows\\": [], \\"depth_source_indices\\": []}\\n        return x, aux\\n\\n\\nclass AttnResVisionTransformer(nn.Module):\\n    def __init__(self, config: VisionConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        self.patch_embed = VisionPatchEmbed(config)\\n        self.blocks = nn.ModuleList(\\n            [AttnResVisionBlock(config, layer_index=index) for index in range(config.n_layers)]\\n        )\\n        self.final_residual = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=config.attnres.temperature,\\n            window_size=config.attnres.window_size,\\n            rmsnorm_keys=config.attnres.rmsnorm_keys,\\n            zero_init_query=config.attnres.zero_init_queries,\\n            include_embedding=config.attnres.include_embedding,\\n            keep_embedding_in_window=config.attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.final_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.apply(self._init_weights)\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Conv2d):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        for module in self.modules():\\n            if isinstance(module, DepthAttentionResidual):\\n                module.capture_weights = enabled\\n                if not enabled:\\n                    module.last_weights = None\\n                    module.last_source_indices = []\\n\\n    def iter_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        return [module for module in self.modules() if isinstance(module, DepthAttentionResidual)]\\n\\n    def forward(self, pixel_values: torch.Tensor, *, return_aux: bool = False) -> tuple[torch.Tensor, dict[str, Any]]:\\n        history: list[torch.Tensor] = [self.patch_embed(pixel_values)]\\n        depth_rows: list[torch.Tensor] = []\\n        depth_source_indices: list[list[int]] = []\\n\\n        for block in self.blocks:\\n            history, block_aux = block(history, return_aux=return_aux)\\n            if return_aux:\\n                for row in block_aux[\\"depth_attention\\"]:\\n                    depth_rows.append(row[\\"mean_weights\\"])\\n                    depth_source_indices.append(row[\\"source_indices\\"])\\n\\n        if self.config.attnres.final_readout:\\n            x, final_stats = self.final_residual(history, return_stats=return_aux)\\n            if return_aux:\\n                depth_rows.append(final_stats[\\"mean_weights\\"])\\n                depth_source_indices.append(final_stats[\\"source_indices\\"])\\n        else:\\n            x = history[-1]\\n\\n        x = self.final_norm(x)\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention_rows\\": depth_rows,\\n                \\"depth_source_indices\\": depth_source_indices,\\n                **contribution_breakdown(depth_rows, depth_source_indices),\\n            }\\n        return x, aux\\n\\n\\nclass BlockAttnResVisionTransformer(nn.Module):\\n    def __init__(self, config: VisionConfig) -> None:\\n        super().__init__()\\n        self.config = config\\n        if config.attnres.num_blocks is None:\\n            raise ValueError(\\"Block AttnRes vision encoder requires attnres.num_blocks\\")\\n        self.block_sizes = _block_sizes(config.n_layers, config.attnres.num_blocks)\\n        boundaries: set[int] = set()\\n        accumulated = 0\\n        for size in self.block_sizes:\\n            accumulated += size\\n            boundaries.add(accumulated - 1)\\n        self.block_boundaries = boundaries\\n\\n        self.patch_embed = VisionPatchEmbed(config)\\n        self.blocks = nn.ModuleList(\\n            [BlockAttnResVisionBlock(config, layer_index=index) for index in range(config.n_layers)]\\n        )\\n        self.final_residual = DepthAttentionResidual(\\n            config.d_model,\\n            temperature=config.attnres.temperature,\\n            window_size=None,\\n            rmsnorm_keys=config.attnres.rmsnorm_keys,\\n            zero_init_query=config.attnres.zero_init_queries,\\n            include_embedding=config.attnres.include_embedding,\\n            keep_embedding_in_window=config.attnres.keep_embedding_in_window,\\n            eps=config.norm_eps,\\n        )\\n        self.final_norm = RMSNorm(config.d_model, eps=config.norm_eps)\\n        self.apply(self._init_weights)\\n\\n    def _init_weights(self, module: nn.Module) -> None:\\n        if isinstance(module, nn.Linear):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n        elif isinstance(module, nn.Conv2d):\\n            nn.init.normal_(module.weight, mean=0.0, std=self.config.init_std)\\n            if module.bias is not None:\\n                nn.init.zeros_(module.bias)\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        for module in self.modules():\\n            if isinstance(module, DepthAttentionResidual):\\n                module.capture_weights = enabled\\n                if not enabled:\\n                    module.last_weights = None\\n                    module.last_source_indices = []\\n\\n    def iter_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        return [module for module in self.modules() if isinstance(module, DepthAttentionResidual)]\\n\\n    def forward(self, pixel_values: torch.Tensor, *, return_aux: bool = False) -> tuple[torch.Tensor, dict[str, Any]]:\\n        x0 = self.patch_embed(pixel_values)\\n        blocks: list[torch.Tensor] = [x0]\\n        partial: torch.Tensor | None = None\\n        depth_rows: list[torch.Tensor] = []\\n        depth_source_indices: list[list[int]] = []\\n\\n        for layer_index, block in enumerate(self.blocks):\\n            partial, block_aux = block(blocks, partial, return_aux=return_aux)\\n            if return_aux:\\n                for row in block_aux[\\"depth_attention\\"]:\\n                    depth_rows.append(row[\\"mean_weights\\"])\\n                    depth_source_indices.append(row[\\"source_indices\\"])\\n            if layer_index in self.block_boundaries:\\n                blocks.append(partial)\\n                partial = None\\n\\n        if self.config.attnres.final_readout:\\n            x, final_stats = self.final_residual(blocks, return_stats=return_aux)\\n            if return_aux:\\n                depth_rows.append(final_stats[\\"mean_weights\\"])\\n                depth_source_indices.append(final_stats[\\"source_indices\\"])\\n        else:\\n            x = blocks[-1]\\n\\n        x = self.final_norm(x)\\n        aux: dict[str, Any] = {}\\n        if return_aux:\\n            aux = {\\n                \\"depth_attention_rows\\": depth_rows,\\n                \\"depth_source_indices\\": depth_source_indices,\\n                **contribution_breakdown(depth_rows, depth_source_indices),\\n            }\\n        return x, aux\\n\\n\\ndef build_vision_encoder(config: VisionConfig) -> nn.Module:\\n    if config.residual == \\"baseline\\":\\n        return BaselineVisionTransformer(config)\\n    if config.residual == \\"attnres\\":\\n        return AttnResVisionTransformer(config)\\n    if config.residual == \\"block_attnres\\":\\n        return BlockAttnResVisionTransformer(config)\\n    raise ValueError(f\\"Unsupported vision residual: {config.residual}\\")\\n", "src/models/vlm_attnres.py": "from __future__ import annotations\\n\\nfrom contextlib import nullcontext\\nfrom dataclasses import dataclass\\nfrom typing import Any, Sequence\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\ntry:\\n    from transformers import AutoModel, AutoProcessor\\nexcept ImportError:  # pragma: no cover - optional for TinyAttnResVLM-only workflows\\n    AutoModel = None  # type: ignore[assignment,misc]\\n    AutoProcessor = None  # type: ignore[assignment,misc]\\n\\nfrom src.metrics.norms import perplexity_from_loss\\nfrom src.models.attnres import DepthAttentionResidual, GPTAttnRes, GPTBlockAttnRes\\nfrom src.models.baseline import GPTBaseline\\nfrom src.models.vision_attnres import VisionConfig, build_vision_encoder\\nfrom src.utils.config import AttnResConfig, ModelConfig\\n\\n\\ndef masked_language_model_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\\n    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)\\n\\n\\ndef _build_decoder(decoder_config: ModelConfig) -> nn.Module:\\n    if decoder_config.architecture == \\"baseline\\":\\n        return GPTBaseline(decoder_config)\\n    if decoder_config.architecture == \\"block_attnres\\" or (\\n        decoder_config.architecture == \\"attnres\\" and decoder_config.attnres.mode == \\"block\\"\\n    ):\\n        return GPTBlockAttnRes(decoder_config)\\n    if decoder_config.architecture == \\"attnres\\":\\n        return GPTAttnRes(decoder_config)\\n    raise ValueError(f\\"Unsupported decoder architecture: {decoder_config.architecture}\\")\\n\\n\\nclass TinyAttnResVLM(nn.Module):\\n    \\"\\"\\"Trainable tiny ViT encoder + linear connector + existing GPT decoder.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        *,\\n        vision_config: VisionConfig,\\n        decoder_config: ModelConfig,\\n        encoder_residual: str = \\"baseline\\",\\n        decoder_residual: str = \\"baseline\\",\\n    ) -> None:\\n        super().__init__()\\n        vision_cfg = VisionConfig(\\n            image_size=vision_config.image_size,\\n            patch_size=vision_config.patch_size,\\n            in_channels=vision_config.in_channels,\\n            d_model=vision_config.d_model,\\n            n_layers=vision_config.n_layers,\\n            n_heads=vision_config.n_heads,\\n            d_ff=vision_config.d_ff,\\n            dropout=vision_config.dropout,\\n            bias=vision_config.bias,\\n            norm_eps=vision_config.norm_eps,\\n            init_std=vision_config.init_std,\\n            residual=encoder_residual,  # type: ignore[arg-type]\\n            attnres=AttnResConfig(**vision_config.attnres.__dict__),\\n        )\\n\\n        decoder_attnres = AttnResConfig(**decoder_config.attnres.__dict__)\\n        if decoder_residual == \\"block_attnres\\":\\n            decoder_architecture = \\"block_attnres\\"\\n            decoder_attnres.enabled = True\\n            decoder_attnres.mode = \\"block\\"\\n            if decoder_attnres.num_blocks is None:\\n                decoder_attnres.num_blocks = max(1, decoder_config.n_layers // 2)\\n        elif decoder_residual == \\"attnres\\":\\n            decoder_architecture = \\"attnres\\"\\n            decoder_attnres.enabled = True\\n            decoder_attnres.mode = \\"full\\"\\n        else:\\n            decoder_architecture = \\"baseline\\"\\n            decoder_attnres.enabled = False\\n\\n        decoder_cfg = ModelConfig(\\n            architecture=decoder_architecture,\\n            size_name=decoder_config.size_name,\\n            vocab_size=decoder_config.vocab_size,\\n            max_seq_len=decoder_config.max_seq_len,\\n            d_model=decoder_config.d_model,\\n            n_layers=decoder_config.n_layers,\\n            n_heads=decoder_config.n_heads,\\n            d_ff=decoder_config.d_ff,\\n            dropout=decoder_config.dropout,\\n            bias=decoder_config.bias,\\n            tie_weights=decoder_config.tie_weights,\\n            norm_eps=decoder_config.norm_eps,\\n            init_std=decoder_config.init_std,\\n            attnres=decoder_attnres,\\n        )\\n\\n        self.encoder_residual = encoder_residual\\n        self.decoder_residual = decoder_residual\\n        self.vision_config = vision_cfg\\n        self.encoder = build_vision_encoder(vision_cfg)\\n        self.connector = nn.Linear(vision_cfg.d_model, decoder_cfg.d_model)\\n        self.decoder = _build_decoder(decoder_cfg)\\n        nn.init.normal_(self.connector.weight, mean=0.0, std=decoder_cfg.init_std)\\n        if self.connector.bias is not None:\\n            nn.init.zeros_(self.connector.bias)\\n\\n    @property\\n    def decoder_config(self) -> ModelConfig:\\n        return self.decoder.config\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        if hasattr(self.encoder, \\"set_weight_capture\\"):\\n            self.encoder.set_weight_capture(enabled)\\n        if hasattr(self.decoder, \\"set_weight_capture\\"):\\n            self.decoder.set_weight_capture(enabled)\\n\\n    def iter_encoder_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        if hasattr(self.encoder, \\"iter_depth_residuals\\"):\\n            return self.encoder.iter_depth_residuals()\\n        return []\\n\\n    def iter_decoder_depth_residuals(self) -> list[DepthAttentionResidual]:\\n        if hasattr(self.decoder, \\"iter_depth_residuals\\"):\\n            return self.decoder.iter_depth_residuals()\\n        return []\\n\\n    def forward(\\n        self,\\n        *,\\n        pixel_values: torch.Tensor,\\n        input_ids: torch.Tensor,\\n        targets: torch.Tensor | None = None,\\n        return_aux: bool = False,\\n    ) -> dict[str, Any]:\\n        vision_hidden, vision_aux = self.encoder(pixel_values, return_aux=return_aux)\\n        prefix_embeddings = self.connector(vision_hidden.to(dtype=self.connector.weight.dtype))\\n        logits, decoder_aux = self.decoder(\\n            input_ids,\\n            return_aux=return_aux,\\n            prefix_embeddings=prefix_embeddings,\\n        )\\n        prefix_len = prefix_embeddings.size(1)\\n        target_len = input_ids.size(1) if targets is None else targets.size(1)\\n        text_logits = logits[:, prefix_len : prefix_len + target_len, :]\\n        payload: dict[str, Any] = {\\n            \\"logits\\": text_logits,\\n            \\"prefix_length\\": prefix_len,\\n            \\"vision_aux\\": vision_aux if return_aux else {},\\n            \\"decoder_aux\\": decoder_aux if return_aux else {},\\n        }\\n        if targets is not None:\\n            loss = masked_language_model_loss(text_logits, targets)\\n            payload[\\"loss\\"] = loss\\n            payload[\\"perplexity\\"] = perplexity_from_loss(float(loss.item()))\\n        return payload\\n\\n\\n@dataclass\\nclass VisionLanguageAlphaSummary:\\n    vision_rows: list[list[float]]\\n    language_rows: list[list[float]]\\n    source_indices: list[list[int]]\\n    vision_entropy: list[float]\\n    language_entropy: list[float]\\n    vision_embedding: list[float]\\n    language_embedding: list[float]\\n\\n\\nclass SiglipAttnResCaptioner(nn.Module):\\n    def __init__(\\n        self,\\n        *,\\n        vision_model_name: str,\\n        decoder_config: ModelConfig,\\n    ) -> None:\\n        super().__init__()\\n        if AutoProcessor is None or AutoModel is None:\\n            raise ImportError(\\"transformers is required for SiglipAttnResCaptioner\\")\\n        processor = AutoProcessor.from_pretrained(vision_model_name)\\n        vision_backbone = AutoModel.from_pretrained(vision_model_name)\\n        if hasattr(vision_backbone, \\"vision_model\\"):\\n            vision_model = vision_backbone.vision_model\\n        else:\\n            vision_model = vision_backbone\\n\\n        self.processor = processor\\n        self.vision_model_name = vision_model_name\\n        self.vision_encoder = vision_model.eval()\\n        for parameter in self.vision_encoder.parameters():\\n            parameter.requires_grad = False\\n\\n        vision_hidden_size = int(getattr(self.vision_encoder.config, \\"hidden_size\\"))\\n        self.connector = nn.Linear(vision_hidden_size, decoder_config.d_model)\\n        self.decoder = _build_decoder(decoder_config)\\n\\n    @property\\n    def decoder_config(self) -> ModelConfig:\\n        return self.decoder.config\\n\\n    @property\\n    def supports_alpha_analysis(self) -> bool:\\n        return isinstance(self.decoder, GPTAttnRes)\\n\\n    def set_weight_capture(self, enabled: bool) -> None:\\n        if hasattr(self.decoder, \\"set_weight_capture\\"):\\n            self.decoder.set_weight_capture(enabled)\\n\\n    def encode_vision(self, pixel_values: torch.Tensor) -> torch.Tensor:\\n        with torch.no_grad():\\n            outputs = self.vision_encoder(pixel_values=pixel_values)\\n        return outputs.last_hidden_state\\n\\n    def forward(\\n        self,\\n        *,\\n        pixel_values: torch.Tensor | None = None,\\n        vision_hidden: torch.Tensor | None = None,\\n        input_ids: torch.Tensor,\\n        targets: torch.Tensor | None = None,\\n        return_aux: bool = False,\\n    ) -> dict[str, Any]:\\n        if vision_hidden is None:\\n            if pixel_values is None:\\n                raise ValueError(\\"Provide pixel_values or precomputed vision_hidden\\")\\n            vision_hidden = self.encode_vision(pixel_values)\\n        vision_hidden = vision_hidden.to(dtype=self.connector.weight.dtype)\\n        prefix_embeddings = self.connector(vision_hidden)\\n        logits, aux = self.decoder(\\n            input_ids,\\n            return_aux=return_aux,\\n            prefix_embeddings=prefix_embeddings,\\n        )\\n        prefix_len = prefix_embeddings.size(1)\\n        target_len = input_ids.size(1) if targets is None else targets.size(1)\\n        text_logits = logits[:, prefix_len : prefix_len + target_len, :]\\n\\n        payload: dict[str, Any] = {\\n            \\"logits\\": text_logits,\\n            \\"prefix_length\\": prefix_len,\\n            \\"aux\\": aux,\\n        }\\n        if targets is not None:\\n            loss = masked_language_model_loss(text_logits, targets)\\n            payload[\\"loss\\"] = loss\\n            payload[\\"perplexity\\"] = perplexity_from_loss(float(loss.item()))\\n        return payload\\n\\n\\ndef _row_entropy(row: Sequence[float]) -> float:\\n    array = np.asarray(row, dtype=np.float64)\\n    array = np.clip(array, 1e-8, 1.0)\\n    return float(-(array * np.log(array)).sum())\\n\\n\\ndef _embedding_contribution(row: Sequence[float], indices: Sequence[int]) -> float:\\n    if 0 not in indices:\\n        return 0.0\\n    return float(row[list(indices).index(0)])\\n\\n\\n@torch.no_grad()\\ndef summarize_alpha_by_token_type(\\n    model: SiglipAttnResCaptioner,\\n    dataloader: torch.utils.data.DataLoader,\\n    *,\\n    device: torch.device,\\n    max_batches: int | None = None,\\n    mixed_precision: bool = False,\\n    amp_dtype: torch.dtype = torch.float16,\\n) -> VisionLanguageAlphaSummary:\\n    if not model.supports_alpha_analysis:\\n        raise ValueError(\\"Alpha summarization is only available for AttnRes decoders.\\")\\n    model.eval()\\n    model.set_weight_capture(True)\\n\\n    vision_sums: list[torch.Tensor] = []\\n    language_sums: list[torch.Tensor] = []\\n    vision_counts: list[float] = []\\n    language_counts: list[float] = []\\n    source_indices: list[list[int]] = []\\n\\n    for batch_index, batch in enumerate(dataloader):\\n        if max_batches is not None and batch_index >= max_batches:\\n            break\\n\\n        pixel_values = batch.get(\\"pixel_values\\")\\n        input_ids = batch[\\"input_ids\\"].to(device)\\n        text_mask = batch[\\"text_mask\\"].cpu()\\n        vision_hidden = batch.get(\\"vision_hidden\\")\\n        model_kwargs = {\\n            \\"pixel_values\\": pixel_values.to(device) if pixel_values is not None and vision_hidden is None else None,\\n            \\"vision_hidden\\": vision_hidden.to(device) if vision_hidden is not None else None,\\n            \\"input_ids\\": input_ids,\\n            \\"return_aux\\": False,\\n        }\\n        autocast_context = (\\n            torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=True)\\n            if device.type == \\"cuda\\" and mixed_precision\\n            else nullcontext()\\n        )\\n        with autocast_context:\\n            output = model(**model_kwargs)\\n        prefix_len = int(output[\\"prefix_length\\"])\\n\\n        for site_index, residual in enumerate(model.decoder.iter_depth_residuals()):\\n            weights = residual.last_weights\\n            indices = residual.last_source_indices\\n            if weights is None:\\n                continue\\n\\n            seq_total = int(weights.size(2))\\n            batch_size = int(weights.size(1))\\n            vision_mask = torch.zeros((batch_size, seq_total), dtype=torch.float32)\\n            language_mask = torch.zeros((batch_size, seq_total), dtype=torch.float32)\\n            vision_mask[:, :prefix_len] = 1.0\\n            language_width = min(text_mask.size(1), seq_total - prefix_len)\\n            language_mask[:, prefix_len : prefix_len + language_width] = text_mask[:, :language_width].float()\\n\\n            vision_denom = float(vision_mask.sum().item())\\n            language_denom = float(language_mask.sum().item())\\n            vision_row = (weights * vision_mask.unsqueeze(0)).sum(dim=(1, 2))\\n            language_row = (weights * language_mask.unsqueeze(0)).sum(dim=(1, 2))\\n\\n            if len(vision_sums) <= site_index:\\n                vision_sums.append(vision_row.clone())\\n                language_sums.append(language_row.clone())\\n                vision_counts.append(vision_denom)\\n                language_counts.append(language_denom)\\n                source_indices.append(list(indices))\\n            else:\\n                vision_sums[site_index] += vision_row\\n                language_sums[site_index] += language_row\\n                vision_counts[site_index] += vision_denom\\n                language_counts[site_index] += language_denom\\n\\n    model.set_weight_capture(False)\\n\\n    vision_rows: list[list[float]] = []\\n    language_rows: list[list[float]] = []\\n    for site_index in range(len(vision_sums)):\\n        vision_rows.append((vision_sums[site_index] / max(vision_counts[site_index], 1.0)).tolist())\\n        language_rows.append((language_sums[site_index] / max(language_counts[site_index], 1.0)).tolist())\\n\\n    vision_entropy = [_row_entropy(row) for row in vision_rows]\\n    language_entropy = [_row_entropy(row) for row in language_rows]\\n    vision_embedding = [_embedding_contribution(row, indices) for row, indices in zip(vision_rows, source_indices)]\\n    language_embedding = [_embedding_contribution(row, indices) for row, indices in zip(language_rows, source_indices)]\\n\\n    return VisionLanguageAlphaSummary(\\n        vision_rows=vision_rows,\\n        language_rows=language_rows,\\n        source_indices=source_indices,\\n        vision_entropy=vision_entropy,\\n        language_entropy=language_entropy,\\n        vision_embedding=vision_embedding,\\n        language_embedding=language_embedding,\\n    )\\n", "src/metrics/depth_metrics.py": "from __future__ import annotations\\n\\nfrom typing import Any, Iterable, Mapping\\n\\nimport torch\\n\\n\\ndef contribution_breakdown(\\n    mean_weights: Iterable[torch.Tensor],\\n    source_indices: Iterable[list[int]],\\n) -> dict[str, float]:\\n    embedding_values: list[float] = []\\n    early_values: list[float] = []\\n    late_values: list[float] = []\\n    entropy_values: list[float] = []\\n\\n    for weights, indices in zip(mean_weights, source_indices):\\n        weights_cpu = weights.detach().cpu()\\n        if not indices:\\n            continue\\n        if 0 in indices:\\n            embedding_values.append(float(weights_cpu[indices.index(0)].item()))\\n        non_embedding_positions = [position for position, index in enumerate(indices) if index != 0]\\n        if non_embedding_positions:\\n            split = max(1, len(non_embedding_positions) // 2)\\n            early_positions = non_embedding_positions[:split]\\n            late_positions = non_embedding_positions[split:]\\n            early_values.append(float(weights_cpu[early_positions].sum().item()))\\n            late_values.append(float(weights_cpu[late_positions].sum().item()) if late_positions else 0.0)\\n        entropy = -(weights_cpu * weights_cpu.clamp_min(1e-8).log()).sum().item()\\n        entropy_values.append(float(entropy))\\n\\n    def _mean(values: list[float]) -> float:\\n        return float(sum(values) / max(1, len(values)))\\n\\n    return {\\n        \\"embedding_contribution\\": _mean(embedding_values),\\n        \\"early_contribution\\": _mean(early_values),\\n        \\"late_contribution\\": _mean(late_values),\\n        \\"depth_attention_entropy\\": _mean(entropy_values),\\n    }\\n\\n\\ndef average_scalars(rows: list[Mapping[str, float]]) -> dict[str, float]:\\n    if not rows:\\n        return {}\\n    keys = sorted(rows[0].keys())\\n    return {\\n        key: sum(float(row[key]) for row in rows) / len(rows)\\n        for key in keys\\n    }\\n\\n\\ndef average_depth_artifacts(\\n    depth_rows_batches: list[list[torch.Tensor]],\\n    source_indices_batches: list[list[list[int]]],\\n) -> dict[str, Any]:\\n    if not depth_rows_batches:\\n        return {\\n            \\"depth_attention_rows\\": [],\\n            \\"depth_source_indices\\": [],\\n            \\"mean_embedding_contribution\\": None,\\n            \\"mean_early_contribution\\": None,\\n            \\"mean_late_contribution\\": None,\\n            \\"mean_depth_attention_entropy\\": None,\\n        }\\n\\n    sums: list[torch.Tensor] = []\\n    source_indices = source_indices_batches[0]\\n    for batch_rows in depth_rows_batches:\\n        if not sums:\\n            sums = [row.detach().cpu().clone() for row in batch_rows]\\n        else:\\n            for index, row in enumerate(batch_rows):\\n                sums[index] += row.detach().cpu()\\n\\n    averages = [row / len(depth_rows_batches) for row in sums]\\n    contributions = contribution_breakdown(averages, source_indices)\\n    return {\\n        \\"depth_attention_rows\\": [row.tolist() for row in averages],\\n        \\"depth_source_indices\\": source_indices,\\n        \\"mean_embedding_contribution\\": contributions[\\"embedding_contribution\\"],\\n        \\"mean_early_contribution\\": contributions[\\"early_contribution\\"],\\n        \\"mean_late_contribution\\": contributions[\\"late_contribution\\"],\\n        \\"mean_depth_attention_entropy\\": contributions[\\"depth_attention_entropy\\"],\\n    }\\n", "src/metrics/norms.py": "from __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, DefaultDict\\n\\nimport torch\\nimport torch.nn as nn\\n\\n\\ndef language_model_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\\n    return torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))\\n\\n\\ndef second_half_language_model_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\\n    half_start = targets.size(1) // 2\\n    tail_logits = logits[:, half_start:, :]\\n    tail_targets = targets[:, half_start:]\\n    return language_model_loss(tail_logits, tail_targets)\\n\\n\\ndef position_wise_language_model_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:\\n    per_token = torch.nn.functional.cross_entropy(\\n        logits.reshape(-1, logits.size(-1)),\\n        targets.reshape(-1),\\n        reduction=\'none\',\\n    )\\n    per_token = per_token.view_as(targets)\\n    return per_token.mean(dim=0)\\n\\n\\ndef perplexity_from_loss(loss: float) -> float:\\n    return float(torch.exp(torch.tensor(min(loss, 20.0))).item())\\n\\n\\ndef _magnitude(tensor: torch.Tensor) -> float:\\n    return float(tensor.detach().float().norm(dim=-1).mean().item())\\n\\n\\ndef _layer_input_site(module_name: str) -> str | None:\\n    \\"\\"\\"Return the canonical depth site for a transformer\'s pre-attention input.\\"\\"\\"\\n    suffix = \\".attn_norm\\"\\n    if not module_name.endswith(suffix):\\n        return None\\n    site = module_name[: -len(suffix)]\\n    parts = site.split(\\".\\")\\n    if len(parts) < 2 or parts[-2] != \\"blocks\\" or not parts[-1].isdigit():\\n        return None\\n    return site\\n\\n\\ndef _layer_number(site: str) -> int:\\n    parts = site.split(\\".\\")\\n    for index, part in enumerate(parts[:-1]):\\n        if part == \\"blocks\\":\\n            try:\\n                return int(parts[index + 1])\\n            except ValueError:\\n                break\\n    return -1\\n\\n\\n@dataclass\\nclass LayerInputMagnitudeTracker:\\n    \\"\\"\\"Measure paper-style ``h_l`` and ``dL/dh_l`` at every transformer layer.\\n\\n    The canonical site is the tensor entering each block\'s pre-attention RMSNorm:\\n    baseline ``x`` and Full/Block AttnRes ``attn_input``. Forward and gradient\\n    magnitudes therefore use one identical definition across all architectures.\\n    \\"\\"\\"\\n\\n    input_sums: DefaultDict[str, float] = field(default_factory=lambda: defaultdict(float))\\n    input_counts: DefaultDict[str, int] = field(default_factory=lambda: defaultdict(int))\\n    gradient_sums: DefaultDict[str, float] = field(default_factory=lambda: defaultdict(float))\\n    gradient_counts: DefaultDict[str, int] = field(default_factory=lambda: defaultdict(int))\\n    handles: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)\\n\\n    def register(self, model: nn.Module) -> None:\\n        for name, module in model.named_modules():\\n            site = _layer_input_site(name)\\n            if site is not None:\\n                self.handles.append(module.register_forward_pre_hook(self._forward_pre_hook(site)))\\n\\n    def _forward_pre_hook(self, site: str):\\n        def hook(_module: nn.Module, inputs: tuple[torch.Tensor, ...]) -> None:\\n            if not inputs:\\n                return\\n            layer_input = inputs[0]\\n            self.input_sums[site] += _magnitude(layer_input)\\n            self.input_counts[site] += 1\\n            if layer_input.requires_grad:\\n                layer_input.register_hook(self._gradient_hook(site))\\n\\n        return hook\\n\\n    def _gradient_hook(self, site: str):\\n        def hook(gradient: torch.Tensor) -> None:\\n            self.gradient_sums[site] += _magnitude(gradient)\\n            self.gradient_counts[site] += 1\\n\\n        return hook\\n\\n    @staticmethod\\n    def _averages(sums: DefaultDict[str, float], counts: DefaultDict[str, int]) -> dict[str, float]:\\n        sites = sorted(sums, key=lambda site: (_layer_number(site), site))\\n        return {site: sums[site] / counts[site] for site in sites if counts[site] > 0}\\n\\n    def reset_step(self) -> None:\\n        self.input_sums.clear()\\n        self.input_counts.clear()\\n        self.gradient_sums.clear()\\n        self.gradient_counts.clear()\\n\\n    def snapshot(self) -> dict[str, dict[str, float]]:\\n        return {\\n            \\"layer_input_magnitudes\\": self._averages(self.input_sums, self.input_counts),\\n            \\"layer_input_gradient_magnitudes\\": self._averages(self.gradient_sums, self.gradient_counts),\\n        }\\n\\n    def close(self) -> None:\\n        for handle in self.handles:\\n            handle.remove()\\n        self.handles.clear()\\n\\n\\ndef last_layer_input_magnitude(layer_input_magnitudes: dict[str, float]) -> float | None:\\n    if not layer_input_magnitudes:\\n        return None\\n    last_site = max(layer_input_magnitudes, key=lambda site: (_layer_number(site), site))\\n    return float(layer_input_magnitudes[last_site])\\n\\n\\n# Full AttnRes layer-1 pre-attn history: [emb, L0 attn out, L0 mlp out].\\n_FULL_LAYER1_SOURCE_NAMES = (\\"emb\\", \\"l0_attn\\", \\"l0_mlp\\")\\n# Block AttnRes layer-1 (second layer of first block): [emb, partial after L0].\\n_BLOCK_LAYER1_SOURCE_NAMES = (\\"emb\\", \\"partial_after_l0\\")\\n\\n\\n@dataclass\\nclass Layer1DepthAttentionProbe:\\n    \\"\\"\\"Log layer-1 depth-attention weights and raw source magnitudes.\\n\\n    Tracks the pre-attention ``DepthAttentionResidual`` at transformer layer 1\\n    (index 1). Used to monitor the Full AttnRes layer-1 magnitude spike seen at\\n    90M: softmax concentrating on a large L0-attn source. No-op for baseline.\\n    \\"\\"\\"\\n\\n    architecture: str = \\"baseline\\"\\n    weight_sums: DefaultDict[str, float] = field(default_factory=lambda: defaultdict(float))\\n    mag_sums: DefaultDict[str, float] = field(default_factory=lambda: defaultdict(float))\\n    mixed_mag_sum: float = 0.0\\n    entropy_sum: float = 0.0\\n    n_sources: int = 0\\n    counts: int = 0\\n    _wrapped: Any = None\\n    _original_forward: Any = None\\n\\n    def register(self, model: nn.Module) -> None:\\n        architecture = getattr(getattr(model, \\"config\\", None), \\"architecture\\", \\"baseline\\")\\n        mode = getattr(getattr(getattr(model, \\"config\\", None), \\"attnres\\", None), \\"mode\\", None)\\n        if architecture == \\"block_attnres\\" or (architecture == \\"attnres\\" and mode == \\"block\\"):\\n            self.architecture = \\"block_attnres\\"\\n        elif architecture == \\"attnres\\":\\n            self.architecture = \\"attnres\\"\\n        else:\\n            self.architecture = \\"baseline\\"\\n            return\\n        if not hasattr(model, \\"blocks\\") or len(model.blocks) < 2:\\n            return\\n        mixer = model.blocks[1].pre_attn_res\\n        self._wrapped = mixer\\n        self._original_forward = mixer.forward\\n\\n        def wrapped_forward(history, *, return_stats: bool = False):\\n            mixed, stats = self._original_forward(history, return_stats=return_stats)\\n            self._record(history, mixed, stats if return_stats else None, mixer)\\n            return mixed, stats\\n\\n        mixer.forward = wrapped_forward  # type: ignore[method-assign]\\n\\n    def _source_names(self, n_sources: int) -> tuple[str, ...]:\\n        if self.architecture == \\"attnres\\" and n_sources == len(_FULL_LAYER1_SOURCE_NAMES):\\n            return _FULL_LAYER1_SOURCE_NAMES\\n        if self.architecture == \\"block_attnres\\" and n_sources == len(_BLOCK_LAYER1_SOURCE_NAMES):\\n            return _BLOCK_LAYER1_SOURCE_NAMES\\n        return tuple(f\\"source_{index}\\" for index in range(n_sources))\\n\\n    def _record(self, history, mixed: torch.Tensor, stats: dict | None, mixer: nn.Module) -> None:\\n        selected_history, _indices = mixer._select_history(history)\\n        n_sources = len(selected_history)\\n        self.n_sources = n_sources\\n        names = self._source_names(n_sources)\\n        if stats is not None and \\"mean_weights\\" in stats:\\n            mean_weights = stats[\\"mean_weights\\"]\\n            weights = [float(mean_weights[i].item()) for i in range(n_sources)]\\n            entropy = float(stats.get(\\"entropy\\", 0.0))\\n        else:\\n            values = torch.stack(selected_history, dim=0)\\n            keys = mixer.key_norm(values)\\n            logits = torch.einsum(\\"d,sbtd->sbt\\", mixer.query, keys) / max(mixer.temperature, 1e-6)\\n            weight_tensor = torch.softmax(logits, dim=0)\\n            weights = weight_tensor.detach().float().mean(dim=(1, 2)).tolist()\\n            entropy = float(\\n                (-(weight_tensor.detach().float() * weight_tensor.detach().float().clamp_min(1e-8).log())\\n                 .sum(dim=0)\\n                 .mean())\\n                .item()\\n            )\\n        for name, source, weight in zip(names, selected_history, weights):\\n            self.weight_sums[name] += float(weight)\\n            self.mag_sums[name] += _magnitude(source)\\n        self.mixed_mag_sum += _magnitude(mixed)\\n        self.entropy_sum += entropy\\n        self.counts += 1\\n\\n    def reset_step(self) -> None:\\n        self.weight_sums.clear()\\n        self.mag_sums.clear()\\n        self.mixed_mag_sum = 0.0\\n        self.entropy_sum = 0.0\\n        self.counts = 0\\n\\n    def snapshot(self) -> dict[str, Any]:\\n        if self.architecture == \\"baseline\\" or self.counts == 0:\\n            return {}\\n        scale = float(self.counts)\\n        payload: dict[str, Any] = {\\n            \\"layer1_depth_attn/n_sources\\": float(self.n_sources),\\n            \\"layer1_depth_attn/entropy\\": self.entropy_sum / scale,\\n            \\"layer1_depth_attn/mixed_mag\\": self.mixed_mag_sum / scale,\\n        }\\n        for name in self.weight_sums:\\n            payload[f\\"layer1_depth_attn/weight_{name}\\"] = self.weight_sums[name] / scale\\n            payload[f\\"layer1_depth_attn/mag_{name}\\"] = self.mag_sums[name] / scale\\n        return payload\\n\\n    def close(self) -> None:\\n        if self._wrapped is not None and self._original_forward is not None:\\n            self._wrapped.forward = self._original_forward\\n        self._wrapped = None\\n        self._original_forward = None\\n", "src/utils/config.py": "from __future__ import annotations\\n\\nimport warnings\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\nfrom typing import Any, Optional\\n\\nimport yaml\\n\\n\\n@dataclass\\nclass AttnResConfig:\\n    enabled: bool = False\\n    mode: str = \'full\'\\n    num_blocks: Optional[int] = None\\n    window_size: Optional[int] = None\\n    temperature: float = 1.0\\n    rmsnorm_keys: bool = True\\n    zero_init_queries: bool = True\\n    include_embedding: bool = True\\n    keep_embedding_in_window: bool = True\\n    final_readout: bool = True\\n\\n\\n@dataclass\\nclass ModelConfig:\\n    architecture: str = \'baseline\'\\n    size_name: str = \'small\'\\n    vocab_size: int = 0\\n    max_seq_len: int = 128\\n    d_model: int = 384\\n    n_layers: int = 6\\n    n_heads: int = 6\\n    d_ff: int = 1536\\n    dropout: float = 0.1\\n    bias: bool = True\\n    tie_weights: bool = True\\n    norm_eps: float = 1e-5\\n    init_std: float = 0.02\\n    attnres: AttnResConfig = field(default_factory=AttnResConfig)\\n\\n\\n@dataclass\\nclass DataConfig:\\n    dataset_type: str = \'tinystories\'\\n    dataset_name: str = \'tinystories\'\\n    tokenizer_name: str = \'gpt2\'\\n    text_path: Optional[str] = None\\n    train_text_path: Optional[str] = None\\n    val_text_path: Optional[str] = None\\n    train_split: str = \'train\'\\n    val_split: str = \'validation\'\\n    fineweb_subset: str = \'sample-10BT\'\\n    val_fraction: float = 0.005\\n    hash_modulo: int = 1000\\n    block_size: int = 128\\n    context_lengths: Optional[list[int]] = None\\n    batch_size: int = 16\\n    eval_batch_size: int = 16\\n    num_workers: int = 0\\n    pin_memory: bool = True\\n    max_train_examples: Optional[int] = None\\n    max_train_tokens: Optional[int] = 250_000\\n    max_val_examples: Optional[int] = None\\n    max_val_tokens: Optional[int] = 50_000\\n\\n\\n@dataclass\\nclass TrainingConfig:\\n    max_steps: int = 300\\n    grad_accum_steps: int = 1\\n    learning_rate: float = 3e-4\\n    min_lr: float = 3e-5\\n    warmup_steps: int = 30\\n    weight_decay: float = 0.1\\n    beta1: float = 0.9\\n    beta2: float = 0.95\\n    adam_eps: float = 1e-8\\n    grad_clip: float = 1.0\\n    mixed_precision: bool = True\\n    amp_dtype: str = \'bfloat16\'\\n    log_interval: int = 10\\n    eval_interval: int = 100\\n    checkpoint_interval: int = 100\\n    probe_interval: int = 100\\n    # Optional stdout step progress (off by default; W&B/file logging is separate).\\n    console_step_tracking: bool = False\\n    console_step_interval: int = 1\\n    eval_max_batches: Optional[int] = 10\\n    device: str = \'auto\'\\n    resume_from: Optional[str] = None\\n    allow_resume_mismatch: bool = False\\n\\n\\n@dataclass\\nclass WandbConfig:\\n    enabled: bool = True\\n    project: str = \'attnres-next-scale\'\\n    entity: Optional[str] = None\\n    mode: str = \'auto\'\\n    group: Optional[str] = None\\n    job_type: str = \'train\'\\n    tags: list[str] = field(default_factory=list)\\n    run_name: Optional[str] = None\\n\\n\\n@dataclass\\nclass LoggingConfig:\\n    output_root: str = \'outputs\'\\n    save_probes: bool = True\\n    save_checkpoints: bool = True\\n    keep_last_k_checkpoints: int = 2\\n    wandb: WandbConfig = field(default_factory=WandbConfig)\\n\\n\\n@dataclass\\nclass EvaluationConfig:\\n    max_batches: Optional[int] = 10\\n    positionwise_steps: list[int] = field(default_factory=list)\\n    positionwise_max_batches: Optional[int] = None\\n\\n\\n@dataclass\\nclass ExperimentConfig:\\n    name: str = \'first_run\'\\n    stage: str = \'first_run\'\\n    seed: int = 1337\\n    deterministic: bool = False\\n    notes: str = \'\'\\n\\n\\n@dataclass\\nclass Config:\\n    experiment: ExperimentConfig = field(default_factory=ExperimentConfig)\\n    data: DataConfig = field(default_factory=DataConfig)\\n    model: ModelConfig = field(default_factory=ModelConfig)\\n    training: TrainingConfig = field(default_factory=TrainingConfig)\\n    logging: LoggingConfig = field(default_factory=LoggingConfig)\\n    evaluation: EvaluationConfig = field(default_factory=EvaluationConfig)\\n    batching: dict[str, dict[str, int]] = field(default_factory=dict)\\n\\n\\ndef _normalize_batching(values: dict[str, Any]) -> dict[str, dict[str, int]]:\\n    normalized: dict[str, dict[str, int]] = {}\\n    for key, value in values.items():\\n        if not isinstance(value, dict):\\n            raise ValueError(f\'batching.{key} must be a mapping\')\\n        payload = dict(value)\\n        if \'grad_accum\' in payload and \'grad_accum_steps\' not in payload:\\n            payload[\'grad_accum_steps\'] = payload.pop(\'grad_accum\')\\n        if \'batch_size\' not in payload or \'grad_accum_steps\' not in payload:\\n            raise ValueError(f\'batching.{key} must define batch_size and grad_accum_steps\')\\n        normalized[key] = {\\n            \'batch_size\': int(payload[\'batch_size\']),\\n            \'grad_accum_steps\': int(payload[\'grad_accum_steps\']),\\n        }\\n    return normalized\\n\\n\\ndef _construct_config(values: dict[str, Any]) -> Config:\\n    logging_values = values.get(\'logging\', {})\\n    wandb_values = logging_values.get(\'wandb\', values.get(\'wandb\', {}))\\n    return Config(\\n        experiment=ExperimentConfig(**values.get(\'experiment\', {})),\\n        data=DataConfig(**values.get(\'data\', {})),\\n        model=ModelConfig(\\n            **{\\n                **values.get(\'model\', {}),\\n                \'attnres\': AttnResConfig(**values.get(\'model\', {}).get(\'attnres\', {})),\\n            }\\n        ),\\n        training=TrainingConfig(**values.get(\'training\', {})),\\n        logging=LoggingConfig(\\n            **{\\n                **logging_values,\\n                \'wandb\': WandbConfig(**wandb_values),\\n            }\\n        ),\\n        evaluation=EvaluationConfig(**values.get(\'evaluation\', {})),\\n        batching=_normalize_batching(values.get(\'batching\', {})),\\n    )\\n\\n\\ndef config_to_dict(config: Config) -> dict[str, Any]:\\n    return asdict(config)\\n\\n\\ndef apply_overrides(config_dict: dict[str, Any], overrides: list[str]) -> dict[str, Any]:\\n    for override in overrides:\\n        key, raw_value = override.split(\'=\', maxsplit=1)\\n        cursor = config_dict\\n        parts = key.split(\'.\')\\n        for part in parts[:-1]:\\n            cursor = cursor.setdefault(part, {})\\n        cursor[parts[-1]] = yaml.safe_load(raw_value)\\n    return config_dict\\n\\n\\ndef _validate_cap_pair(name_a: str, value_a: Optional[int], name_b: str, value_b: Optional[int]) -> None:\\n    if value_a is not None and value_a <= 0:\\n        raise ValueError(f\'{name_a} must be positive when set\')\\n    if value_b is not None and value_b <= 0:\\n        raise ValueError(f\'{name_b} must be positive when set\')\\n    if value_a is not None and value_b is not None:\\n        raise ValueError(f\'Set at most one of {name_a} and {name_b}\')\\n\\n\\ndef validate_config(config: Config) -> Config:\\n    if config.model.architecture not in {\'baseline\', \'attnres\', \'block_attnres\'}:\\n        raise ValueError(f\'Unsupported architecture: {config.model.architecture}\')\\n    if config.model.attnres.mode not in {\'full\', \'block\'}:\\n        raise ValueError(\'model.attnres.mode must be one of: full, block\')\\n    if config.model.size_name not in {\'small\', \'medium\', \'large\'}:\\n        raise ValueError(\'model.size_name must be one of: small, medium, large\')\\n    if config.model.d_model % config.model.n_heads != 0:\\n        raise ValueError(\'model.d_model must be divisible by model.n_heads\')\\n    if config.data.block_size > config.model.max_seq_len:\\n        raise ValueError(\'data.block_size must be <= model.max_seq_len\')\\n    if config.training.min_lr > config.training.learning_rate:\\n        raise ValueError(\'training.min_lr must be <= training.learning_rate\')\\n    if config.training.console_step_interval <= 0:\\n        raise ValueError(\'training.console_step_interval must be positive\')\\n    if config.data.dataset_type not in {\'tinystories\', \'local_text\', \'fineweb_edu\'}:\\n        raise ValueError(\'data.dataset_type must be one of: tinystories, local_text, fineweb_edu\')\\n    if config.data.dataset_type == \'fineweb_edu\':\\n        if not 0.0 < config.data.val_fraction < 1.0:\\n            raise ValueError(\'data.val_fraction must be in (0, 1) for fineweb_edu\')\\n        if config.data.hash_modulo < 2:\\n            raise ValueError(\'data.hash_modulo must be >= 2 for fineweb_edu\')\\n    _validate_cap_pair(\\n        \'data.max_train_examples\',\\n        config.data.max_train_examples,\\n        \'data.max_train_tokens\',\\n        config.data.max_train_tokens,\\n    )\\n    _validate_cap_pair(\\n        \'data.max_val_examples\',\\n        config.data.max_val_examples,\\n        \'data.max_val_tokens\',\\n        config.data.max_val_tokens,\\n    )\\n    if config.data.context_lengths is not None:\\n        if not config.data.context_lengths:\\n            raise ValueError(\'data.context_lengths must not be empty when set\')\\n        for context in config.data.context_lengths:\\n            if context <= 0:\\n                raise ValueError(\'data.context_lengths values must be positive\')\\n            if context > config.model.max_seq_len:\\n                raise ValueError(\'each data.context_lengths value must be <= model.max_seq_len\')\\n    for key, batching in config.batching.items():\\n        if not key.startswith(\'ctx\'):\\n            raise ValueError(\'batching keys must look like ctx256 or ctx512\')\\n        if batching[\'batch_size\'] <= 0 or batching[\'grad_accum_steps\'] <= 0:\\n            raise ValueError(f\'batching.{key} values must be positive\')\\n    if config.logging.wandb.mode not in {\'auto\', \'online\', \'offline\', \'disabled\'}:\\n        raise ValueError(\'logging.wandb.mode must be one of: auto, online, offline, disabled\')\\n    if config.logging.wandb.enabled and not config.logging.wandb.project:\\n        raise ValueError(\'logging.wandb.project must be non-empty when W&B is enabled\')\\n    if any(not isinstance(tag, str) for tag in config.logging.wandb.tags):\\n        raise ValueError(\'logging.wandb.tags must contain strings only\')\\n    for step in config.evaluation.positionwise_steps:\\n        if step <= 0:\\n            raise ValueError(\'evaluation.positionwise_steps values must be positive\')\\n        if step > config.training.max_steps:\\n            raise ValueError(\'evaluation.positionwise_steps values must be <= training.max_steps\')\\n    if config.evaluation.positionwise_max_batches is not None and config.evaluation.positionwise_max_batches <= 0:\\n        raise ValueError(\'evaluation.positionwise_max_batches must be positive when set\')\\n    if config.model.architecture == \'block_attnres\':\\n        config.model.attnres.enabled = True\\n        config.model.attnres.mode = \'block\'\\n    elif config.model.architecture == \'attnres\':\\n        config.model.attnres.enabled = True\\n    if config.model.attnres.enabled and config.model.attnres.mode == \'block\':\\n        num_blocks = config.model.attnres.num_blocks\\n        if num_blocks is None or num_blocks < 1:\\n            raise ValueError(\'model.attnres.num_blocks must be a positive integer in block mode\')\\n        if num_blocks > config.model.n_layers:\\n            raise ValueError(\'model.attnres.num_blocks must be <= model.n_layers\')\\n        if config.model.attnres.window_size is not None:\\n            warnings.warn(\'model.attnres.window_size is ignored in block mode (v1).\')\\n    return config\\n\\n\\ndef load_config(path: str | Path, overrides: Optional[list[str]] = None) -> Config:\\n    with Path(path).open(\'r\', encoding=\'utf-8\') as handle:\\n        payload = yaml.safe_load(handle) or {}\\n    if overrides:\\n        payload = apply_overrides(payload, overrides)\\n    return validate_config(_construct_config(payload))\\n\\n\\ndef load_config_from_dict(payload: dict[str, Any]) -> Config:\\n    return validate_config(_construct_config(payload))\\n\\n\\ndef save_config(config: Config, path: str | Path) -> None:\\n    with Path(path).open(\'w\', encoding=\'utf-8\') as handle:\\n        yaml.safe_dump(config_to_dict(config), handle, sort_keys=False)\\n", "src/utils/runtime.py": "from __future__ import annotations\\n\\nimport math\\nimport random\\nfrom pathlib import Path\\nfrom typing import Any, Iterator\\n\\nimport numpy as np\\nimport torch\\nfrom torch.utils.data import DataLoader\\n\\n\\ndef seed_everything(seed: int, deterministic: bool = False) -> None:\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    torch.cuda.manual_seed_all(seed)\\n    if deterministic:\\n        torch.backends.cudnn.deterministic = True\\n        torch.backends.cudnn.benchmark = False\\n\\n\\ndef manual_seed_generator(seed: int) -> torch.Generator:\\n    generator = torch.Generator()\\n    generator.manual_seed(seed)\\n    return generator\\n\\n\\ndef seed_worker(worker_id: int) -> None:\\n    worker_seed = torch.initial_seed() % 2**32\\n    np.random.seed(worker_seed)\\n    random.seed(worker_seed)\\n\\n\\ndef get_device(device: str = \\"auto\\") -> torch.device:\\n    if device == \\"auto\\":\\n        if torch.cuda.is_available():\\n            return torch.device(\\"cuda\\")\\n        if getattr(torch.backends, \\"mps\\", None) is not None and torch.backends.mps.is_available():\\n            return torch.device(\\"mps\\")\\n        return torch.device(\\"cpu\\")\\n    return torch.device(device)\\n\\n\\ndef ensure_dir(path: str | Path) -> Path:\\n    path = Path(path)\\n    path.mkdir(parents=True, exist_ok=True)\\n    return path\\n\\n\\ndef sanitize_name(value: str) -> str:\\n    return \\"\\".join(char if char.isalnum() else \\"_\\" for char in value.lower()).strip(\\"_\\")\\n\\n\\ndef amp_dtype_from_string(name: str) -> torch.dtype:\\n    key = name.lower()\\n    mapping = {\\n        \\"float16\\": torch.float16,\\n        \\"fp16\\": torch.float16,\\n        \\"bfloat16\\": torch.bfloat16,\\n        \\"bf16\\": torch.bfloat16,\\n        \\"float32\\": torch.float32,\\n        \\"fp32\\": torch.float32,\\n    }\\n    if key not in mapping:\\n        raise ValueError(f\\"Unsupported AMP dtype: {name}\\")\\n    return mapping[key]\\n\\n\\ndef count_parameters(model: torch.nn.Module) -> dict[str, int]:\\n    total = sum(param.numel() for param in model.parameters())\\n    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)\\n    return {\\"total\\": total, \\"trainable\\": trainable}\\n\\n\\ndef overall_grad_norm(model: torch.nn.Module) -> float:\\n    total = 0.0\\n    for param in model.parameters():\\n        if param.grad is None:\\n            continue\\n        total += param.grad.detach().float().pow(2).sum().item()\\n    return math.sqrt(total)\\n\\n\\ndef cycle(loader: DataLoader) -> Iterator[dict[str, torch.Tensor]]:\\n    while True:\\n        for batch in loader:\\n            yield batch\\n\\n\\ndef get_rng_state() -> dict[str, Any]:\\n    state: dict[str, Any] = {\\n        \\"python\\": random.getstate(),\\n        \\"numpy\\": np.random.get_state(),\\n        \\"torch\\": torch.get_rng_state(),\\n    }\\n    if torch.cuda.is_available():\\n        state[\\"torch_cuda\\"] = torch.cuda.get_rng_state_all()\\n    return state\\n\\n\\ndef _coerce_torch_rng_state(state: Any) -> torch.Tensor:\\n    \\"\\"\\"Checkpoints may deserialize RNG state as a non-uint8 tensor.\\"\\"\\"\\n    if isinstance(state, torch.Tensor):\\n        return state.to(dtype=torch.uint8).cpu()\\n    return torch.as_tensor(state, dtype=torch.uint8)\\n\\n\\ndef set_rng_state(state: dict[str, Any]) -> None:\\n    random.setstate(state[\\"python\\"])\\n    np.random.set_state(state[\\"numpy\\"])\\n    torch.set_rng_state(_coerce_torch_rng_state(state[\\"torch\\"]))\\n    if torch.cuda.is_available() and \\"torch_cuda\\" in state:\\n        cuda_states = [_coerce_torch_rng_state(s) for s in state[\\"torch_cuda\\"]]\\n        torch.cuda.set_rng_state_all(cuda_states)\\n", "src/vlm/synthetic_vqa.py": "from __future__ import annotations\\n\\nimport hashlib\\nfrom dataclasses import dataclass\\nfrom typing import Any, Literal\\n\\nimport numpy as np\\nimport torch\\nfrom torch.utils.data import Dataset\\n\\nCOLORS = (\\"red\\", \\"green\\", \\"blue\\", \\"yellow\\")\\nSHAPES = (\\"circle\\", \\"square\\", \\"triangle\\")\\nDIGITS = (\\"zero\\", \\"one\\", \\"two\\", \\"three\\", \\"four\\", \\"five\\", \\"six\\", \\"seven\\", \\"eight\\", \\"nine\\")\\nCOUNT_WORDS = (\\"one\\", \\"two\\", \\"three\\", \\"four\\")\\nLOCATIONS = (\\n    \\"top_left\\",\\n    \\"top\\",\\n    \\"top_right\\",\\n    \\"left\\",\\n    \\"center\\",\\n    \\"right\\",\\n    \\"bottom_left\\",\\n    \\"bottom\\",\\n    \\"bottom_right\\",\\n)\\nFAMILIES = (\\"local_detail\\", \\"attribute\\", \\"counting\\", \\"location\\", \\"relation\\")\\nYES_NO = (\\"yes\\", \\"no\\")\\n\\nCOLOR_RGB = {\\n    \\"red\\": (1.0, 0.1, 0.1),\\n    \\"green\\": (0.1, 0.8, 0.1),\\n    \\"blue\\": (0.1, 0.3, 1.0),\\n    \\"yellow\\": (0.95, 0.9, 0.1),\\n}\\n\\n# Hard-coded 3x5 digit glyphs. 1 = ink, 0 = background.\\nDIGIT_GLYPHS: dict[int, tuple[str, ...]] = {\\n    0: (\\"111\\", \\"101\\", \\"101\\", \\"101\\", \\"111\\"),\\n    1: (\\"010\\", \\"110\\", \\"010\\", \\"010\\", \\"111\\"),\\n    2: (\\"111\\", \\"001\\", \\"111\\", \\"100\\", \\"111\\"),\\n    3: (\\"111\\", \\"001\\", \\"111\\", \\"001\\", \\"111\\"),\\n    4: (\\"101\\", \\"101\\", \\"111\\", \\"001\\", \\"001\\"),\\n    5: (\\"111\\", \\"100\\", \\"111\\", \\"001\\", \\"111\\"),\\n    6: (\\"111\\", \\"100\\", \\"111\\", \\"101\\", \\"111\\"),\\n    7: (\\"111\\", \\"001\\", \\"010\\", \\"010\\", \\"010\\"),\\n    8: (\\"111\\", \\"101\\", \\"111\\", \\"101\\", \\"111\\"),\\n    9: (\\"111\\", \\"101\\", \\"111\\", \\"001\\", \\"111\\"),\\n}\\n\\nSPECIAL_TOKENS = (\\"<pad>\\", \\"<bos>\\", \\"<eos>\\", \\"<answer>\\")\\nQUESTION_WORDS = (\\n    \\"what\\",\\n    \\"digit\\",\\n    \\"is\\",\\n    \\"inside\\",\\n    \\"the\\",\\n    \\"color\\",\\n    \\"shape\\",\\n    \\"how\\",\\n    \\"many\\",\\n    \\"objects\\",\\n    \\"are\\",\\n    \\"there\\",\\n    \\"where\\",\\n    \\"left\\",\\n    \\"of\\",\\n    \\"above\\",\\n    \\"below\\",\\n    \\"right\\",\\n)\\n\\n\\n@dataclass(frozen=True)\\nclass SceneObject:\\n    color: str\\n    shape: str\\n    location: str\\n    digit: int | None\\n\\n\\n@dataclass(frozen=True)\\nclass VQAExample:\\n    example_index: int\\n    split: str\\n    family: str\\n    question: str\\n    answer: str\\n    objects: tuple[SceneObject, ...]\\n    image: np.ndarray  # float32 CHW in [0, 1]\\n\\n\\nclass VQATokenizer:\\n    def __init__(self) -> None:\\n        vocab = list(SPECIAL_TOKENS)\\n        for token in (\\n            *QUESTION_WORDS,\\n            *COLORS,\\n            *SHAPES,\\n            *DIGITS,\\n            *COUNT_WORDS,\\n            *LOCATIONS,\\n            *YES_NO,\\n        ):\\n            if token not in vocab:\\n                vocab.append(token)\\n        self.token_to_id = {token: index for index, token in enumerate(vocab)}\\n        self.id_to_token = {index: token for token, index in self.token_to_id.items()}\\n        self.pad_token_id = self.token_to_id[\\"<pad>\\"]\\n        self.bos_token_id = self.token_to_id[\\"<bos>\\"]\\n        self.eos_token_id = self.token_to_id[\\"<eos>\\"]\\n        self.answer_token_id = self.token_to_id[\\"<answer>\\"]\\n\\n    @property\\n    def vocab_size(self) -> int:\\n        return len(self.token_to_id)\\n\\n    @property\\n    def vocab(self) -> dict[str, int]:\\n        return dict(self.token_to_id)\\n\\n    def encode(self, text: str) -> list[int]:\\n        return [self.token_to_id[token] for token in text.split()]\\n\\n    def decode(self, ids: list[int] | torch.Tensor) -> str:\\n        if isinstance(ids, torch.Tensor):\\n            ids = ids.tolist()\\n        return \\" \\".join(self.id_to_token[int(token_id)] for token_id in ids)\\n\\n    def encode_supervised(\\n        self,\\n        question: str,\\n        answer: str,\\n        *,\\n        supervise_eos: bool = True,\\n    ) -> dict[str, list[int]]:\\n        question_ids = self.encode(question)\\n        answer_id = self.token_to_id[answer]\\n        input_ids = [self.bos_token_id, *question_ids, self.answer_token_id, answer_id, self.eos_token_id]\\n        targets = [-100] * len(input_ids)\\n        answer_position = len(input_ids) - 2\\n        targets[answer_position] = answer_id\\n        if supervise_eos:\\n            targets[answer_position + 1] = self.eos_token_id\\n        return {\\n            \\"input_ids\\": input_ids,\\n            \\"targets\\": targets,\\n            \\"answer_position\\": answer_position,\\n            \\"answer_id\\": answer_id,\\n        }\\n\\n\\ndef _rng(split_seed: int, example_index: int) -> np.random.Generator:\\n    material = f\\"{split_seed}:{example_index}\\".encode(\\"utf-8\\")\\n    digest = hashlib.sha256(material).digest()\\n    seed = int.from_bytes(digest[:8], \\"little\\")\\n    return np.random.default_rng(seed)\\n\\n\\ndef _cell_center(location: str, image_size: int) -> tuple[int, int]:\\n    row = LOCATIONS.index(location) // 3\\n    col = LOCATIONS.index(location) % 3\\n    cell = image_size // 3\\n    return row * cell + cell // 2, col * cell + cell // 2\\n\\n\\ndef _draw_circle(canvas: np.ndarray, cy: int, cx: int, radius: int, color: tuple[float, float, float]) -> None:\\n    yy, xx = np.ogrid[: canvas.shape[1], : canvas.shape[2]]\\n    mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius**2\\n    for channel, value in enumerate(color):\\n        canvas[channel][mask] = value\\n\\n\\ndef _draw_square(canvas: np.ndarray, cy: int, cx: int, half: int, color: tuple[float, float, float]) -> None:\\n    y0, y1 = max(0, cy - half), min(canvas.shape[1], cy + half)\\n    x0, x1 = max(0, cx - half), min(canvas.shape[2], cx + half)\\n    for channel, value in enumerate(color):\\n        canvas[channel, y0:y1, x0:x1] = value\\n\\n\\ndef _draw_triangle(canvas: np.ndarray, cy: int, cx: int, half: int, color: tuple[float, float, float]) -> None:\\n    for row in range(-half, half + 1):\\n        width = int(half * (1.0 - abs(row) / max(half, 1)))\\n        y = cy + row\\n        if y < 0 or y >= canvas.shape[1]:\\n            continue\\n        x0, x1 = max(0, cx - width), min(canvas.shape[2], cx + width + 1)\\n        for channel, value in enumerate(color):\\n            canvas[channel, y, x0:x1] = value\\n\\n\\ndef _draw_digit(canvas: np.ndarray, cy: int, cx: int, digit: int) -> None:\\n    glyph = DIGIT_GLYPHS[digit]\\n    ink = (0.05, 0.05, 0.05)\\n    for row, line in enumerate(glyph):\\n        for col, bit in enumerate(line):\\n            if bit != \\"1\\":\\n                continue\\n            y = cy - 2 + row\\n            x = cx - 1 + col\\n            if 0 <= y < canvas.shape[1] and 0 <= x < canvas.shape[2]:\\n                for channel, value in enumerate(ink):\\n                    canvas[channel, y, x] = value\\n\\n\\ndef render_scene(objects: tuple[SceneObject, ...], *, image_size: int = 64) -> np.ndarray:\\n    canvas = np.full((3, image_size, image_size), 0.92, dtype=np.float32)\\n    half = max(4, image_size // 10)\\n    for obj in objects:\\n        cy, cx = _cell_center(obj.location, image_size)\\n        color = COLOR_RGB[obj.color]\\n        if obj.shape == \\"circle\\":\\n            _draw_circle(canvas, cy, cx, half, color)\\n        elif obj.shape == \\"square\\":\\n            _draw_square(canvas, cy, cx, half, color)\\n        else:\\n            _draw_triangle(canvas, cy, cx, half, color)\\n        if obj.digit is not None:\\n            _draw_digit(canvas, cy, cx, obj.digit)\\n    return canvas\\n\\n\\ndef _sample_objects(rng: np.random.Generator) -> tuple[SceneObject, ...]:\\n    count = int(rng.integers(2, 5))\\n    locations = list(rng.choice(LOCATIONS, size=count, replace=False))\\n    objects: list[SceneObject] = []\\n    used_color_shape: set[tuple[str, str]] = set()\\n    for location in locations:\\n        for _ in range(32):\\n            color = str(rng.choice(COLORS))\\n            shape = str(rng.choice(SHAPES))\\n            if (color, shape) in used_color_shape:\\n                continue\\n            used_color_shape.add((color, shape))\\n            digit = int(rng.integers(0, 10)) if rng.random() < 0.7 else None\\n            objects.append(SceneObject(color=color, shape=shape, location=str(location), digit=digit))\\n            break\\n        else:\\n            color = str(rng.choice(COLORS))\\n            shape = str(rng.choice(SHAPES))\\n            objects.append(SceneObject(color=color, shape=shape, location=str(location), digit=None))\\n    return tuple(objects)\\n\\n\\ndef _unique_by(objects: tuple[SceneObject, ...], attr: str, value: str) -> SceneObject | None:\\n    matches = [obj for obj in objects if getattr(obj, attr) == value]\\n    return matches[0] if len(matches) == 1 else None\\n\\n\\ndef _build_question(rng: np.random.Generator, objects: tuple[SceneObject, ...]) -> tuple[str, str, str]:\\n    families = list(FAMILIES)\\n    rng.shuffle(families)\\n    for family in families:\\n        if family == \\"local_detail\\":\\n            candidates = [\\n                obj\\n                for obj in objects\\n                if obj.digit is not None\\n                and sum(item.color == obj.color and item.shape == obj.shape for item in objects) == 1\\n            ]\\n            if not candidates:\\n                continue\\n            obj = candidates[int(rng.integers(0, len(candidates)))]\\n            question = f\\"what digit is inside the {obj.color} {obj.shape}\\"\\n            return family, question, DIGITS[obj.digit]\\n\\n        if family == \\"attribute\\":\\n            if rng.random() < 0.5:\\n                shape = str(rng.choice(SHAPES))\\n                obj = _unique_by(objects, \\"shape\\", shape)\\n                if obj is None:\\n                    continue\\n                return family, f\\"what color is the {shape}\\", obj.color\\n            color = str(rng.choice(COLORS))\\n            obj = _unique_by(objects, \\"color\\", color)\\n            if obj is None:\\n                continue\\n            return family, f\\"what shape is {color}\\", obj.shape\\n\\n        if family == \\"counting\\":\\n            if rng.random() < 0.5:\\n                shape = str(rng.choice(SHAPES))\\n                count = sum(obj.shape == shape for obj in objects)\\n                if count == 0:\\n                    continue\\n                return family, f\\"how many {shape}s are there\\", COUNT_WORDS[count - 1]\\n            color = str(rng.choice(COLORS))\\n            count = sum(obj.color == color for obj in objects)\\n            if count == 0:\\n                continue\\n            return family, f\\"how many {color} objects are there\\", COUNT_WORDS[count - 1]\\n\\n        if family == \\"location\\":\\n            unique = [\\n                obj\\n                for obj in objects\\n                if sum(item.color == obj.color and item.shape == obj.shape for item in objects) == 1\\n            ]\\n            if not unique:\\n                continue\\n            obj = unique[int(rng.integers(0, len(unique)))]\\n            return family, f\\"where is the {obj.color} {obj.shape}\\", obj.location\\n\\n        if family == \\"relation\\":\\n            if len(objects) < 2:\\n                continue\\n            indices = rng.choice(len(objects), size=2, replace=False)\\n            left, right = objects[int(indices[0])], objects[int(indices[1])]\\n            if left.location == right.location:\\n                continue\\n            left_row, left_col = divmod(LOCATIONS.index(left.location), 3)\\n            right_row, right_col = divmod(LOCATIONS.index(right.location), 3)\\n            if rng.random() < 0.5:\\n                question = f\\"is the {left.color} {left.shape} left of the {right.color} {right.shape}\\"\\n                answer = \\"yes\\" if left_col < right_col else \\"no\\"\\n            else:\\n                question = f\\"is the {left.color} {left.shape} above the {right.color} {right.shape}\\"\\n                answer = \\"yes\\" if left_row < right_row else \\"no\\"\\n            return family, question, answer\\n\\n    # Deterministic fallback that is always unambiguous.\\n    for obj in objects:\\n        if sum(item.shape == obj.shape for item in objects) == 1:\\n            return \\"attribute\\", f\\"what color is the {obj.shape}\\", obj.color\\n    obj = objects[0]\\n    return \\"location\\", f\\"where is the {obj.color} {obj.shape}\\", obj.location\\n\\n\\ndef generate_example(\\n    *,\\n    split: str,\\n    split_seed: int,\\n    example_index: int,\\n    image_size: int = 64,\\n) -> VQAExample:\\n    rng = _rng(split_seed, example_index)\\n    objects = _sample_objects(rng)\\n    family, question, answer = _build_question(rng, objects)\\n    image = render_scene(objects, image_size=image_size)\\n    return VQAExample(\\n        example_index=example_index,\\n        split=split,\\n        family=family,\\n        question=question,\\n        answer=answer,\\n        objects=objects,\\n        image=image,\\n    )\\n\\n\\nclass SyntheticVQADataset(Dataset[dict[str, Any]]):\\n    def __init__(\\n        self,\\n        *,\\n        split: Literal[\\"train\\", \\"validation\\", \\"test\\"],\\n        size: int,\\n        split_seed: int,\\n        tokenizer: VQATokenizer,\\n        image_size: int = 64,\\n        supervise_eos: bool = True,\\n    ) -> None:\\n        self.split = split\\n        self.size = size\\n        self.split_seed = split_seed\\n        self.tokenizer = tokenizer\\n        self.image_size = image_size\\n        self.supervise_eos = supervise_eos\\n\\n    def __len__(self) -> int:\\n        return self.size\\n\\n    def __getitem__(self, index: int) -> dict[str, Any]:\\n        example = generate_example(\\n            split=self.split,\\n            split_seed=self.split_seed,\\n            example_index=index,\\n            image_size=self.image_size,\\n        )\\n        encoded = self.tokenizer.encode_supervised(\\n            example.question,\\n            example.answer,\\n            supervise_eos=self.supervise_eos,\\n        )\\n        return {\\n            \\"pixel_values\\": torch.from_numpy(example.image),\\n            \\"input_ids\\": torch.tensor(encoded[\\"input_ids\\"], dtype=torch.long),\\n            \\"targets\\": torch.tensor(encoded[\\"targets\\"], dtype=torch.long),\\n            \\"answer_position\\": encoded[\\"answer_position\\"],\\n            \\"answer_id\\": encoded[\\"answer_id\\"],\\n            \\"family\\": example.family,\\n            \\"question\\": example.question,\\n            \\"answer\\": example.answer,\\n            \\"example_index\\": example.example_index,\\n        }\\n\\n\\ndef collate_vqa_batch(examples: list[dict[str, Any]], *, pad_token_id: int) -> dict[str, Any]:\\n    max_len = max(int(example[\\"input_ids\\"].numel()) for example in examples)\\n    batch_size = len(examples)\\n    input_ids = torch.full((batch_size, max_len), pad_token_id, dtype=torch.long)\\n    targets = torch.full((batch_size, max_len), -100, dtype=torch.long)\\n    answer_positions = torch.zeros(batch_size, dtype=torch.long)\\n    answer_ids = torch.zeros(batch_size, dtype=torch.long)\\n    pixel_values = torch.stack([example[\\"pixel_values\\"] for example in examples], dim=0)\\n    families: list[str] = []\\n    questions: list[str] = []\\n    answers: list[str] = []\\n    for row, example in enumerate(examples):\\n        width = int(example[\\"input_ids\\"].numel())\\n        input_ids[row, :width] = example[\\"input_ids\\"]\\n        targets[row, :width] = example[\\"targets\\"]\\n        answer_positions[row] = int(example[\\"answer_position\\"])\\n        answer_ids[row] = int(example[\\"answer_id\\"])\\n        families.append(example[\\"family\\"])\\n        questions.append(example[\\"question\\"])\\n        answers.append(example[\\"answer\\"])\\n    return {\\n        \\"pixel_values\\": pixel_values,\\n        \\"input_ids\\": input_ids,\\n        \\"targets\\": targets,\\n        \\"answer_positions\\": answer_positions,\\n        \\"answer_ids\\": answer_ids,\\n        \\"families\\": families,\\n        \\"questions\\": questions,\\n        \\"answers\\": answers,\\n    }\\n", "src/vlm/ablation/__init__.py": "from __future__ import annotations\\n\\nfrom src.vlm.ablation.config import (\\n    BLOCK_VARIANTS,\\n    PRIMARY_VARIANTS,\\n    VARIANTS,\\n    AblationExperimentConfig,\\n    resolve_experiment_config,\\n)\\nfrom src.vlm.ablation.runner import run_ablation_experiment\\n\\n__all__ = [\\n    \\"BLOCK_VARIANTS\\",\\n    \\"PRIMARY_VARIANTS\\",\\n    \\"VARIANTS\\",\\n    \\"AblationExperimentConfig\\",\\n    \\"resolve_experiment_config\\",\\n    \\"run_ablation_experiment\\",\\n]\\n", "src/vlm/ablation/config.py": "from __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\nfrom typing import Any, Literal\\n\\nfrom src.utils.config import AttnResConfig, ModelConfig\\nfrom src.models.vision_attnres import VisionConfig\\n\\nVARIANTS: dict[str, dict[str, str]] = {\\n    \\"baseline\\": {\\"encoder\\": \\"baseline\\", \\"decoder\\": \\"baseline\\"},\\n    \\"encoder_full\\": {\\"encoder\\": \\"attnres\\", \\"decoder\\": \\"baseline\\"},\\n    \\"decoder_full\\": {\\"encoder\\": \\"baseline\\", \\"decoder\\": \\"attnres\\"},\\n    \\"both_full\\": {\\"encoder\\": \\"attnres\\", \\"decoder\\": \\"attnres\\"},\\n    \\"encoder_block\\": {\\"encoder\\": \\"block_attnres\\", \\"decoder\\": \\"baseline\\"},\\n    \\"decoder_block\\": {\\"encoder\\": \\"baseline\\", \\"decoder\\": \\"block_attnres\\"},\\n    \\"both_block\\": {\\"encoder\\": \\"block_attnres\\", \\"decoder\\": \\"block_attnres\\"},\\n}\\n\\nPRIMARY_VARIANTS = [\\"baseline\\", \\"encoder_full\\", \\"decoder_full\\", \\"both_full\\"]\\nBLOCK_VARIANTS = [\\"encoder_block\\", \\"decoder_block\\", \\"both_block\\"]\\n\\nSMOKE_SIZES = {\\"train\\": 128, \\"validation\\": 64, \\"test\\": 64}\\nQUICK_SIZES = {\\"train\\": 4_000, \\"validation\\": 800, \\"test\\": 800}\\nFULL_SIZES = {\\"train\\": 12_000, \\"validation\\": 2_000, \\"test\\": 2_000}\\n\\n\\n@dataclass\\nclass AblationExperimentConfig:\\n    run_mode: Literal[\\"smoke\\", \\"quick\\", \\"full\\"] = \\"quick\\"\\n    resume: bool = True\\n    force_restart: bool = False\\n    run_primary_full_grid: bool = True\\n    run_block_grid: bool = True\\n    seeds: list[int] = field(default_factory=lambda: [0, 1, 2])\\n    primary_variants: list[str] = field(default_factory=lambda: list(PRIMARY_VARIANTS))\\n    block_variants: list[str] = field(default_factory=lambda: list(BLOCK_VARIANTS))\\n\\n    batch_size: int = 64\\n    num_workers: int = 2\\n    checkpoint_interval: int = 100\\n    evaluation_interval: int = 1\\n    max_epochs: int = 15\\n    early_stopping_patience: int = 4\\n    learning_rate: float = 3e-4\\n    weight_decay: float = 0.05\\n    beta1: float = 0.9\\n    beta2: float = 0.95\\n    grad_clip: float = 1.0\\n    warmup_fraction: float = 0.05\\n    min_lr_ratio: float = 0.1\\n    mixed_precision: bool = True\\n    amp_dtype: str = \\"auto\\"\\n    supervise_eos: bool = True\\n\\n    image_size: int = 64\\n    patch_size: int = 8\\n    vision_d_model: int = 128\\n    vision_n_layers: int = 6\\n    vision_n_heads: int = 4\\n    vision_d_ff: int = 512\\n    decoder_d_model: int = 128\\n    decoder_n_layers: int = 6\\n    decoder_n_heads: int = 4\\n    decoder_d_ff: int = 512\\n    dropout: float = 0.0\\n    num_blocks: int = 3\\n    max_seq_len: int = 64\\n    dataset_seed_offset: int = 17\\n\\n    project_root: str = \\"\\"\\n    train_size: int = 0\\n    validation_size: int = 0\\n    test_size: int = 0\\n\\n    # W&B (mirrors other notebooks; project is VLM-specific)\\n    wandb_enabled: bool = True\\n    wandb_project: str = \\"attnres-next-scale-vlm\\"\\n    wandb_entity: str = \\"atin5551-uc-davis\\"\\n    wandb_mode: str = \\"auto\\"  # auto | online | offline | disabled\\n    wandb_resume: str = \\"allow\\"  # allow | must | never\\n    wandb_tags: list[str] = field(default_factory=list)\\n    wandb_log_interval: int = 1  # log train loss every N optimizer steps (1 = every step)\\n\\n    def requested_variants(self) -> list[str]:\\n        variants: list[str] = []\\n        if self.run_primary_full_grid:\\n            variants.extend(self.primary_variants)\\n        if self.run_block_grid:\\n            variants.extend(self.block_variants)\\n        return variants\\n\\n    def to_dict(self) -> dict[str, Any]:\\n        return asdict(self)\\n\\n\\ndef dataset_sizes_for_mode(run_mode: str) -> dict[str, int]:\\n    if run_mode == \\"smoke\\":\\n        return dict(SMOKE_SIZES)\\n    if run_mode == \\"quick\\":\\n        return dict(QUICK_SIZES)\\n    if run_mode == \\"full\\":\\n        return dict(FULL_SIZES)\\n    raise ValueError(f\\"Unknown run_mode: {run_mode}\\")\\n\\n\\ndef resolve_experiment_config(config: AblationExperimentConfig) -> AblationExperimentConfig:\\n    sizes = dataset_sizes_for_mode(config.run_mode)\\n    config.train_size = sizes[\\"train\\"]\\n    config.validation_size = sizes[\\"validation\\"]\\n    config.test_size = sizes[\\"test\\"]\\n    if config.run_mode == \\"smoke\\":\\n        config.batch_size = min(config.batch_size, 16)\\n        config.max_epochs = min(config.max_epochs, 3)\\n        config.num_workers = 0\\n    return config\\n\\n\\ndef canonical_config_payload(config: AblationExperimentConfig) -> dict[str, Any]:\\n    payload = config.to_dict()\\n    for key in (\\n        \\"project_root\\",\\n        \\"resume\\",\\n        \\"force_restart\\",\\n        \\"wandb_enabled\\",\\n        \\"wandb_project\\",\\n        \\"wandb_entity\\",\\n        \\"wandb_mode\\",\\n        \\"wandb_resume\\",\\n        \\"wandb_tags\\",\\n        \\"wandb_log_interval\\",\\n    ):\\n        payload.pop(key, None)\\n    return payload\\n\\n\\ndef config_hash(config: AblationExperimentConfig) -> str:\\n    serialized = json.dumps(canonical_config_payload(config), sort_keys=True, separators=(\\",\\", \\":\\"))\\n    return hashlib.sha256(serialized.encode(\\"utf-8\\")).hexdigest()[:16]\\n\\n\\ndef build_vision_config(config: AblationExperimentConfig, residual: str) -> VisionConfig:\\n    attnres = AttnResConfig(\\n        enabled=residual != \\"baseline\\",\\n        mode=\\"block\\" if residual == \\"block_attnres\\" else \\"full\\",\\n        num_blocks=config.num_blocks if residual == \\"block_attnres\\" else None,\\n        final_readout=True,\\n        zero_init_queries=True,\\n        rmsnorm_keys=True,\\n        include_embedding=True,\\n    )\\n    return VisionConfig(\\n        image_size=config.image_size,\\n        patch_size=config.patch_size,\\n        d_model=config.vision_d_model,\\n        n_layers=config.vision_n_layers,\\n        n_heads=config.vision_n_heads,\\n        d_ff=config.vision_d_ff,\\n        dropout=config.dropout,\\n        residual=residual,  # type: ignore[arg-type]\\n        attnres=attnres,\\n    )\\n\\n\\ndef build_decoder_config(config: AblationExperimentConfig, residual: str, vocab_size: int) -> ModelConfig:\\n    attnres = AttnResConfig(\\n        enabled=residual != \\"baseline\\",\\n        mode=\\"block\\" if residual == \\"block_attnres\\" else \\"full\\",\\n        num_blocks=config.num_blocks if residual == \\"block_attnres\\" else None,\\n        final_readout=True,\\n        zero_init_queries=True,\\n        rmsnorm_keys=True,\\n        include_embedding=True,\\n    )\\n    architecture = \\"baseline\\" if residual == \\"baseline\\" else residual\\n    return ModelConfig(\\n        architecture=architecture,\\n        size_name=\\"small\\",\\n        vocab_size=vocab_size,\\n        max_seq_len=config.max_seq_len,\\n        d_model=config.decoder_d_model,\\n        n_layers=config.decoder_n_layers,\\n        n_heads=config.decoder_n_heads,\\n        d_ff=config.decoder_d_ff,\\n        dropout=config.dropout,\\n        tie_weights=False,\\n        attnres=attnres,\\n    )\\n\\n\\ndef run_dir_for(project_root: Path, variant: str, seed: int, cfg_hash: str) -> Path:\\n    return project_root / \\"runs\\" / variant / f\\"seed_{seed}\\" / cfg_hash\\n", "src/vlm/ablation/io_utils.py": "from __future__ import annotations\\n\\nimport json\\nimport os\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\n\\n\\ndef ensure_dir(path: Path | str) -> Path:\\n    path = Path(path)\\n    path.mkdir(parents=True, exist_ok=True)\\n    return path\\n\\n\\ndef atomic_write_bytes(path: Path, data: bytes) -> None:\\n    path = Path(path)\\n    ensure_dir(path.parent)\\n    tmp = path.with_name(f\\".{path.name}.tmp\\")\\n    with tmp.open(\\"wb\\") as handle:\\n        handle.write(data)\\n        handle.flush()\\n        os.fsync(handle.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef atomic_write_text(path: Path, text: str) -> None:\\n    atomic_write_bytes(path, text.encode(\\"utf-8\\"))\\n\\n\\ndef atomic_write_json(path: Path, payload: Any) -> None:\\n    atomic_write_text(path, json.dumps(payload, indent=2, sort_keys=True, default=str))\\n\\n\\ndef atomic_torch_save(path: Path, payload: Any) -> None:\\n    path = Path(path)\\n    ensure_dir(path.parent)\\n    tmp = path.with_name(f\\".{path.name}.tmp\\")\\n    torch.save(payload, tmp)\\n    with tmp.open(\\"rb\\") as handle:\\n        os.fsync(handle.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef append_jsonl(path: Path, payload: dict[str, Any]) -> None:\\n    ensure_dir(path.parent)\\n    with path.open(\\"a\\", encoding=\\"utf-8\\") as handle:\\n        handle.write(json.dumps(payload, default=str) + \\"\\\\n\\")\\n        handle.flush()\\n\\n\\ndef create_project_layout(project_root: Path) -> dict[str, Path]:\\n    names = [\\n        \\"source\\",\\n        \\"configs\\",\\n        \\"checkpoints\\",\\n        \\"runs\\",\\n        \\"logs\\",\\n        \\"metrics\\",\\n        \\"summaries\\",\\n        \\"plots\\",\\n        \\"tables\\",\\n        \\"cache\\",\\n        \\"manifests\\",\\n    ]\\n    paths = {name: ensure_dir(project_root / name) for name in names}\\n    return paths\\n", "src/vlm/ablation/source_hash.py": "from __future__ import annotations\\n\\nimport hashlib\\nfrom pathlib import Path\\nfrom typing import Iterable\\n\\n# Canonical ablation-critical modules that must stay synchronized between the\\n# repository and the notebook-written Drive source tree.\\nCANONICAL_SOURCE_RELPATHS: tuple[str, ...] = (\\n    \\"src/models/baseline.py\\",\\n    \\"src/models/attnres.py\\",\\n    \\"src/models/vision_attnres.py\\",\\n    \\"src/models/vlm_attnres.py\\",\\n    \\"src/metrics/depth_metrics.py\\",\\n    \\"src/metrics/norms.py\\",\\n    \\"src/utils/config.py\\",\\n    \\"src/utils/runtime.py\\",\\n    \\"src/vlm/synthetic_vqa.py\\",\\n    \\"src/vlm/ablation/__init__.py\\",\\n    \\"src/vlm/ablation/config.py\\",\\n    \\"src/vlm/ablation/io_utils.py\\",\\n    \\"src/vlm/ablation/source_hash.py\\",\\n    \\"src/vlm/ablation/init_sync.py\\",\\n    \\"src/vlm/ablation/manifest.py\\",\\n    \\"src/vlm/ablation/checkpoint.py\\",\\n    \\"src/vlm/ablation/eval.py\\",\\n    \\"src/vlm/ablation/routing.py\\",\\n    \\"src/vlm/ablation/train.py\\",\\n    \\"src/vlm/ablation/correctness.py\\",\\n    \\"src/vlm/ablation/aggregate.py\\",\\n    \\"src/vlm/ablation/plots.py\\",\\n    \\"src/vlm/ablation/runner.py\\",\\n    \\"src/vlm/ablation/wandb_logger.py\\",\\n)\\n\\n\\ndef file_sha256(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        while True:\\n            chunk = handle.read(1024 * 1024)\\n            if not chunk:\\n                break\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef hash_source_tree(root: Path, relpaths: Iterable[str] = CANONICAL_SOURCE_RELPATHS) -> dict[str, str]:\\n    payload: dict[str, str] = {}\\n    for relpath in relpaths:\\n        path = Path(root) / relpath\\n        if not path.exists():\\n            payload[relpath] = \\"MISSING\\"\\n            continue\\n        payload[relpath] = file_sha256(path)\\n    return payload\\n\\n\\ndef combined_source_hash(hashes: dict[str, str]) -> str:\\n    material = json_dumps_sorted(hashes).encode(\\"utf-8\\")\\n    return hashlib.sha256(material).hexdigest()[:16]\\n\\n\\ndef json_dumps_sorted(payload: dict[str, str]) -> str:\\n    import json\\n\\n    return json.dumps(payload, sort_keys=True, separators=(\\",\\", \\":\\"))\\n\\n\\ndef compare_source_hashes(\\n    left_root: Path,\\n    right_root: Path,\\n    *,\\n    relpaths: Iterable[str] = CANONICAL_SOURCE_RELPATHS,\\n) -> dict[str, object]:\\n    left = hash_source_tree(left_root, relpaths)\\n    right = hash_source_tree(right_root, relpaths)\\n    mismatches = {\\n        relpath: {\\"left\\": left[relpath], \\"right\\": right[relpath]}\\n        for relpath in left\\n        if left.get(relpath) != right.get(relpath)\\n    }\\n    return {\\n        \\"left_hash\\": combined_source_hash(left),\\n        \\"right_hash\\": combined_source_hash(right),\\n        \\"match\\": not mismatches,\\n        \\"mismatches\\": mismatches,\\n        \\"left\\": left,\\n        \\"right\\": right,\\n    }\\n", "src/vlm/ablation/init_sync.py": "from __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport torch\\nimport torch.nn as nn\\n\\nfrom src.models.attnres import DepthAttentionResidual\\nfrom src.models.vlm_attnres import TinyAttnResVLM\\n\\n\\ndef _is_attnres_param(name: str) -> bool:\\n    lowered = name.lower()\\n    return (\\n        \\"pre_attn_res\\" in lowered\\n        or \\"pre_mlp_res\\" in lowered\\n        or \\"final_residual\\" in lowered\\n        or name.endswith(\\".query\\")\\n        or \\".key_norm.\\" in lowered\\n    )\\n\\n\\ndef shared_parameter_pairs(reference: TinyAttnResVLM, candidate: TinyAttnResVLM) -> list[tuple[str, str]]:\\n    \\"\\"\\"Map shared parameter names between a baseline-compatible reference and a variant.\\"\\"\\"\\n    ref_names = {name for name, _ in reference.named_parameters() if not _is_attnres_param(name)}\\n    cand_names = {name for name, _ in candidate.named_parameters() if not _is_attnres_param(name)}\\n    shared = sorted(ref_names & cand_names)\\n    return [(name, name) for name in shared]\\n\\n\\ndef copy_shared_weights(reference: TinyAttnResVLM, candidate: TinyAttnResVLM) -> int:\\n    ref_state = reference.state_dict()\\n    cand_state = candidate.state_dict()\\n    copied = 0\\n    for ref_name, cand_name in shared_parameter_pairs(reference, candidate):\\n        if ref_name in ref_state and cand_name in cand_state:\\n            if cand_state[cand_name].shape == ref_state[ref_name].shape:\\n                cand_state[cand_name] = ref_state[ref_name].clone()\\n                copied += 1\\n    candidate.load_state_dict(cand_state, strict=False)\\n    return copied\\n\\n\\n@torch.no_grad()\\ndef validate_shared_initialization(\\n    reference: TinyAttnResVLM,\\n    candidate: TinyAttnResVLM,\\n    *,\\n    atol: float = 0.0,\\n    rtol: float = 0.0,\\n) -> dict[str, Any]:\\n    mismatches: list[str] = []\\n    checked = 0\\n    ref_params = dict(reference.named_parameters())\\n    cand_params = dict(candidate.named_parameters())\\n    for ref_name, cand_name in shared_parameter_pairs(reference, candidate):\\n        left = ref_params[ref_name]\\n        right = cand_params[cand_name]\\n        checked += 1\\n        if left.shape != right.shape or not torch.allclose(left, right, atol=atol, rtol=rtol):\\n            mismatches.append(ref_name)\\n    if mismatches:\\n        raise ValueError(\\n            \\"Shared initialization mismatch for parameters: \\" + \\", \\".join(mismatches[:20])\\n        )\\n    return {\\n        \\"checked\\": checked,\\n        \\"mismatches\\": mismatches,\\n        \\"ok\\": True,\\n        \\"reference_encoder\\": reference.encoder_residual,\\n        \\"reference_decoder\\": reference.decoder_residual,\\n        \\"candidate_encoder\\": candidate.encoder_residual,\\n        \\"candidate_decoder\\": candidate.decoder_residual,\\n    }\\n\\n\\ndef assert_encoder_decoder_attnres_separate(model: TinyAttnResVLM) -> dict[str, Any]:\\n    encoder_ids = {id(module) for module in model.iter_encoder_depth_residuals()}\\n    decoder_ids = {id(module) for module in model.iter_decoder_depth_residuals()}\\n    overlap = encoder_ids & decoder_ids\\n    if overlap:\\n        raise ValueError(\\"Encoder and decoder DepthAttentionResidual modules must be separate\\")\\n    encoder_params = {id(param) for param in model.encoder.parameters()}\\n    decoder_params = {id(param) for param in model.decoder.parameters()}\\n    if encoder_params & decoder_params:\\n        raise ValueError(\\"Encoder and decoder parameters must not be shared\\")\\n    return {\\n        \\"encoder_residual_modules\\": len(encoder_ids),\\n        \\"decoder_residual_modules\\": len(decoder_ids),\\n        \\"ok\\": True,\\n    }\\n\\n\\ndef count_attnres_parameters(model: nn.Module) -> int:\\n    total = 0\\n    for name, param in model.named_parameters():\\n        if _is_attnres_param(name) or isinstance(param, nn.Parameter) and name.endswith(\\"query\\"):\\n            # Count DepthAttentionResidual owned params via module walk as well.\\n            pass\\n    for module in model.modules():\\n        if isinstance(module, DepthAttentionResidual):\\n            total += sum(param.numel() for param in module.parameters())\\n    return total\\n", "src/vlm/ablation/manifest.py": "from __future__ import annotations\\n\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom src.vlm.ablation.io_utils import atomic_write_json, ensure_dir\\n\\n\\nclass ExperimentManifest:\\n    def __init__(self, path: Path) -> None:\\n        self.path = Path(path)\\n        ensure_dir(self.path.parent)\\n        self.data: dict[str, Any] = {\\"runs\\": {}, \\"updated_at\\": None}\\n        if self.path.exists():\\n            import json\\n\\n            self.data = json.loads(self.path.read_text(encoding=\\"utf-8\\"))\\n\\n    def run_key(self, variant: str, seed: int, config_hash: str) -> str:\\n        return f\\"{variant}/seed_{seed}/{config_hash}\\"\\n\\n    def get(self, variant: str, seed: int, config_hash: str) -> dict[str, Any] | None:\\n        return self.data[\\"runs\\"].get(self.run_key(variant, seed, config_hash))\\n\\n    def upsert(self, variant: str, seed: int, config_hash: str, **fields: Any) -> dict[str, Any]:\\n        key = self.run_key(variant, seed, config_hash)\\n        row = dict(self.data[\\"runs\\"].get(key, {}))\\n        now = time.strftime(\\"%Y-%m-%dT%H:%M:%SZ\\", time.gmtime())\\n        if \\"start_time\\" not in row and fields.get(\\"status\\") == \\"running\\":\\n            row[\\"start_time\\"] = now\\n        row.update(fields)\\n        row[\\"variant\\"] = variant\\n        row[\\"seed\\"] = seed\\n        row[\\"config_hash\\"] = config_hash\\n        row[\\"last_update_time\\"] = now\\n        if fields.get(\\"status\\") == \\"completed\\":\\n            row[\\"completion_time\\"] = now\\n        self.data[\\"runs\\"][key] = row\\n        self.data[\\"updated_at\\"] = now\\n        self.save()\\n        return row\\n\\n    def save(self) -> None:\\n        atomic_write_json(self.path, self.data)\\n\\n    def summarize(self) -> dict[str, list[str]]:\\n        buckets = {\\n            \\"pending\\": [],\\n            \\"running\\": [],\\n            \\"interrupted\\": [],\\n            \\"completed\\": [],\\n            \\"failed\\": [],\\n        }\\n        for key, row in self.data[\\"runs\\"].items():\\n            status = str(row.get(\\"status\\", \\"pending\\"))\\n            buckets.setdefault(status, []).append(key)\\n        return buckets\\n", "src/vlm/ablation/checkpoint.py": "from __future__ import annotations\\n\\nimport shutil\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\n\\nfrom src.utils.runtime import get_rng_state, set_rng_state\\nfrom src.vlm.ablation.io_utils import atomic_torch_save, atomic_write_json, ensure_dir\\n\\n\\ndef archive_run_dir(run_dir: Path) -> Path:\\n    run_dir = Path(run_dir)\\n    stamp = time.strftime(\\"%Y%m%dT%H%M%SZ\\", time.gmtime())\\n    archive = run_dir.parent / f\\"{run_dir.name}_archived_{stamp}\\"\\n    if run_dir.exists():\\n        shutil.move(str(run_dir), str(archive))\\n    return archive\\n\\n\\ndef build_checkpoint_payload(\\n    *,\\n    model: torch.nn.Module,\\n    optimizer: torch.optim.Optimizer,\\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\\n    scaler: torch.amp.GradScaler,\\n    epoch: int,\\n    batch_index: int,\\n    global_step: int,\\n    best_val_accuracy: float,\\n    early_stopping_bad_epochs: int,\\n    examples_processed: int,\\n    elapsed_training_time: float,\\n    model_config: dict[str, Any],\\n    dataset_config: dict[str, Any],\\n    tokenizer_vocab: dict[str, int],\\n    variant: str,\\n    seed: int,\\n    config_hash: str,\\n    source_code_hash: str,\\n    status: str = \\"running\\",\\n) -> dict[str, Any]:\\n    return {\\n        \\"model\\": model.state_dict(),\\n        \\"optimizer\\": optimizer.state_dict(),\\n        \\"scheduler\\": scheduler.state_dict(),\\n        \\"scaler\\": scaler.state_dict() if scaler.is_enabled() else None,\\n        \\"epoch\\": epoch,\\n        \\"batch_index\\": batch_index,\\n        \\"global_step\\": global_step,\\n        \\"best_val_accuracy\\": best_val_accuracy,\\n        \\"early_stopping_bad_epochs\\": early_stopping_bad_epochs,\\n        \\"examples_processed\\": examples_processed,\\n        \\"elapsed_training_time\\": elapsed_training_time,\\n        \\"rng_state\\": get_rng_state(),\\n        \\"model_config\\": model_config,\\n        \\"dataset_config\\": dataset_config,\\n        \\"tokenizer_vocab\\": tokenizer_vocab,\\n        \\"variant\\": variant,\\n        \\"seed\\": seed,\\n        \\"config_hash\\": config_hash,\\n        \\"source_code_hash\\": source_code_hash,\\n        \\"status\\": status,\\n        \\"saved_at\\": time.strftime(\\"%Y-%m-%dT%H:%M:%SZ\\", time.gmtime()),\\n    }\\n\\n\\ndef save_checkpoint(path: Path, payload: dict[str, Any]) -> None:\\n    atomic_torch_save(path, payload)\\n\\n\\ndef load_checkpoint(path: Path, map_location: str | torch.device = \\"cpu\\") -> dict[str, Any]:\\n    try:\\n        return torch.load(path, map_location=map_location, weights_only=False)\\n    except TypeError:\\n        return torch.load(path, map_location=map_location)\\n\\n\\ndef restore_training_state(\\n    checkpoint: dict[str, Any],\\n    *,\\n    model: torch.nn.Module,\\n    optimizer: torch.optim.Optimizer,\\n    scheduler: torch.optim.lr_scheduler.LRScheduler,\\n    scaler: torch.amp.GradScaler,\\n) -> None:\\n    model.load_state_dict(checkpoint[\\"model\\"])\\n    optimizer.load_state_dict(checkpoint[\\"optimizer\\"])\\n    scheduler.load_state_dict(checkpoint[\\"scheduler\\"])\\n    if checkpoint.get(\\"scaler\\") is not None and scaler.is_enabled():\\n        scaler.load_state_dict(checkpoint[\\"scaler\\"])\\n    if \\"rng_state\\" in checkpoint:\\n        set_rng_state(checkpoint[\\"rng_state\\"])\\n\\n\\ndef validate_checkpoint_compatibility(\\n    checkpoint: dict[str, Any],\\n    *,\\n    variant: str,\\n    seed: int,\\n    config_hash: str,\\n    source_code_hash: str | None = None,\\n    force_restart: bool = False,\\n) -> None:\\n    if force_restart:\\n        return\\n    expected = {\\n        \\"variant\\": variant,\\n        \\"seed\\": seed,\\n        \\"config_hash\\": config_hash,\\n    }\\n    for key, value in expected.items():\\n        if checkpoint.get(key) != value:\\n            raise ValueError(\\n                f\\"Incompatible checkpoint for {key}: expected {value}, found {checkpoint.get(key)}\\"\\n            )\\n    if source_code_hash is not None and checkpoint.get(\\"source_code_hash\\") not in {None, source_code_hash}:\\n        raise ValueError(\\n            \\"Checkpoint source_code_hash mismatch. Set FORCE_RESTART=True to archive and restart.\\"\\n        )\\n\\n\\ndef write_status(run_dir: Path, payload: dict[str, Any]) -> None:\\n    atomic_write_json(ensure_dir(run_dir) / \\"status.json\\", payload)\\n\\n\\ndef mark_completed(run_dir: Path) -> None:\\n    (ensure_dir(run_dir) / \\"completed.marker\\").write_text(\\"ok\\\\n\\", encoding=\\"utf-8\\")\\n", "src/vlm/ablation/eval.py": "from __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom contextlib import nullcontext\\nfrom typing import Any\\n\\nimport torch\\nfrom torch.utils.data import DataLoader\\n\\nfrom src.models.vlm_attnres import TinyAttnResVLM\\nfrom src.vlm.ablation.routing import collect_routing_batch_stats\\n\\n\\n@torch.no_grad()\\ndef evaluate_model(\\n    model: TinyAttnResVLM,\\n    loader: DataLoader,\\n    *,\\n    device: torch.device,\\n    amp_dtype: torch.dtype | None = None,\\n    capture_routing: bool = False,\\n    max_batches: int | None = None,\\n) -> dict[str, Any]:\\n    model.eval()\\n    if capture_routing:\\n        model.set_weight_capture(True)\\n\\n    total_loss = 0.0\\n    total_examples = 0\\n    correct = 0\\n    family_correct: dict[str, int] = defaultdict(int)\\n    family_total: dict[str, int] = defaultdict(int)\\n    routing_rows: list[dict[str, Any]] = []\\n\\n    autocast_enabled = device.type == \\"cuda\\" and amp_dtype is not None\\n    for batch_index, batch in enumerate(loader):\\n        if max_batches is not None and batch_index >= max_batches:\\n            break\\n        pixel_values = batch[\\"pixel_values\\"].to(device)\\n        input_ids = batch[\\"input_ids\\"].to(device)\\n        targets = batch[\\"targets\\"].to(device)\\n        answer_positions = batch[\\"answer_positions\\"].to(device)\\n        answer_ids = batch[\\"answer_ids\\"].to(device)\\n        families = batch[\\"families\\"]\\n\\n        autocast = (\\n            torch.autocast(device_type=\\"cuda\\", dtype=amp_dtype, enabled=True)\\n            if autocast_enabled\\n            else nullcontext()\\n        )\\n        with autocast:\\n            output = model(\\n                pixel_values=pixel_values,\\n                input_ids=input_ids,\\n                targets=targets,\\n                return_aux=capture_routing,\\n            )\\n        loss = output[\\"loss\\"]\\n        logits = output[\\"logits\\"]\\n        preds = logits.argmax(dim=-1)\\n        batch_size = input_ids.size(0)\\n        total_loss += float(loss.item()) * batch_size\\n        total_examples += batch_size\\n\\n        for row in range(batch_size):\\n            position = int(answer_positions[row].item())\\n            pred_id = int(preds[row, position].item())\\n            target_id = int(answer_ids[row].item())\\n            family = families[row]\\n            family_total[family] += 1\\n            if pred_id == target_id:\\n                correct += 1\\n                family_correct[family] += 1\\n\\n        if capture_routing:\\n            routing_rows.append(\\n                collect_routing_batch_stats(\\n                    model,\\n                    families=families,\\n                    prefix_length=int(output[\\"prefix_length\\"]),\\n                    text_length=input_ids.size(1),\\n                )\\n            )\\n\\n    if capture_routing:\\n        model.set_weight_capture(False)\\n\\n    accuracy = correct / max(1, total_examples)\\n    family_accuracy = {\\n        family: family_correct[family] / max(1, family_total[family])\\n        for family in sorted(family_total)\\n    }\\n    return {\\n        \\"loss\\": total_loss / max(1, total_examples),\\n        \\"accuracy\\": accuracy,\\n        \\"correct\\": correct,\\n        \\"total\\": total_examples,\\n        \\"family_accuracy\\": family_accuracy,\\n        \\"family_total\\": dict(family_total),\\n        \\"routing\\": routing_rows,\\n    }\\n", "src/vlm/ablation/routing.py": "from __future__ import annotations\\n\\nfrom collections import defaultdict\\nfrom typing import Any\\n\\nimport torch\\n\\nfrom src.models.vlm_attnres import TinyAttnResVLM\\n\\n\\ndef _stats_from_weights(weights: torch.Tensor, source_indices: list[int]) -> dict[str, Any]:\\n    # weights: [sources, batch, tokens]\\n    mean_weights = weights.float().mean(dim=(1, 2))\\n    entropy = -(weights.float() * weights.float().clamp_min(1e-8).log()).sum(dim=0).mean()\\n    max_prob = weights.float().amax(dim=0).mean()\\n    effective = torch.exp(entropy)\\n    embedding = 0.0\\n    if 0 in source_indices:\\n        embedding = float(mean_weights[source_indices.index(0)].item())\\n    return {\\n        \\"source_indices\\": list(source_indices),\\n        \\"mean_weights\\": mean_weights.cpu().tolist(),\\n        \\"entropy\\": float(entropy.item()),\\n        \\"max_source_prob\\": float(max_prob.item()),\\n        \\"effective_sources\\": float(effective.item()),\\n        \\"embedding_contribution\\": embedding,\\n    }\\n\\n\\ndef _site_family_stats(\\n    weights: torch.Tensor,\\n    source_indices: list[int],\\n    families: list[str],\\n) -> dict[str, dict[str, Any]]:\\n    payload: dict[str, dict[str, Any]] = {}\\n    for family in sorted(set(families)):\\n        mask = torch.tensor([item == family for item in families], dtype=torch.bool)\\n        if not bool(mask.any()):\\n            continue\\n        family_weights = weights[:, mask, :]\\n        payload[family] = _stats_from_weights(family_weights, source_indices)\\n    return payload\\n\\n\\ndef collect_routing_batch_stats(\\n    model: TinyAttnResVLM,\\n    *,\\n    families: list[str],\\n    prefix_length: int,\\n    text_length: int,\\n) -> dict[str, Any]:\\n    encoder_sites: list[dict[str, Any]] = []\\n    decoder_sites: list[dict[str, Any]] = []\\n\\n    for site_index, residual in enumerate(model.iter_encoder_depth_residuals()):\\n        if residual.last_weights is None:\\n            continue\\n        weights = residual.last_weights\\n        indices = residual.last_source_indices\\n        site = {\\n            \\"site_index\\": site_index,\\n            **_stats_from_weights(weights, indices),\\n            \\"by_family\\": _site_family_stats(weights, indices, families),\\n        }\\n        encoder_sites.append(site)\\n\\n    for site_index, residual in enumerate(model.iter_decoder_depth_residuals()):\\n        if residual.last_weights is None:\\n            continue\\n        weights = residual.last_weights\\n        indices = residual.last_source_indices\\n        seq_len = int(weights.size(2))\\n        vision_slice = weights[:, :, : min(prefix_length, seq_len)]\\n        text_slice = weights[:, :, prefix_length : prefix_length + text_length]\\n        site = {\\n            \\"site_index\\": site_index,\\n            **_stats_from_weights(weights, indices),\\n            \\"by_family\\": _site_family_stats(weights, indices, families),\\n            \\"visual_prefix\\": _stats_from_weights(vision_slice, indices) if vision_slice.numel() else {},\\n            \\"text_positions\\": _stats_from_weights(text_slice, indices) if text_slice.numel() else {},\\n        }\\n        decoder_sites.append(site)\\n\\n    return {\\n        \\"encoder_routing\\": encoder_sites,\\n        \\"decoder_routing\\": decoder_sites,\\n    }\\n\\n\\ndef aggregate_routing_rows(rows: list[dict[str, Any]]) -> dict[str, Any]:\\n    def _aggregate_sites(namespace: str) -> list[dict[str, Any]]:\\n        if not rows:\\n            return []\\n        n_sites = max((len(row.get(namespace, [])) for row in rows), default=0)\\n        aggregated: list[dict[str, Any]] = []\\n        for site_index in range(n_sites):\\n            entropies: list[float] = []\\n            max_probs: list[float] = []\\n            effective: list[float] = []\\n            embeddings: list[float] = []\\n            mean_weight_sums: list[list[float]] = []\\n            source_indices: list[int] = []\\n            family_buckets: dict[str, dict[str, list[float]]] = defaultdict(\\n                lambda: {\\"entropy\\": [], \\"embedding\\": [], \\"max_source_prob\\": []}\\n            )\\n            for row in rows:\\n                sites = row.get(namespace, [])\\n                if site_index >= len(sites):\\n                    continue\\n                site = sites[site_index]\\n                source_indices = site.get(\\"source_indices\\", source_indices)\\n                entropies.append(float(site[\\"entropy\\"]))\\n                max_probs.append(float(site[\\"max_source_prob\\"]))\\n                effective.append(float(site[\\"effective_sources\\"]))\\n                embeddings.append(float(site[\\"embedding_contribution\\"]))\\n                mean_weight_sums.append([float(value) for value in site[\\"mean_weights\\"]])\\n                for family, family_stats in site.get(\\"by_family\\", {}).items():\\n                    family_buckets[family][\\"entropy\\"].append(float(family_stats[\\"entropy\\"]))\\n                    family_buckets[family][\\"embedding\\"].append(float(family_stats[\\"embedding_contribution\\"]))\\n                    family_buckets[family][\\"max_source_prob\\"].append(float(family_stats[\\"max_source_prob\\"]))\\n            if not mean_weight_sums:\\n                continue\\n            width = len(mean_weight_sums[0])\\n            mean_weights = [\\n                sum(row[index] for row in mean_weight_sums) / len(mean_weight_sums)\\n                for index in range(width)\\n            ]\\n            aggregated.append(\\n                {\\n                    \\"site_index\\": site_index,\\n                    \\"source_indices\\": source_indices,\\n                    \\"mean_weights\\": mean_weights,\\n                    \\"entropy\\": sum(entropies) / max(1, len(entropies)),\\n                    \\"max_source_prob\\": sum(max_probs) / max(1, len(max_probs)),\\n                    \\"effective_sources\\": sum(effective) / max(1, len(effective)),\\n                    \\"embedding_contribution\\": sum(embeddings) / max(1, len(embeddings)),\\n                    \\"by_family\\": {\\n                        family: {\\n                            key: sum(values) / max(1, len(values))\\n                            for key, values in bucket.items()\\n                        }\\n                        for family, bucket in family_buckets.items()\\n                    },\\n                }\\n            )\\n        return aggregated\\n\\n    return {\\n        \\"encoder_routing\\": _aggregate_sites(\\"encoder_routing\\"),\\n        \\"decoder_routing\\": _aggregate_sites(\\"decoder_routing\\"),\\n    }\\n", "src/vlm/ablation/train.py": "from __future__ import annotations\\n\\nimport json\\nimport math\\nimport time\\nfrom contextlib import nullcontext\\nfrom functools import partial\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\nfrom torch.optim import AdamW\\nfrom torch.utils.data import DataLoader\\n\\nfrom src.models.vlm_attnres import TinyAttnResVLM\\nfrom src.utils.runtime import amp_dtype_from_string, count_parameters, seed_everything\\nfrom src.vlm.ablation.checkpoint import (\\n    archive_run_dir,\\n    build_checkpoint_payload,\\n    load_checkpoint,\\n    mark_completed,\\n    restore_training_state,\\n    save_checkpoint,\\n    validate_checkpoint_compatibility,\\n    write_status,\\n)\\nfrom src.vlm.ablation.config import (\\n    AblationExperimentConfig,\\n    VARIANTS,\\n    build_decoder_config,\\n    build_vision_config,\\n    config_hash,\\n    run_dir_for,\\n)\\nfrom src.vlm.ablation.eval import evaluate_model\\nfrom src.vlm.ablation.init_sync import copy_shared_weights, validate_shared_initialization\\nfrom src.vlm.ablation.io_utils import append_jsonl, atomic_write_json, ensure_dir\\nfrom src.vlm.ablation.manifest import ExperimentManifest\\nfrom src.vlm.ablation.routing import aggregate_routing_rows\\nfrom src.vlm.ablation.wandb_logger import AblationWandbLogger\\nfrom src.vlm.synthetic_vqa import SyntheticVQADataset, VQATokenizer, collate_vqa_batch\\n\\n\\ndef _select_amp_dtype(requested: str, device: torch.device) -> torch.dtype | None:\\n    if device.type != \\"cuda\\":\\n        return None\\n    if requested == \\"auto\\":\\n        major, _ = torch.cuda.get_device_capability(device)\\n        # bf16 is reliable on Ampere+ (sm80+); T4 is sm75 -> float16.\\n        return torch.bfloat16 if major >= 8 else torch.float16\\n    return amp_dtype_from_string(requested)\\n\\n\\ndef build_scheduler(\\n    optimizer: torch.optim.Optimizer,\\n    *,\\n    warmup_steps: int,\\n    total_steps: int,\\n    min_lr_ratio: float,\\n) -> torch.optim.lr_scheduler.LambdaLR:\\n    def schedule(step: int) -> float:\\n        if step < warmup_steps:\\n            return float(step + 1) / max(1, warmup_steps)\\n        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)\\n        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))\\n        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine\\n\\n    return torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)\\n\\n\\ndef build_dataloaders(\\n    config: AblationExperimentConfig,\\n    tokenizer: VQATokenizer,\\n) -> dict[str, DataLoader]:\\n    collate = partial(collate_vqa_batch, pad_token_id=tokenizer.pad_token_id)\\n    loaders: dict[str, DataLoader] = {}\\n    split_seeds = {\\n        \\"train\\": config.dataset_seed_offset,\\n        \\"validation\\": config.dataset_seed_offset + 1,\\n        \\"test\\": config.dataset_seed_offset + 2,\\n    }\\n    sizes = {\\n        \\"train\\": config.train_size,\\n        \\"validation\\": config.validation_size,\\n        \\"test\\": config.test_size,\\n    }\\n    for split, size in sizes.items():\\n        dataset = SyntheticVQADataset(\\n            split=split,  # type: ignore[arg-type]\\n            size=size,\\n            split_seed=split_seeds[split],\\n            tokenizer=tokenizer,\\n            image_size=config.image_size,\\n            supervise_eos=config.supervise_eos,\\n        )\\n        loaders[split] = DataLoader(\\n            dataset,\\n            batch_size=config.batch_size,\\n            shuffle=(split == \\"train\\"),\\n            num_workers=config.num_workers,\\n            collate_fn=collate,\\n            pin_memory=True,\\n        )\\n    return loaders\\n\\n\\ndef build_model_for_variant(\\n    config: AblationExperimentConfig,\\n    *,\\n    variant: str,\\n    vocab_size: int,\\n    seed: int,\\n    reference: TinyAttnResVLM | None = None,\\n) -> tuple[TinyAttnResVLM, dict[str, Any]]:\\n    residual = VARIANTS[variant]\\n    seed_everything(seed, deterministic=True)\\n    vision_config = build_vision_config(config, residual[\\"encoder\\"])\\n    decoder_config = build_decoder_config(config, residual[\\"decoder\\"], vocab_size=vocab_size)\\n    model = TinyAttnResVLM(\\n        vision_config=vision_config,\\n        decoder_config=decoder_config,\\n        encoder_residual=residual[\\"encoder\\"],\\n        decoder_residual=residual[\\"decoder\\"],\\n    )\\n    init_meta: dict[str, Any] = {\\"copied_shared_tensors\\": 0, \\"shared_init\\": None}\\n    if reference is not None:\\n        init_meta[\\"copied_shared_tensors\\"] = copy_shared_weights(reference, model)\\n        init_meta[\\"shared_init\\"] = validate_shared_initialization(reference, model)\\n    return model, init_meta\\n\\n\\ndef train_variant_seed(\\n    config: AblationExperimentConfig,\\n    *,\\n    variant: str,\\n    seed: int,\\n    project_root: Path,\\n    manifest: ExperimentManifest,\\n    source_code_hash: str,\\n    device: torch.device,\\n) -> dict[str, Any]:\\n    cfg_hash = config_hash(config)\\n    run_dir = run_dir_for(project_root, variant, seed, cfg_hash)\\n    completed_marker = run_dir / \\"completed.marker\\"\\n    last_path = run_dir / \\"last.pt\\"\\n    best_path = run_dir / \\"best.pt\\"\\n\\n    if completed_marker.exists() and config.resume and not config.force_restart:\\n        metrics_path = run_dir / \\"final_test_metrics.json\\"\\n        metrics = json.loads(metrics_path.read_text(encoding=\\"utf-8\\")) if metrics_path.exists() else {}\\n        manifest.upsert(\\n            variant,\\n            seed,\\n            cfg_hash,\\n            status=\\"completed\\",\\n            run_directory=str(run_dir),\\n            latest_checkpoint=str(last_path),\\n            best_checkpoint=str(best_path),\\n            best_validation_accuracy=metrics.get(\\"validation_accuracy\\"),\\n        )\\n        return {\\"status\\": \\"skipped_completed\\", \\"run_dir\\": str(run_dir), \\"metrics\\": metrics}\\n\\n    if config.force_restart and run_dir.exists():\\n        archive_run_dir(run_dir)\\n\\n    ensure_dir(run_dir)\\n    tokenizer = VQATokenizer()\\n    loaders = build_dataloaders(config, tokenizer)\\n\\n    seed_everything(seed, deterministic=True)\\n    reference, _ = build_model_for_variant(\\n        config,\\n        variant=\\"baseline\\",\\n        vocab_size=tokenizer.vocab_size,\\n        seed=seed,\\n    )\\n    model, init_meta = build_model_for_variant(\\n        config,\\n        variant=variant,\\n        vocab_size=tokenizer.vocab_size,\\n        seed=seed,\\n        reference=reference if variant != \\"baseline\\" else None,\\n    )\\n    if variant == \\"baseline\\":\\n        init_meta[\\"shared_init\\"] = validate_shared_initialization(reference, model)\\n\\n    model = model.to(device)\\n    amp_dtype = _select_amp_dtype(config.amp_dtype, device)\\n    optimizer = AdamW(\\n        model.parameters(),\\n        lr=config.learning_rate,\\n        weight_decay=config.weight_decay,\\n        betas=(config.beta1, config.beta2),\\n    )\\n    steps_per_epoch = max(1, len(loaders[\\"train\\"]))\\n    total_steps = steps_per_epoch * config.max_epochs\\n    warmup_steps = max(1, int(total_steps * config.warmup_fraction))\\n    scheduler = build_scheduler(\\n        optimizer,\\n        warmup_steps=warmup_steps,\\n        total_steps=total_steps,\\n        min_lr_ratio=config.min_lr_ratio,\\n    )\\n    scaler = torch.amp.GradScaler(\\n        \\"cuda\\",\\n        enabled=device.type == \\"cuda\\" and config.mixed_precision and amp_dtype == torch.float16,\\n    )\\n\\n    start_epoch = 0\\n    global_step = 0\\n    best_val_accuracy = -1.0\\n    bad_epochs = 0\\n    examples_processed = 0\\n    elapsed_training_time = 0.0\\n    resumed = False\\n\\n    if config.resume and last_path.exists() and not config.force_restart:\\n        checkpoint = load_checkpoint(last_path, map_location=device)\\n        validate_checkpoint_compatibility(\\n            checkpoint,\\n            variant=variant,\\n            seed=seed,\\n            config_hash=cfg_hash,\\n            source_code_hash=source_code_hash,\\n            force_restart=False,\\n        )\\n        restore_training_state(\\n            checkpoint,\\n            model=model,\\n            optimizer=optimizer,\\n            scheduler=scheduler,\\n            scaler=scaler,\\n        )\\n        start_epoch = int(checkpoint[\\"epoch\\"])\\n        global_step = int(checkpoint[\\"global_step\\"])\\n        best_val_accuracy = float(checkpoint[\\"best_val_accuracy\\"])\\n        bad_epochs = int(checkpoint[\\"early_stopping_bad_epochs\\"])\\n        examples_processed = int(checkpoint[\\"examples_processed\\"])\\n        elapsed_training_time = float(checkpoint[\\"elapsed_training_time\\"])\\n        resumed = True\\n\\n    param_counts = count_parameters(model)\\n    baseline_params = count_parameters(reference)\\n    param_increase_pct = 100.0 * (param_counts[\\"total\\"] - baseline_params[\\"total\\"]) / max(1, baseline_params[\\"total\\"])\\n\\n    wandb_logger = AblationWandbLogger(\\n        config=config,\\n        variant=variant,\\n        seed=seed,\\n        config_hash=cfg_hash,\\n        run_dir=run_dir,\\n        extra_config={\\n            \\"parameter_count\\": param_counts,\\n            \\"parameter_increase_pct\\": param_increase_pct,\\n            \\"amp_dtype\\": str(amp_dtype),\\n            \\"source_code_hash\\": source_code_hash,\\n            \\"resumed\\": resumed,\\n        },\\n    )\\n    wandb_logger.update_summary(\\n        {\\n            \\"variant\\": variant,\\n            \\"seed\\": seed,\\n            \\"encoder_residual\\": VARIANTS[variant][\\"encoder\\"],\\n            \\"decoder_residual\\": VARIANTS[variant][\\"decoder\\"],\\n            \\"parameter_count\\": param_counts[\\"total\\"],\\n            \\"parameter_increase_pct\\": param_increase_pct,\\n            \\"device\\": str(device),\\n            \\"amp_dtype\\": str(amp_dtype),\\n            \\"resumed\\": resumed,\\n            **wandb_logger.metadata(),\\n        }\\n    )\\n\\n    atomic_write_json(\\n        run_dir / \\"config.json\\",\\n        {\\n            \\"experiment\\": config.to_dict(),\\n            \\"variant\\": variant,\\n            \\"seed\\": seed,\\n            \\"config_hash\\": cfg_hash,\\n            \\"source_code_hash\\": source_code_hash,\\n            \\"init_validation\\": init_meta,\\n            \\"parameter_count\\": param_counts,\\n            \\"parameter_increase_pct\\": param_increase_pct,\\n            \\"amp_dtype\\": str(amp_dtype),\\n            \\"wandb\\": wandb_logger.metadata(),\\n        },\\n    )\\n\\n    manifest.upsert(\\n        variant,\\n        seed,\\n        cfg_hash,\\n        status=\\"running\\",\\n        run_directory=str(run_dir),\\n        latest_checkpoint=str(last_path) if last_path.exists() else None,\\n        best_checkpoint=str(best_path) if best_path.exists() else None,\\n        current_epoch=start_epoch,\\n        global_step=global_step,\\n        best_validation_accuracy=best_val_accuracy if best_val_accuracy >= 0 else None,\\n    )\\n\\n    def _save(path: Path, *, epoch: int, batch_index: int, status: str = \\"running\\") -> None:\\n        payload = build_checkpoint_payload(\\n            model=model,\\n            optimizer=optimizer,\\n            scheduler=scheduler,\\n            scaler=scaler,\\n            epoch=epoch,\\n            batch_index=batch_index,\\n            global_step=global_step,\\n            best_val_accuracy=best_val_accuracy,\\n            early_stopping_bad_epochs=bad_epochs,\\n            examples_processed=examples_processed,\\n            elapsed_training_time=elapsed_training_time,\\n            model_config={\\n                \\"encoder_residual\\": model.encoder_residual,\\n                \\"decoder_residual\\": model.decoder_residual,\\n                \\"vision\\": model.vision_config.__dict__,\\n                \\"decoder\\": model.decoder_config.__dict__,\\n            },\\n            dataset_config={\\n                \\"train_size\\": config.train_size,\\n                \\"validation_size\\": config.validation_size,\\n                \\"test_size\\": config.test_size,\\n                \\"image_size\\": config.image_size,\\n            },\\n            tokenizer_vocab=tokenizer.vocab,\\n            variant=variant,\\n            seed=seed,\\n            config_hash=cfg_hash,\\n            source_code_hash=source_code_hash,\\n            status=status,\\n        )\\n        # Dataclass nested objects are not JSON-safe in torch save metadata; convert lightly.\\n        payload[\\"model_config\\"][\\"vision\\"] = {\\n            key: (value if not hasattr(value, \\"__dict__\\") else value.__dict__)\\n            for key, value in model.vision_config.__dict__.items()\\n        }\\n        payload[\\"model_config\\"][\\"decoder\\"] = {\\n            key: (value if not hasattr(value, \\"__dict__\\") else value.__dict__)\\n            for key, value in model.decoder_config.__dict__.items()\\n        }\\n        save_checkpoint(path, payload)\\n        write_status(\\n            run_dir,\\n            {\\n                \\"status\\": status,\\n                \\"epoch\\": epoch,\\n                \\"global_step\\": global_step,\\n                \\"best_val_accuracy\\": best_val_accuracy,\\n                \\"examples_processed\\": examples_processed,\\n            },\\n        )\\n        manifest.upsert(\\n            variant,\\n            seed,\\n            cfg_hash,\\n            status=status if status != \\"running\\" else \\"running\\",\\n            run_directory=str(run_dir),\\n            latest_checkpoint=str(last_path),\\n            best_checkpoint=str(best_path) if best_path.exists() else None,\\n            current_epoch=epoch,\\n            global_step=global_step,\\n            best_validation_accuracy=best_val_accuracy if best_val_accuracy >= 0 else None,\\n        )\\n\\n    peak_alloc = 0\\n    peak_reserved = 0\\n    train_start = time.perf_counter()\\n    if device.type == \\"cuda\\":\\n        torch.cuda.reset_peak_memory_stats(device)\\n\\n    try:\\n        for epoch in range(start_epoch, config.max_epochs):\\n            model.train()\\n            epoch_loss = 0.0\\n            epoch_examples = 0\\n            epoch_t0 = time.perf_counter()\\n            for batch_index, batch in enumerate(loaders[\\"train\\"]):\\n                pixel_values = batch[\\"pixel_values\\"].to(device, non_blocking=True)\\n                input_ids = batch[\\"input_ids\\"].to(device, non_blocking=True)\\n                targets = batch[\\"targets\\"].to(device, non_blocking=True)\\n\\n                optimizer.zero_grad(set_to_none=True)\\n                autocast = (\\n                    torch.autocast(device_type=\\"cuda\\", dtype=amp_dtype, enabled=amp_dtype is not None)\\n                    if device.type == \\"cuda\\" and config.mixed_precision\\n                    else nullcontext()\\n                )\\n                with autocast:\\n                    output = model(\\n                        pixel_values=pixel_values,\\n                        input_ids=input_ids,\\n                        targets=targets,\\n                        return_aux=False,\\n                    )\\n                    loss = output[\\"loss\\"]\\n                if scaler.is_enabled():\\n                    scaler.scale(loss).backward()\\n                    scaler.unscale_(optimizer)\\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)\\n                    scaler.step(optimizer)\\n                    scaler.update()\\n                else:\\n                    loss.backward()\\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)\\n                    optimizer.step()\\n                scheduler.step()\\n\\n                batch_size = input_ids.size(0)\\n                epoch_loss += float(loss.item()) * batch_size\\n                epoch_examples += batch_size\\n                examples_processed += batch_size\\n                global_step += 1\\n\\n                if config.wandb_log_interval > 0 and global_step % config.wandb_log_interval == 0:\\n                    wandb_logger.log(\\n                        {\\n                            \\"train/loss\\": float(loss.item()),\\n                            \\"train/learning_rate\\": float(optimizer.param_groups[0][\\"lr\\"]),\\n                            \\"train/epoch\\": epoch,\\n                        },\\n                        step=global_step,\\n                    )\\n\\n                if config.checkpoint_interval > 0 and global_step % config.checkpoint_interval == 0:\\n                    elapsed_training_time += time.perf_counter() - epoch_t0\\n                    _save(last_path, epoch=epoch, batch_index=batch_index)\\n                    epoch_t0 = time.perf_counter()\\n\\n            elapsed_training_time += time.perf_counter() - epoch_t0\\n            train_loss = epoch_loss / max(1, epoch_examples)\\n            throughput = epoch_examples / max(1e-6, time.perf_counter() - (train_start if epoch == start_epoch else epoch_t0))\\n            append_jsonl(\\n                run_dir / \\"train_metrics.jsonl\\",\\n                {\\n                    \\"epoch\\": epoch,\\n                    \\"global_step\\": global_step,\\n                    \\"loss\\": train_loss,\\n                    \\"examples\\": epoch_examples,\\n                    \\"examples_per_sec\\": throughput,\\n                },\\n            )\\n            wandb_logger.log(\\n                {\\n                    \\"train/epoch_loss\\": train_loss,\\n                    \\"train/examples_per_sec\\": throughput,\\n                    \\"train/epoch\\": epoch,\\n                },\\n                step=global_step,\\n            )\\n\\n            _save(last_path, epoch=epoch + 1, batch_index=0)\\n            val_metrics = evaluate_model(\\n                model,\\n                loaders[\\"validation\\"],\\n                device=device,\\n                amp_dtype=amp_dtype if config.mixed_precision else None,\\n                capture_routing=True,\\n            )\\n            append_jsonl(\\n                run_dir / \\"validation_metrics.jsonl\\",\\n                {\\n                    \\"epoch\\": epoch,\\n                    \\"global_step\\": global_step,\\n                    \\"loss\\": val_metrics[\\"loss\\"],\\n                    \\"accuracy\\": val_metrics[\\"accuracy\\"],\\n                    \\"family_accuracy\\": val_metrics[\\"family_accuracy\\"],\\n                },\\n            )\\n            routing_summary = aggregate_routing_rows(val_metrics[\\"routing\\"])\\n            append_jsonl(\\n                run_dir / \\"routing_metrics.jsonl\\",\\n                {\\"epoch\\": epoch, \\"global_step\\": global_step, **routing_summary},\\n            )\\n            if val_metrics[\\"accuracy\\"] > best_val_accuracy:\\n                best_val_accuracy = float(val_metrics[\\"accuracy\\"])\\n                bad_epochs = 0\\n                _save(best_path, epoch=epoch + 1, batch_index=0)\\n            else:\\n                bad_epochs += 1\\n\\n            wandb_logger.log(\\n                {\\n                    \\"val/loss\\": val_metrics[\\"loss\\"],\\n                    \\"val/accuracy\\": val_metrics[\\"accuracy\\"],\\n                    \\"val/family_accuracy\\": val_metrics[\\"family_accuracy\\"],\\n                    \\"val/best_accuracy\\": best_val_accuracy,\\n                    \\"encoder_routing/n_sites\\": len(routing_summary.get(\\"encoder_routing\\", [])),\\n                    \\"decoder_routing/n_sites\\": len(routing_summary.get(\\"decoder_routing\\", [])),\\n                },\\n                step=global_step,\\n            )\\n            _save(last_path, epoch=epoch + 1, batch_index=0)\\n\\n            if device.type == \\"cuda\\":\\n                peak_alloc = max(peak_alloc, int(torch.cuda.max_memory_allocated(device)))\\n                peak_reserved = max(peak_reserved, int(torch.cuda.max_memory_reserved(device)))\\n\\n            if bad_epochs >= config.early_stopping_patience:\\n                break\\n\\n        # Final evaluation on best checkpoint when available.\\n        if best_path.exists():\\n            best_ckpt = load_checkpoint(best_path, map_location=device)\\n            model.load_state_dict(best_ckpt[\\"model\\"])\\n\\n        val_final = evaluate_model(\\n            model,\\n            loaders[\\"validation\\"],\\n            device=device,\\n            amp_dtype=amp_dtype if config.mixed_precision else None,\\n            capture_routing=True,\\n        )\\n        test_final = evaluate_model(\\n            model,\\n            loaders[\\"test\\"],\\n            device=device,\\n            amp_dtype=amp_dtype if config.mixed_precision else None,\\n            capture_routing=True,\\n        )\\n        routing_summary = {\\n            \\"encoder_routing\\": aggregate_routing_rows(val_final[\\"routing\\"])[\\"encoder_routing\\"],\\n            \\"decoder_routing\\": aggregate_routing_rows(val_final[\\"routing\\"])[\\"decoder_routing\\"],\\n            \\"test_encoder_routing\\": aggregate_routing_rows(test_final[\\"routing\\"])[\\"encoder_routing\\"],\\n            \\"test_decoder_routing\\": aggregate_routing_rows(test_final[\\"routing\\"])[\\"decoder_routing\\"],\\n        }\\n        atomic_write_json(run_dir / \\"routing_summary.json\\", routing_summary)\\n\\n        duration = elapsed_training_time\\n        final_metrics = {\\n            \\"variant\\": variant,\\n            \\"seed\\": seed,\\n            \\"config_hash\\": cfg_hash,\\n            \\"encoder_residual\\": VARIANTS[variant][\\"encoder\\"],\\n            \\"decoder_residual\\": VARIANTS[variant][\\"decoder\\"],\\n            \\"validation_loss\\": val_final[\\"loss\\"],\\n            \\"validation_accuracy\\": val_final[\\"accuracy\\"],\\n            \\"test_loss\\": test_final[\\"loss\\"],\\n            \\"test_accuracy\\": test_final[\\"accuracy\\"],\\n            \\"family_accuracy_validation\\": val_final[\\"family_accuracy\\"],\\n            \\"family_accuracy_test\\": test_final[\\"family_accuracy\\"],\\n            \\"parameter_count\\": param_counts[\\"total\\"],\\n            \\"parameter_increase_pct\\": param_increase_pct,\\n            \\"peak_allocated_bytes\\": peak_alloc,\\n            \\"peak_reserved_bytes\\": peak_reserved,\\n            \\"examples_per_sec\\": examples_processed / max(1e-6, duration),\\n            \\"training_duration_sec\\": duration,\\n            \\"best_epoch\\": start_epoch,  # overwritten below from best checkpoint when present\\n            \\"best_validation_accuracy\\": best_val_accuracy,\\n            \\"checkpoint_last\\": str(last_path),\\n            \\"checkpoint_best\\": str(best_path),\\n            \\"resumed\\": resumed,\\n            \\"init_validation\\": init_meta,\\n        }\\n        if best_path.exists():\\n            best_ckpt = load_checkpoint(best_path, map_location=\\"cpu\\")\\n            final_metrics[\\"best_epoch\\"] = int(best_ckpt[\\"epoch\\"])\\n        final_metrics[\\"wandb\\"] = wandb_logger.metadata()\\n        atomic_write_json(run_dir / \\"final_test_metrics.json\\", final_metrics)\\n        wandb_logger.update_summary(\\n            {\\n                \\"final/validation_loss\\": final_metrics[\\"validation_loss\\"],\\n                \\"final/validation_accuracy\\": final_metrics[\\"validation_accuracy\\"],\\n                \\"final/test_loss\\": final_metrics[\\"test_loss\\"],\\n                \\"final/test_accuracy\\": final_metrics[\\"test_accuracy\\"],\\n                \\"final/family_accuracy_test\\": final_metrics[\\"family_accuracy_test\\"],\\n                \\"final/best_epoch\\": final_metrics[\\"best_epoch\\"],\\n                \\"final/best_validation_accuracy\\": final_metrics[\\"best_validation_accuracy\\"],\\n                \\"final/parameter_count\\": final_metrics[\\"parameter_count\\"],\\n                \\"final/parameter_increase_pct\\": final_metrics[\\"parameter_increase_pct\\"],\\n                \\"final/peak_allocated_bytes\\": final_metrics[\\"peak_allocated_bytes\\"],\\n                \\"final/training_duration_sec\\": final_metrics[\\"training_duration_sec\\"],\\n                \\"checkpoint_best\\": final_metrics[\\"checkpoint_best\\"],\\n                \\"checkpoint_last\\": final_metrics[\\"checkpoint_last\\"],\\n            }\\n        )\\n        wandb_logger.log(\\n            {\\n                \\"test/loss\\": test_final[\\"loss\\"],\\n                \\"test/accuracy\\": test_final[\\"accuracy\\"],\\n                \\"test/family_accuracy\\": test_final[\\"family_accuracy\\"],\\n            },\\n            step=global_step,\\n        )\\n        _save(last_path, epoch=config.max_epochs, batch_index=0, status=\\"completed\\")\\n        mark_completed(run_dir)\\n        manifest.upsert(\\n            variant,\\n            seed,\\n            cfg_hash,\\n            status=\\"completed\\",\\n            run_directory=str(run_dir),\\n            latest_checkpoint=str(last_path),\\n            best_checkpoint=str(best_path),\\n            current_epoch=final_metrics[\\"best_epoch\\"],\\n            global_step=global_step,\\n            best_validation_accuracy=best_val_accuracy,\\n        )\\n        wandb_logger.finish(status=\\"completed\\")\\n        return {\\"status\\": \\"completed\\", \\"run_dir\\": str(run_dir), \\"metrics\\": final_metrics, \\"resumed\\": resumed}\\n    except Exception as exc:  # noqa: BLE001 - persist failure into manifest for resume UX\\n        write_status(run_dir, {\\"status\\": \\"failed\\", \\"error\\": str(exc)})\\n        if last_path.exists() or global_step > 0:\\n            try:\\n                _save(last_path, epoch=start_epoch, batch_index=0, status=\\"interrupted\\")\\n            except Exception:  # noqa: BLE001\\n                pass\\n        manifest.upsert(\\n            variant,\\n            seed,\\n            cfg_hash,\\n            status=\\"failed\\",\\n            run_directory=str(run_dir),\\n            latest_checkpoint=str(last_path) if last_path.exists() else None,\\n            best_checkpoint=str(best_path) if best_path.exists() else None,\\n            current_epoch=start_epoch,\\n            global_step=global_step,\\n            best_validation_accuracy=best_val_accuracy if best_val_accuracy >= 0 else None,\\n            error_message=str(exc),\\n        )\\n        wandb_logger.update_summary({\\"error_message\\": str(exc)})\\n        wandb_logger.finish(status=\\"failed\\")\\n        raise\\n", "src/vlm/ablation/correctness.py": "from __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\n\\nfrom src.models.attnres import DepthAttentionResidual\\nfrom src.models.baseline import BidirectionalSelfAttention\\nfrom src.models.vision_attnres import VisionConfig, build_vision_encoder\\nfrom src.models.vlm_attnres import TinyAttnResVLM\\nfrom src.utils.config import AttnResConfig, ModelConfig\\nfrom src.utils.runtime import seed_everything\\nfrom src.vlm.ablation.config import VARIANTS, AblationExperimentConfig, build_decoder_config, build_vision_config\\nfrom src.vlm.ablation.init_sync import (\\n    assert_encoder_decoder_attnres_separate,\\n    copy_shared_weights,\\n    validate_shared_initialization,\\n)\\nfrom src.vlm.ablation.io_utils import atomic_write_json\\nfrom src.vlm.ablation.train import build_model_for_variant\\nfrom src.vlm.synthetic_vqa import VQATokenizer, generate_example\\n\\n\\ndef _tiny_decoder_config(vocab_size: int, residual: str) -> ModelConfig:\\n    attnres = AttnResConfig(\\n        enabled=residual != \\"baseline\\",\\n        mode=\\"block\\" if residual == \\"block_attnres\\" else \\"full\\",\\n        num_blocks=2 if residual == \\"block_attnres\\" else None,\\n        final_readout=True,\\n    )\\n    return ModelConfig(\\n        architecture=\\"baseline\\" if residual == \\"baseline\\" else residual,\\n        size_name=\\"small\\",\\n        vocab_size=vocab_size,\\n        max_seq_len=32,\\n        d_model=32,\\n        n_layers=2,\\n        n_heads=4,\\n        d_ff=64,\\n        dropout=0.0,\\n        tie_weights=False,\\n        attnres=attnres,\\n    )\\n\\n\\ndef _tiny_vision_config(residual: str) -> VisionConfig:\\n    attnres = AttnResConfig(\\n        enabled=residual != \\"baseline\\",\\n        mode=\\"block\\" if residual == \\"block_attnres\\" else \\"full\\",\\n        num_blocks=2 if residual == \\"block_attnres\\" else None,\\n        final_readout=True,\\n    )\\n    return VisionConfig(\\n        image_size=32,\\n        patch_size=8,\\n        d_model=32,\\n        n_layers=2,\\n        n_heads=4,\\n        d_ff=64,\\n        dropout=0.0,\\n        residual=residual,  # type: ignore[arg-type]\\n        attnres=attnres,\\n    )\\n\\n\\ndef run_correctness_checks(*, device: torch.device, report_path: Path | None = None) -> dict[str, Any]:\\n    if device.type != \\"cuda\\":\\n        raise RuntimeError(\\"Correctness checks for the ablation grid require CUDA.\\")\\n\\n    results: dict[str, Any] = {\\"passed\\": [], \\"failed\\": []}\\n\\n    def check(name: str, fn) -> None:\\n        try:\\n            fn()\\n            results[\\"passed\\"].append(name)\\n        except Exception as exc:  # noqa: BLE001\\n            results[\\"failed\\"].append({\\"name\\": name, \\"error\\": str(exc)})\\n            raise\\n\\n    tokenizer = VQATokenizer()\\n\\n    def test_dataset_determinism() -> None:\\n        a = generate_example(split=\\"train\\", split_seed=7, example_index=3, image_size=64)\\n        b = generate_example(split=\\"train\\", split_seed=7, example_index=3, image_size=64)\\n        assert a.question == b.question and a.answer == b.answer\\n        assert (a.image == b.image).all()\\n\\n    def test_unambiguous_answers() -> None:\\n        for index in range(32):\\n            example = generate_example(split=\\"train\\", split_seed=11, example_index=index, image_size=64)\\n            assert example.answer in tokenizer.token_to_id\\n\\n    def test_patch_shape() -> None:\\n        encoder = build_vision_encoder(_tiny_vision_config(\\"baseline\\")).to(device)\\n        pixels = torch.randn(2, 3, 32, 32, device=device)\\n        tokens, _ = encoder(pixels)\\n        assert tokens.shape == (2, 16, 32)\\n\\n    def test_bidirectional_shape() -> None:\\n        attn = BidirectionalSelfAttention(32, 4, 0.0).to(device)\\n        x = torch.randn(2, 16, 32, device=device)\\n        out, _ = attn(x)\\n        assert out.shape == x.shape\\n\\n    def test_encoder_forward_backward(residual: str) -> None:\\n        encoder = build_vision_encoder(_tiny_vision_config(residual)).to(device)\\n        pixels = torch.randn(2, 3, 32, 32, device=device, requires_grad=False)\\n        tokens, aux = encoder(pixels, return_aux=True)\\n        loss = tokens.float().pow(2).mean()\\n        loss.backward()\\n        assert tokens.shape[1] == 16\\n        if residual != \\"baseline\\":\\n            assert aux[\\"depth_attention_rows\\"]\\n\\n    def test_routing_sums(residual: str) -> None:\\n        encoder = build_vision_encoder(_tiny_vision_config(residual)).to(device)\\n        encoder.set_weight_capture(True)\\n        pixels = torch.randn(2, 3, 32, 32, device=device)\\n        encoder(pixels, return_aux=True)\\n        for residual_module in encoder.iter_depth_residuals():\\n            weights = residual_module.last_weights\\n            assert weights is not None\\n            sums = weights.float().sum(dim=0)\\n            assert torch.allclose(sums, torch.ones_like(sums), atol=1e-4)\\n\\n    def test_pseudoquery_grads() -> None:\\n        encoder = build_vision_encoder(_tiny_vision_config(\\"attnres\\")).to(device)\\n        pixels = torch.randn(2, 3, 32, 32, device=device)\\n        tokens, _ = encoder(pixels)\\n        tokens.float().pow(2).mean().backward()\\n        grads = []\\n        for module in encoder.modules():\\n            if isinstance(module, DepthAttentionResidual):\\n                assert module.query.grad is not None\\n                grads.append(float(module.query.grad.abs().sum().item()))\\n        assert any(value > 0 for value in grads)\\n\\n    def test_separate_attnres() -> None:\\n        model = TinyAttnResVLM(\\n            vision_config=_tiny_vision_config(\\"attnres\\"),\\n            decoder_config=_tiny_decoder_config(tokenizer.vocab_size, \\"attnres\\"),\\n            encoder_residual=\\"attnres\\",\\n            decoder_residual=\\"attnres\\",\\n        ).to(device)\\n        assert_encoder_decoder_attnres_separate(model)\\n\\n    def test_all_variants_forward_backward() -> None:\\n        pixels = torch.randn(2, 3, 32, 32, device=device)\\n        input_ids = torch.tensor([[tokenizer.bos_token_id, tokenizer.token_to_id[\\"what\\"], tokenizer.answer_token_id, tokenizer.token_to_id[\\"red\\"], tokenizer.eos_token_id]] * 2, device=device)\\n        targets = torch.full_like(input_ids, -100)\\n        targets[:, 3] = input_ids[:, 3]\\n        targets[:, 4] = input_ids[:, 4]\\n        for variant, residual in VARIANTS.items():\\n            model = TinyAttnResVLM(\\n                vision_config=_tiny_vision_config(residual[\\"encoder\\"]),\\n                decoder_config=_tiny_decoder_config(tokenizer.vocab_size, residual[\\"decoder\\"]),\\n                encoder_residual=residual[\\"encoder\\"],\\n                decoder_residual=residual[\\"decoder\\"],\\n            ).to(device)\\n            out = model(pixel_values=pixels, input_ids=input_ids, targets=targets)\\n            out[\\"loss\\"].backward()\\n\\n    def test_loss_masking() -> None:\\n        model = TinyAttnResVLM(\\n            vision_config=_tiny_vision_config(\\"baseline\\"),\\n            decoder_config=_tiny_decoder_config(tokenizer.vocab_size, \\"baseline\\"),\\n        ).to(device)\\n        pixels = torch.randn(1, 3, 32, 32, device=device)\\n        input_ids = torch.tensor(\\n            [[tokenizer.bos_token_id, tokenizer.token_to_id[\\"what\\"], tokenizer.answer_token_id, tokenizer.token_to_id[\\"red\\"], tokenizer.eos_token_id]],\\n            device=device,\\n        )\\n        targets = torch.full_like(input_ids, -100)\\n        targets[0, 3] = input_ids[0, 3]\\n        targets[0, 4] = input_ids[0, 4]\\n        out = model(pixel_values=pixels, input_ids=input_ids, targets=targets)\\n        assert torch.isfinite(out[\\"loss\\"])\\n        # Only answer/eos supervised: flipping a question token target must not change loss when ignored.\\n        targets_q = targets.clone()\\n        targets_q[0, 1] = tokenizer.token_to_id[\\"what\\"]\\n        # With ignore_index, setting a previously -100 position to a label would change loss.\\n        # Confirm original targets ignore question positions.\\n        assert int((targets == -100).sum().item()) >= 3\\n\\n    def test_shared_init() -> None:\\n        seed_everything(0, deterministic=True)\\n        reference = TinyAttnResVLM(\\n            vision_config=_tiny_vision_config(\\"baseline\\"),\\n            decoder_config=_tiny_decoder_config(tokenizer.vocab_size, \\"baseline\\"),\\n        )\\n        candidate = TinyAttnResVLM(\\n            vision_config=_tiny_vision_config(\\"attnres\\"),\\n            decoder_config=_tiny_decoder_config(tokenizer.vocab_size, \\"attnres\\"),\\n            encoder_residual=\\"attnres\\",\\n            decoder_residual=\\"attnres\\",\\n        )\\n        copy_shared_weights(reference, candidate)\\n        validate_shared_initialization(reference, candidate)\\n\\n    def test_overfit_smoke() -> None:\\n        config = AblationExperimentConfig(\\n            run_mode=\\"smoke\\",\\n            seeds=[0],\\n            batch_size=16,\\n            max_epochs=20,\\n            early_stopping_patience=20,\\n            train_size=128,\\n            validation_size=32,\\n            test_size=32,\\n            vision_d_model=64,\\n            vision_n_layers=2,\\n            vision_n_heads=4,\\n            vision_d_ff=128,\\n            decoder_d_model=64,\\n            decoder_n_layers=2,\\n            decoder_n_heads=4,\\n            decoder_d_ff=128,\\n            num_blocks=2,\\n            num_workers=0,\\n            mixed_precision=True,\\n            amp_dtype=\\"float16\\",\\n        )\\n        from src.vlm.ablation.train import build_dataloaders\\n        from torch.optim import AdamW\\n\\n        tokenizer_local = VQATokenizer()\\n        loaders = build_dataloaders(config, tokenizer_local)\\n        model, _ = build_model_for_variant(\\n            config,\\n            variant=\\"baseline\\",\\n            vocab_size=tokenizer_local.vocab_size,\\n            seed=0,\\n        )\\n        model = model.to(device)\\n        optimizer = AdamW(model.parameters(), lr=3e-3)\\n        model.train()\\n        for _ in range(40):\\n            batch = next(iter(loaders[\\"train\\"]))\\n            optimizer.zero_grad(set_to_none=True)\\n            out = model(\\n                pixel_values=batch[\\"pixel_values\\"].to(device),\\n                input_ids=batch[\\"input_ids\\"].to(device),\\n                targets=batch[\\"targets\\"].to(device),\\n            )\\n            out[\\"loss\\"].backward()\\n            optimizer.step()\\n        # Evaluate on the same tiny loader batch distribution.\\n        model.eval()\\n        with torch.no_grad():\\n            batch = next(iter(loaders[\\"train\\"]))\\n            out = model(\\n                pixel_values=batch[\\"pixel_values\\"].to(device),\\n                input_ids=batch[\\"input_ids\\"].to(device),\\n                targets=batch[\\"targets\\"].to(device),\\n            )\\n            preds = out[\\"logits\\"].argmax(dim=-1)\\n            answer_positions = batch[\\"answer_positions\\"]\\n            correct = 0\\n            for row in range(batch[\\"input_ids\\"].size(0)):\\n                pos = int(answer_positions[row].item())\\n                if int(preds[row, pos].item()) == int(batch[\\"answer_ids\\"][row].item()):\\n                    correct += 1\\n            accuracy = correct / batch[\\"input_ids\\"].size(0)\\n        if accuracy < 0.5:\\n            raise AssertionError(f\\"Overfit smoke accuracy too low: {accuracy}\\")\\n\\n    check(\\"dataset_determinism\\", test_dataset_determinism)\\n    check(\\"unambiguous_answers\\", test_unambiguous_answers)\\n    check(\\"patch_token_shape\\", test_patch_shape)\\n    check(\\"bidirectional_attention_shape\\", test_bidirectional_shape)\\n    check(\\"baseline_encoder_forward_backward\\", lambda: test_encoder_forward_backward(\\"baseline\\"))\\n    check(\\"full_attnres_encoder_forward_backward\\", lambda: test_encoder_forward_backward(\\"attnres\\"))\\n    check(\\"block_attnres_encoder_forward_backward\\", lambda: test_encoder_forward_backward(\\"block_attnres\\"))\\n    check(\\"full_encoder_routing_sums\\", lambda: test_routing_sums(\\"attnres\\"))\\n    check(\\"block_encoder_routing_sums\\", lambda: test_routing_sums(\\"block_attnres\\"))\\n    check(\\"encoder_pseudoquery_gradients\\", test_pseudoquery_grads)\\n    check(\\"encoder_decoder_attnres_separate\\", test_separate_attnres)\\n    check(\\"all_variants_forward_backward\\", test_all_variants_forward_backward)\\n    check(\\"loss_masking\\", test_loss_masking)\\n    check(\\"shared_initialization\\", test_shared_init)\\n    check(\\"tiny_overfit\\", test_overfit_smoke)\\n\\n    results[\\"ok\\"] = not results[\\"failed\\"]\\n    if report_path is not None:\\n        atomic_write_json(report_path, results)\\n    return results\\n", "src/vlm/ablation/aggregate.py": "from __future__ import annotations\\n\\nimport csv\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\n\\nfrom src.vlm.ablation.io_utils import atomic_write_json, ensure_dir\\n\\n\\nFAMILY_KEYS = {\\n    \\"local_detail\\": \\"local_detail_accuracy\\",\\n    \\"attribute\\": \\"attribute_accuracy\\",\\n    \\"counting\\": \\"counting_accuracy\\",\\n    \\"location\\": \\"location_accuracy\\",\\n    \\"relation\\": \\"relation_accuracy\\",\\n}\\n\\n\\ndef _flatten_run_metrics(metrics: dict[str, Any]) -> dict[str, Any]:\\n    family = metrics.get(\\"family_accuracy_test\\", {})\\n    row = {\\n        \\"variant\\": metrics.get(\\"variant\\"),\\n        \\"seed\\": metrics.get(\\"seed\\"),\\n        \\"encoder_residual\\": metrics.get(\\"encoder_residual\\"),\\n        \\"decoder_residual\\": metrics.get(\\"decoder_residual\\"),\\n        \\"validation_accuracy\\": metrics.get(\\"validation_accuracy\\"),\\n        \\"test_accuracy\\": metrics.get(\\"test_accuracy\\"),\\n        \\"parameter_count\\": metrics.get(\\"parameter_count\\"),\\n        \\"parameter_increase_pct\\": metrics.get(\\"parameter_increase_pct\\"),\\n        \\"peak_gpu_memory_bytes\\": metrics.get(\\"peak_allocated_bytes\\"),\\n        \\"throughput_examples_per_sec\\": metrics.get(\\"examples_per_sec\\"),\\n        \\"training_duration_sec\\": metrics.get(\\"training_duration_sec\\"),\\n        \\"best_epoch\\": metrics.get(\\"best_epoch\\"),\\n        \\"config_hash\\": metrics.get(\\"config_hash\\"),\\n    }\\n    for family_name, column in FAMILY_KEYS.items():\\n        row[column] = family.get(family_name)\\n    return row\\n\\n\\ndef collect_run_rows(project_root: Path, config_hash: str) -> list[dict[str, Any]]:\\n    rows: list[dict[str, Any]] = []\\n    runs_root = Path(project_root) / \\"runs\\"\\n    if not runs_root.exists():\\n        return rows\\n    for metrics_path in runs_root.glob(f\\"*/*/{config_hash}/final_test_metrics.json\\"):\\n        metrics = json.loads(metrics_path.read_text(encoding=\\"utf-8\\"))\\n        rows.append(_flatten_run_metrics(metrics))\\n    return rows\\n\\n\\ndef write_csv(path: Path, rows: list[dict[str, Any]]) -> None:\\n    ensure_dir(path.parent)\\n    if not rows:\\n        path.write_text(\\"\\", encoding=\\"utf-8\\")\\n        return\\n    fieldnames = list(rows[0].keys())\\n    with path.open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\\n        writer.writeheader()\\n        for row in rows:\\n            writer.writerow(row)\\n\\n\\ndef aggregate_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:\\n    by_variant: dict[str, list[dict[str, Any]]] = {}\\n    for row in rows:\\n        by_variant.setdefault(str(row[\\"variant\\"]), []).append(row)\\n\\n    numeric_keys = [\\n        \\"validation_accuracy\\",\\n        \\"test_accuracy\\",\\n        \\"local_detail_accuracy\\",\\n        \\"attribute_accuracy\\",\\n        \\"counting_accuracy\\",\\n        \\"location_accuracy\\",\\n        \\"relation_accuracy\\",\\n        \\"parameter_count\\",\\n        \\"parameter_increase_pct\\",\\n        \\"peak_gpu_memory_bytes\\",\\n        \\"throughput_examples_per_sec\\",\\n        \\"training_duration_sec\\",\\n        \\"best_epoch\\",\\n    ]\\n    aggregated: list[dict[str, Any]] = []\\n    for variant, variant_rows in sorted(by_variant.items()):\\n        payload: dict[str, Any] = {\\n            \\"variant\\": variant,\\n            \\"n_seeds\\": len(variant_rows),\\n            \\"encoder_residual\\": variant_rows[0].get(\\"encoder_residual\\"),\\n            \\"decoder_residual\\": variant_rows[0].get(\\"decoder_residual\\"),\\n            \\"label\\": \\"exploratory_mean_std_over_seeds\\",\\n        }\\n        for key in numeric_keys:\\n            values = [float(row[key]) for row in variant_rows if row.get(key) is not None]\\n            if not values:\\n                payload[f\\"{key}_mean\\"] = None\\n                payload[f\\"{key}_std\\"] = None\\n                payload[f\\"{key}_values\\"] = []\\n                continue\\n            payload[f\\"{key}_mean\\"] = float(np.mean(values))\\n            payload[f\\"{key}_std\\"] = float(np.std(values, ddof=0))\\n            payload[f\\"{key}_values\\"] = values\\n        aggregated.append(payload)\\n    return aggregated\\n\\n\\ndef write_tables(project_root: Path, config_hash: str) -> dict[str, Path]:\\n    rows = collect_run_rows(project_root, config_hash)\\n    tables_dir = ensure_dir(Path(project_root) / \\"tables\\")\\n    all_path = tables_dir / \\"all_runs.csv\\"\\n    agg_path = tables_dir / \\"aggregate_results.csv\\"\\n    write_csv(all_path, rows)\\n    aggregated = aggregate_rows(rows)\\n    # Flatten list values for CSV readability.\\n    flat_agg: list[dict[str, Any]] = []\\n    for row in aggregated:\\n        flat = dict(row)\\n        for key, value in list(flat.items()):\\n            if isinstance(value, list):\\n                flat[key] = \\";\\".join(str(item) for item in value)\\n        flat_agg.append(flat)\\n    write_csv(agg_path, flat_agg)\\n    atomic_write_json(tables_dir / \\"aggregate_results.json\\", aggregated)\\n    return {\\"all_runs\\": all_path, \\"aggregate_results\\": agg_path}\\n", "src/vlm/ablation/plots.py": "from __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\n\\nfrom src.vlm.ablation.aggregate import collect_run_rows\\nfrom src.vlm.ablation.io_utils import ensure_dir\\n\\n\\ndef _save_figure(fig: plt.Figure, plots_dir: Path, name: str) -> None:\\n    png = plots_dir / f\\"{name}.png\\"\\n    pdf = plots_dir / f\\"{name}.pdf\\"\\n    fig.tight_layout()\\n    fig.savefig(png, dpi=160)\\n    fig.savefig(pdf)\\n    plt.close(fig)\\n\\n\\ndef _mean_std_by_variant(rows: list[dict[str, Any]], key: str) -> tuple[list[str], list[float], list[float]]:\\n    by_variant: dict[str, list[float]] = {}\\n    for row in rows:\\n        if row.get(key) is None:\\n            continue\\n        by_variant.setdefault(str(row[\\"variant\\"]), []).append(float(row[key]))\\n    variants = sorted(by_variant)\\n    means = [float(np.mean(by_variant[v])) for v in variants]\\n    stds = [float(np.std(by_variant[v], ddof=0)) for v in variants]\\n    return variants, means, stds\\n\\n\\ndef _load_validation_curves(project_root: Path, config_hash: str) -> dict[str, list[dict[str, Any]]]:\\n    curves: dict[str, list[dict[str, Any]]] = {}\\n    runs_root = Path(project_root) / \\"runs\\"\\n    for path in runs_root.glob(f\\"*/*/{config_hash}/validation_metrics.jsonl\\"):\\n        variant = path.parts[-4]\\n        rows = [json.loads(line) for line in path.read_text(encoding=\\"utf-8\\").splitlines() if line.strip()]\\n        curves.setdefault(variant, []).append(rows)\\n    return curves\\n\\n\\ndef _load_routing_summary(project_root: Path, config_hash: str, variant: str) -> dict[str, Any] | None:\\n    runs_root = Path(project_root) / \\"runs\\" / variant\\n    for path in runs_root.glob(f\\"*/{config_hash}/routing_summary.json\\"):\\n        return json.loads(path.read_text(encoding=\\"utf-8\\"))\\n    return None\\n\\n\\ndef generate_plots(project_root: Path, config_hash: str) -> Path:\\n    plots_dir = ensure_dir(Path(project_root) / \\"plots\\")\\n    rows = collect_run_rows(project_root, config_hash)\\n\\n    # 1. Overall test accuracy\\n    variants, means, stds = _mean_std_by_variant(rows, \\"test_accuracy\\")\\n    fig, ax = plt.subplots(figsize=(8, 4))\\n    ax.bar(variants, means, yerr=stds, capsize=4)\\n    ax.set_title(\\"Overall test accuracy by variant (exploratory)\\")\\n    ax.set_ylabel(\\"Accuracy\\")\\n    ax.tick_params(axis=\\"x\\", rotation=30)\\n    _save_figure(fig, plots_dir, \\"01_test_accuracy_by_variant\\")\\n\\n    # 2. Accuracy by family\\n    families = [\\n        \\"local_detail_accuracy\\",\\n        \\"attribute_accuracy\\",\\n        \\"counting_accuracy\\",\\n        \\"location_accuracy\\",\\n        \\"relation_accuracy\\",\\n    ]\\n    if variants:\\n        fig, ax = plt.subplots(figsize=(10, 5))\\n        x = np.arange(len(variants))\\n        width = 0.15\\n        for index, family in enumerate(families):\\n            family_means = []\\n            for variant in variants:\\n                values = [float(row[family]) for row in rows if row[\\"variant\\"] == variant and row.get(family) is not None]\\n                family_means.append(float(np.mean(values)) if values else 0.0)\\n            ax.bar(x + index * width, family_means, width=width, label=family.replace(\\"_accuracy\\", \\"\\"))\\n        ax.set_xticks(x + width * 2)\\n        ax.set_xticklabels(variants, rotation=30)\\n        ax.set_title(\\"Accuracy by question family and variant\\")\\n        ax.legend(fontsize=8)\\n        _save_figure(fig, plots_dir, \\"02_accuracy_by_family\\")\\n\\n    # 3/4 validation curves\\n    curves = _load_validation_curves(project_root, config_hash)\\n    fig, ax = plt.subplots(figsize=(8, 4))\\n    for variant, variant_curves in curves.items():\\n        if not variant_curves:\\n            continue\\n        max_len = max(len(curve) for curve in variant_curves)\\n        matrix = np.full((len(variant_curves), max_len), np.nan)\\n        for row_index, curve in enumerate(variant_curves):\\n            for col_index, point in enumerate(curve):\\n                matrix[row_index, col_index] = point[\\"accuracy\\"]\\n        mean = np.nanmean(matrix, axis=0)\\n        ax.plot(np.arange(len(mean)), mean, label=variant)\\n    ax.set_title(\\"Validation accuracy over training\\")\\n    ax.set_xlabel(\\"Epoch\\")\\n    ax.set_ylabel(\\"Accuracy\\")\\n    ax.legend(fontsize=8)\\n    _save_figure(fig, plots_dir, \\"03_validation_accuracy_curves\\")\\n\\n    fig, ax = plt.subplots(figsize=(8, 4))\\n    for variant, variant_curves in curves.items():\\n        if not variant_curves:\\n            continue\\n        max_len = max(len(curve) for curve in variant_curves)\\n        matrix = np.full((len(variant_curves), max_len), np.nan)\\n        for row_index, curve in enumerate(variant_curves):\\n            for col_index, point in enumerate(curve):\\n                matrix[row_index, col_index] = point[\\"loss\\"]\\n        mean = np.nanmean(matrix, axis=0)\\n        ax.plot(np.arange(len(mean)), mean, label=variant)\\n    ax.set_title(\\"Validation loss over training\\")\\n    ax.set_xlabel(\\"Epoch\\")\\n    ax.set_ylabel(\\"Loss\\")\\n    ax.legend(fontsize=8)\\n    _save_figure(fig, plots_dir, \\"04_validation_loss_curves\\")\\n\\n    # 5/6/7 resource plots\\n    for index, (key, title, name) in enumerate(\\n        [\\n            (\\"parameter_count\\", \\"Parameter count by variant\\", \\"05_parameter_count\\"),\\n            (\\"peak_gpu_memory_bytes\\", \\"Peak GPU memory by variant\\", \\"06_peak_gpu_memory\\"),\\n            (\\"throughput_examples_per_sec\\", \\"Throughput by variant\\", \\"07_throughput\\"),\\n        ],\\n        start=5,\\n    ):\\n        variants, means, stds = _mean_std_by_variant(rows, key)\\n        fig, ax = plt.subplots(figsize=(8, 4))\\n        ax.bar(variants, means, yerr=stds, capsize=4)\\n        ax.set_title(title)\\n        ax.tick_params(axis=\\"x\\", rotation=30)\\n        _save_figure(fig, plots_dir, name)\\n\\n    # Routing heatmaps / entropy: use first available seed summary per interesting variant.\\n    for variant in (\\"encoder_full\\", \\"both_full\\", \\"decoder_full\\", \\"encoder_block\\", \\"both_block\\"):\\n        summary = _load_routing_summary(project_root, config_hash, variant)\\n        if not summary:\\n            continue\\n        for namespace, plot_prefix in (\\n            (\\"encoder_routing\\", \\"08_encoder\\"),\\n            (\\"decoder_routing\\", \\"09_decoder\\"),\\n        ):\\n            sites = summary.get(namespace, [])\\n            if not sites:\\n                continue\\n            families = sorted({family for site in sites for family in site.get(\\"by_family\\", {})})\\n            if families:\\n                matrix = np.array(\\n                    [\\n                        [site.get(\\"by_family\\", {}).get(family, {}).get(\\"embedding\\", np.nan) for family in families]\\n                        for site in sites\\n                    ],\\n                    dtype=float,\\n                )\\n                fig, ax = plt.subplots(figsize=(8, 4))\\n                im = ax.imshow(matrix, aspect=\\"auto\\", cmap=\\"viridis\\")\\n                ax.set_yticks(range(len(sites)))\\n                ax.set_yticklabels([f\\"site_{site[\'site_index\']}\\" for site in sites])\\n                ax.set_xticks(range(len(families)))\\n                ax.set_xticklabels(families, rotation=30)\\n                ax.set_title(f\\"{namespace} embedding contribution by family ({variant})\\")\\n                fig.colorbar(im, ax=ax, fraction=0.046)\\n                _save_figure(fig, plots_dir, f\\"{plot_prefix}_routing_heatmap_{variant}\\")\\n\\n            entropies = [float(site.get(\\"entropy\\", np.nan)) for site in sites]\\n            fig, ax = plt.subplots(figsize=(8, 4))\\n            ax.plot(range(len(entropies)), entropies, marker=\\"o\\")\\n            ax.set_title(f\\"{namespace} entropy by site ({variant})\\")\\n            ax.set_xlabel(\\"Routing site\\")\\n            ax.set_ylabel(\\"Entropy\\")\\n            prefix = \\"10_encoder\\" if namespace.startswith(\\"encoder\\") else \\"11_decoder\\"\\n            _save_figure(fig, plots_dir, f\\"{prefix}_routing_entropy_{variant}\\")\\n\\n    return plots_dir\\n", "src/vlm/ablation/runner.py": "from __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport torch\\n\\nfrom src.vlm.ablation.aggregate import write_tables\\nfrom src.vlm.ablation.config import (\\n    AblationExperimentConfig,\\n    config_hash,\\n    resolve_experiment_config,\\n)\\nfrom src.vlm.ablation.correctness import run_correctness_checks\\nfrom src.vlm.ablation.io_utils import atomic_write_json, create_project_layout, ensure_dir\\nfrom src.vlm.ablation.manifest import ExperimentManifest\\nfrom src.vlm.ablation.plots import generate_plots\\nfrom src.vlm.ablation.source_hash import combined_source_hash, hash_source_tree\\nfrom src.vlm.ablation.train import train_variant_seed\\nfrom src.vlm.ablation.wandb_logger import resolve_wandb_mode\\n\\n\\ndef _log_experiment_artifacts_to_wandb(\\n    config: AblationExperimentConfig,\\n    *,\\n    project_root: Path,\\n    cfg_hash: str,\\n    summary: dict[str, Any],\\n    plots_dir: Path,\\n) -> dict[str, Any] | None:\\n    enabled, mode = resolve_wandb_mode(config)\\n    if not enabled:\\n        return None\\n    try:\\n        import wandb\\n    except Exception as error:  # noqa: BLE001\\n        return {\\"enabled\\": False, \\"error\\": f\\"wandb import failed: {error}\\"}\\n\\n    init_kwargs = {\\n        \\"project\\": config.wandb_project,\\n        \\"entity\\": config.wandb_entity or None,\\n        \\"name\\": f\\"vlm_ablation_summary_{config.run_mode}_{cfg_hash}\\",\\n        \\"id\\": f\\"vlm_ablation_summary_{cfg_hash}\\",\\n        \\"resume\\": \\"allow\\",\\n        \\"mode\\": mode,\\n        \\"dir\\": str(project_root / \\"logs\\"),\\n        \\"job_type\\": \\"summary\\",\\n        \\"group\\": f\\"vlm_ablation_{config.run_mode}_{cfg_hash}\\",\\n        \\"tags\\": [\\"vlm-ablation\\", \\"summary\\", config.run_mode],\\n        \\"config\\": config.to_dict(),\\n    }\\n    init_kwargs = {key: value for key, value in init_kwargs.items() if value is not None}\\n    try:\\n        run = wandb.init(**init_kwargs)\\n    except Exception as error:  # noqa: BLE001\\n        return {\\"enabled\\": False, \\"error\\": str(error)}\\n\\n    for key, value in summary.items():\\n        if isinstance(value, (int, float, bool, str)):\\n            run.summary[key] = value\\n    tables = summary.get(\\"tables\\") or {}\\n    for table_name, table_path in tables.items():\\n        path = Path(table_path)\\n        if path.exists():\\n            artifact = wandb.Artifact(f\\"{cfg_hash}_{table_name}\\", type=\\"table\\")\\n            artifact.add_file(str(path))\\n            run.log_artifact(artifact)\\n    image_payload = {}\\n    for png in sorted(Path(plots_dir).glob(\\"*.png\\")):\\n        image_payload[f\\"plots/{png.stem}\\"] = wandb.Image(str(png))\\n    if image_payload:\\n        run.log(image_payload)\\n    url = None\\n    try:\\n        url = run.get_url() if hasattr(run, \\"get_url\\") else None\\n    except Exception:  # noqa: BLE001\\n        url = None\\n    run.finish()\\n    return {\\"enabled\\": True, \\"mode\\": mode, \\"url\\": url, \\"project\\": config.wandb_project}\\n\\n\\ndef require_cuda() -> torch.device:\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\n            \\"CUDA GPU is unavailable. Select a Colab GPU runtime and rerun. \\"\\n            \\"Training will not fall back to CPU or MPS.\\"\\n        )\\n    return torch.device(\\"cuda\\")\\n\\n\\ndef print_cuda_environment(amp_dtype: torch.dtype | None) -> dict[str, Any]:\\n    device = require_cuda()\\n    props = torch.cuda.get_device_properties(device)\\n    free_bytes, total_bytes = torch.cuda.mem_get_info(device)\\n    info = {\\n        \\"gpu_name\\": props.name,\\n        \\"cuda_version\\": torch.version.cuda,\\n        \\"torch_version\\": torch.__version__,\\n        \\"total_memory_bytes\\": int(total_bytes),\\n        \\"available_memory_bytes\\": int(free_bytes),\\n        \\"amp_dtype\\": str(amp_dtype) if amp_dtype is not None else None,\\n    }\\n    print(json.dumps(info, indent=2))\\n    return info\\n\\n\\ndef run_ablation_experiment(\\n    config: AblationExperimentConfig,\\n    *,\\n    project_root: Path,\\n    source_root: Path | None = None,\\n    skip_correctness: bool = False,\\n) -> dict[str, Any]:\\n    project_root = Path(project_root)\\n    create_project_layout(project_root)\\n    config.project_root = str(project_root)\\n    config = resolve_experiment_config(config)\\n\\n    device = require_cuda()\\n    major, _ = torch.cuda.get_device_capability(device)\\n    amp_dtype = torch.bfloat16 if (config.amp_dtype == \\"auto\\" and major >= 8) else (\\n        torch.float16 if config.amp_dtype == \\"auto\\" else None\\n    )\\n    if config.amp_dtype != \\"auto\\":\\n        from src.utils.runtime import amp_dtype_from_string\\n\\n        amp_dtype = amp_dtype_from_string(config.amp_dtype)\\n    cuda_info = print_cuda_environment(amp_dtype)\\n\\n    source_root = Path(source_root) if source_root is not None else project_root / \\"source\\"\\n    source_hashes = hash_source_tree(source_root)\\n    source_code_hash = combined_source_hash(source_hashes)\\n    atomic_write_json(project_root / \\"summaries\\" / \\"source_hashes.json\\", source_hashes)\\n\\n    cfg_hash = config_hash(config)\\n    atomic_write_json(project_root / \\"configs\\" / f\\"experiment_{cfg_hash}.json\\", config.to_dict())\\n    print(\\"Resolved experiment configuration:\\")\\n    print(json.dumps(config.to_dict(), indent=2, sort_keys=True))\\n\\n    if not skip_correctness:\\n        report = run_correctness_checks(\\n            device=device,\\n            report_path=project_root / \\"summaries\\" / \\"correctness_checks.json\\",\\n        )\\n        if not report[\\"ok\\"]:\\n            raise RuntimeError(f\\"Correctness checks failed: {report[\'failed\']}\\")\\n\\n    manifest = ExperimentManifest(project_root / \\"manifests\\" / \\"experiment_manifest.json\\")\\n    requested = config.requested_variants()\\n    completed: list[str] = []\\n    resumed: list[str] = []\\n    failed: list[str] = []\\n    results: list[dict[str, Any]] = []\\n\\n    for variant in requested:\\n        for seed in config.seeds:\\n            key = f\\"{variant}/seed_{seed}\\"\\n            try:\\n                result = train_variant_seed(\\n                    config,\\n                    variant=variant,\\n                    seed=seed,\\n                    project_root=project_root,\\n                    manifest=manifest,\\n                    source_code_hash=source_code_hash,\\n                    device=device,\\n                )\\n                results.append(result)\\n                if result[\\"status\\"] == \\"skipped_completed\\":\\n                    completed.append(key)\\n                elif result.get(\\"resumed\\"):\\n                    resumed.append(key)\\n                    completed.append(key)\\n                else:\\n                    completed.append(key)\\n            except Exception as exc:  # noqa: BLE001\\n                failed.append(key)\\n                print(f\\"FAILED {key}: {exc}\\")\\n                if config.run_mode == \\"smoke\\":\\n                    raise\\n\\n    tables = write_tables(project_root, cfg_hash)\\n    plots_dir = generate_plots(project_root, cfg_hash)\\n\\n    best_by_variant: dict[str, dict[str, Any]] = {}\\n    for result in results:\\n        metrics = result.get(\\"metrics\\") or {}\\n        variant = metrics.get(\\"variant\\")\\n        if not variant:\\n            continue\\n        current = best_by_variant.get(variant)\\n        if current is None or float(metrics.get(\\"test_accuracy\\", -1)) > float(current.get(\\"test_accuracy\\", -1)):\\n            best_by_variant[variant] = metrics\\n\\n    summary = {\\n        \\"drive_project_root\\": str(project_root),\\n        \\"cuda\\": cuda_info,\\n        \\"run_mode\\": config.run_mode,\\n        \\"config_hash\\": cfg_hash,\\n        \\"source_code_hash\\": source_code_hash,\\n        \\"requested_variants\\": requested,\\n        \\"completed\\": completed,\\n        \\"resumed\\": resumed,\\n        \\"failed\\": failed,\\n        \\"best_by_variant\\": best_by_variant,\\n        \\"tables\\": {key: str(path) for key, path in tables.items()},\\n        \\"plots_dir\\": str(plots_dir),\\n        \\"manifest\\": str(project_root / \\"manifests\\" / \\"experiment_manifest.json\\"),\\n        \\"artifacts_under_drive\\": True,\\n        \\"label\\": \\"exploratory\\",\\n        \\"wandb_project\\": config.wandb_project,\\n        \\"wandb_entity\\": config.wandb_entity,\\n    }\\n    wandb_summary = _log_experiment_artifacts_to_wandb(\\n        config,\\n        project_root=project_root,\\n        cfg_hash=cfg_hash,\\n        summary=summary,\\n        plots_dir=plots_dir,\\n    )\\n    if wandb_summary is not None:\\n        summary[\\"wandb_summary\\"] = wandb_summary\\n    atomic_write_json(project_root / \\"summaries\\" / \\"experiment_summary.json\\", summary)\\n    return summary\\n", "src/vlm/ablation/wandb_logger.py": "from __future__ import annotations\\n\\nimport os\\nfrom pathlib import Path\\nfrom typing import Any, Mapping\\n\\nfrom src.vlm.ablation.config import AblationExperimentConfig, VARIANTS\\n\\n\\ndef _env_truthy(name: str) -> bool:\\n    return os.getenv(name, \\"\\").strip().lower() in {\\"1\\", \\"true\\", \\"yes\\", \\"on\\"}\\n\\n\\ndef resolve_wandb_mode(config: AblationExperimentConfig) -> tuple[bool, str]:\\n    if not config.wandb_enabled:\\n        return False, \\"disabled\\"\\n    env_mode = os.getenv(\\"WANDB_MODE\\", \\"\\").strip().lower()\\n    if env_mode == \\"disabled\\" or _env_truthy(\\"WANDB_DISABLED\\"):\\n        return False, \\"disabled\\"\\n    if env_mode in {\\"online\\", \\"offline\\"}:\\n        return True, env_mode\\n    if config.wandb_mode in {\\"online\\", \\"offline\\", \\"disabled\\"}:\\n        if config.wandb_mode == \\"disabled\\":\\n            return False, \\"disabled\\"\\n        return True, config.wandb_mode\\n    if os.getenv(\\"WANDB_API_KEY\\"):\\n        return True, \\"online\\"\\n    return True, \\"offline\\"\\n\\n\\ndef _flatten(payload: Mapping[str, Any], *, prefix: str = \\"\\", allow_strings: bool = False) -> dict[str, Any]:\\n    flattened: dict[str, Any] = {}\\n    for key, value in payload.items():\\n        if value is None:\\n            continue\\n        name = f\\"{prefix}/{key}\\" if prefix else str(key)\\n        if isinstance(value, Mapping):\\n            flattened.update(_flatten(value, prefix=name, allow_strings=allow_strings))\\n            continue\\n        if hasattr(value, \\"item\\") and not isinstance(value, (str, bytes)):\\n            try:\\n                value = value.item()\\n            except Exception:\\n                pass\\n        if isinstance(value, (int, float, bool)):\\n            flattened[name] = value\\n        elif allow_strings and isinstance(value, str):\\n            flattened[name] = value\\n    return flattened\\n\\n\\ndef stable_run_id(variant: str, seed: int, config_hash: str) -> str:\\n    return f\\"vlm_ablation_{variant}_seed{seed}_{config_hash}\\"\\n\\n\\ndef stable_run_name(variant: str, seed: int, config_hash: str, run_mode: str) -> str:\\n    return f\\"{variant}_seed{seed}_{run_mode}_{config_hash}\\"\\n\\n\\nclass AblationWandbLogger:\\n    \\"\\"\\"Per variant/seed W&B logger mirroring the repo\'s OptionalWandbLogger style.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        *,\\n        config: AblationExperimentConfig,\\n        variant: str,\\n        seed: int,\\n        config_hash: str,\\n        run_dir: Path,\\n        extra_config: Mapping[str, Any] | None = None,\\n    ) -> None:\\n        self.enabled = False\\n        self.mode = \\"disabled\\"\\n        self.project = config.wandb_project\\n        self.entity = config.wandb_entity\\n        self.run_id: str | None = None\\n        self.run_name: str | None = None\\n        self.url: str | None = None\\n        self.error: str | None = None\\n        self._run = None\\n\\n        enabled, mode = resolve_wandb_mode(config)\\n        if not enabled:\\n            return\\n\\n        try:\\n            import wandb\\n        except Exception as error:  # noqa: BLE001\\n            self.error = f\\"wandb import failed: {error}\\"\\n            return\\n\\n        residual = VARIANTS[variant]\\n        run_id = stable_run_id(variant, seed, config_hash)\\n        run_name = stable_run_name(variant, seed, config_hash, config.run_mode)\\n        init_kwargs: dict[str, Any] = {\\n            \\"project\\": config.wandb_project,\\n            \\"entity\\": config.wandb_entity or None,\\n            \\"name\\": run_name,\\n            \\"mode\\": mode,\\n            \\"dir\\": str(run_dir),\\n            \\"job_type\\": \\"train\\",\\n            \\"group\\": f\\"vlm_ablation_{config.run_mode}_{config_hash}\\",\\n            \\"tags\\": sorted(\\n                {\\n                    \\"vlm-ablation\\",\\n                    config.run_mode,\\n                    variant,\\n                    f\\"seed{seed}\\",\\n                    residual[\\"encoder\\"],\\n                    residual[\\"decoder\\"],\\n                    *list(config.wandb_tags),\\n                }\\n            ),\\n            \\"config\\": {\\n                **config.to_dict(),\\n                \\"variant\\": variant,\\n                \\"seed\\": seed,\\n                \\"config_hash\\": config_hash,\\n                \\"encoder_residual\\": residual[\\"encoder\\"],\\n                \\"decoder_residual\\": residual[\\"decoder\\"],\\n                **dict(extra_config or {}),\\n            },\\n        }\\n        if config.wandb_resume != \\"never\\":\\n            init_kwargs[\\"id\\"] = run_id\\n            init_kwargs[\\"resume\\"] = config.wandb_resume\\n        init_kwargs = {key: value for key, value in init_kwargs.items() if value is not None}\\n\\n        try:\\n            run = wandb.init(**init_kwargs)\\n        except Exception as error:  # noqa: BLE001\\n            if mode != \\"online\\":\\n                self.error = f\\"wandb init failed: {error}\\"\\n                return\\n            init_kwargs[\\"mode\\"] = \\"offline\\"\\n            try:\\n                run = wandb.init(**init_kwargs)\\n                mode = \\"offline\\"\\n                self.error = f\\"wandb online init failed, fell back to offline: {error}\\"\\n            except Exception as offline_error:  # noqa: BLE001\\n                self.error = f\\"wandb init failed: {error}; offline fallback failed: {offline_error}\\"\\n                return\\n\\n        self._run = run\\n        self.enabled = True\\n        self.mode = mode\\n        self.run_id = getattr(run, \\"id\\", run_id)\\n        self.run_name = getattr(run, \\"name\\", run_name)\\n        try:\\n            self.url = run.get_url() if hasattr(run, \\"get_url\\") else None\\n        except Exception:  # noqa: BLE001\\n            self.url = None\\n\\n    def metadata(self) -> dict[str, Any]:\\n        return {\\n            \\"wandb_enabled\\": self.enabled,\\n            \\"wandb_mode\\": self.mode,\\n            \\"wandb_project\\": self.project,\\n            \\"wandb_entity\\": self.entity,\\n            \\"wandb_run_id\\": self.run_id,\\n            \\"wandb_run_name\\": self.run_name,\\n            \\"wandb_url\\": self.url,\\n            \\"wandb_error\\": self.error,\\n        }\\n\\n    def log(self, payload: Mapping[str, Any], *, step: int | None = None) -> None:\\n        if self._run is None:\\n            return\\n        metrics = _flatten(payload)\\n        metrics.pop(\\"step\\", None)\\n        if metrics:\\n            self._run.log(metrics, step=step)\\n\\n    def update_summary(self, payload: Mapping[str, Any]) -> None:\\n        if self._run is None:\\n            return\\n        for key, value in _flatten(payload, allow_strings=True).items():\\n            self._run.summary[key] = value\\n\\n    def log_image(self, key: str, path: Path, *, step: int | None = None) -> None:\\n        if self._run is None:\\n            return\\n        try:\\n            import wandb\\n        except Exception:  # noqa: BLE001\\n            return\\n        payload = {key: wandb.Image(str(path))}\\n        self._run.log(payload, step=step)\\n\\n    def finish(self, *, status: str = \\"completed\\") -> None:\\n        if self._run is None:\\n            return\\n        exit_code = 0 if status == \\"completed\\" else 1\\n        try:\\n            self._run.finish(exit_code=exit_code)\\n        except TypeError:\\n            self._run.finish()\\n        except Exception:  # noqa: BLE001\\n            return\\n        finally:\\n            self._run = None\\n"}')
EXPECTED_SOURCE_HASHES = json.loads('{"src/__init__.py": "b28889a0f51eff06d10bcfbc1bf642430fe29a1a9dc772bf1ba6c2f88da7c7fa", "src/metrics/__init__.py": "9ac0d7f983b8105c3c751a0cdb8dd858121d57e368bb082158cb49ceb0401a06", "src/metrics/depth_metrics.py": "fc1076e0cd80d31c689de64f3893f160634996d086e3cc5451586c09a1045a53", "src/metrics/norms.py": "4ef01ea30496765d7475c390ec849a519be8b3247c7856a841da4d6c70f5aa23", "src/models/__init__.py": "f48959d4af9bf1aafba8fee92fc39f70538eda8b202b4d437d300fd0fb07a0c9", "src/models/attnres.py": "d940e68b2c08666311a94f7cca3429f93092f5085b4cf96c664455ef0d45b01c", "src/models/baseline.py": "9908c085e2defd7ccc44885a17a78409d4c39fecb997429ae631104f67f01a6c", "src/models/vision_attnres.py": "c9f08fb1479eae6a0771d11301aeb37e5bfd5c07d2787d116395390b2d1fad7d", "src/models/vlm_attnres.py": "5f3c50d22dfbd7bd74891d225ca51e1c527a0d5980c39409e7b11eecc4a46f09", "src/utils/__init__.py": "fc20d1087e4e454d837823136b667cf02c2014380d149edbce5ee1ccd8575081", "src/utils/config.py": "749ff89c8dc1066646d674af6a0225e3fe2c0fd11c10d290e3269716d48d7ded", "src/utils/runtime.py": "6afd07e8bbe334c7ca99c797526d3adab43d599e4b2105af3bf9d61a49da2197", "src/vlm/__init__.py": "bfb4b08533abf21063b2a73565f7f6c1246a639fb9900802285e6d703397e4b6", "src/vlm/ablation/__init__.py": "c023679c84e8b0f4506d410b88fc297fba3a240a3916187cd25b92c81cb7d266", "src/vlm/ablation/aggregate.py": "ee2f86eae95ab8218429f3583b26060dd2754dea7eb87e5804ef01916a520ea8", "src/vlm/ablation/checkpoint.py": "dadafefcb19b10488626dbabac4c9baa71dd024888fab68e3b61ca59f71bd5f2", "src/vlm/ablation/config.py": "1b699fadf9b26227dd9352ed90b7c12b6ad2f34a8cf33e21c8dab34ff6b39127", "src/vlm/ablation/correctness.py": "1a6a205ebc386a7ebf0cc5cd00d4e00f660dca29183546a7bd3ef47dc09639e3", "src/vlm/ablation/eval.py": "9d015824e39e618ae4c087fa1f17835e43d09dbc554d7d42a0f5640957ce6d6d", "src/vlm/ablation/init_sync.py": "02d73279b01776f2e7f9604a6a7119ca452c529273d6c22ef5b1ff3b72b96178", "src/vlm/ablation/io_utils.py": "6d3ca43111656c28d793f88fd3ec33912b587f25be2a3664c5ef56291465cd11", "src/vlm/ablation/manifest.py": "fdb2c8e1501e7e2c0f08cd06c36b2bbe72da2e7814257cd964f710a3079e5a70", "src/vlm/ablation/plots.py": "3a39ba42e5568ae1d8ec0eca50913f432b66de7de9f32347e7d7b6f90cec928d", "src/vlm/ablation/routing.py": "4787bb9ff35bea191ced1ad094af10631667de7f65e224fee7702abeb9a5b00b", "src/vlm/ablation/runner.py": "6e83b9db1b2d79acda9f759caf4fb81805de38ab920d0590262cd9caa08c2916", "src/vlm/ablation/source_hash.py": "ed17a27400266734fa2586f041f82ddd294d82633be58657d624544cb536c3f1", "src/vlm/ablation/train.py": "1c169fe5e6ac34892ef190300f29737ad137a56116b3f59bc1bdd12d8a70b133", "src/vlm/ablation/wandb_logger.py": "4f968e517e9306609dc8d32cbab7bcefd55deed172253308128b02d2f19c5f06", "src/vlm/synthetic_vqa.py": "ecc49473fdd34e42b59a7eec5051e6fe97f23172663367a1f698a5fda52bf3df"}')

SOURCE_ROOT = PROJECT_ROOT / "source"


def _sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


written = []
for relpath, text in SOURCE_FILES.items():
    path = SOURCE_ROOT / relpath
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    digest = _sha256_text(text)
    expected = EXPECTED_SOURCE_HASHES[relpath]
    if digest != expected:
        raise RuntimeError(f"Embedded source hash mismatch for {relpath}")
    written.append(relpath)

# Validate on-disk hashes match the notebook manifest.
mismatches = []
for relpath, expected in EXPECTED_SOURCE_HASHES.items():
    actual = _sha256_text((SOURCE_ROOT / relpath).read_text(encoding="utf-8"))
    if actual != expected:
        mismatches.append(relpath)
if mismatches:
    raise RuntimeError(f"Drive source diverged from notebook embed: {mismatches}")

if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

print(f"Wrote {len(written)} modules under {SOURCE_ROOT}")
print("Source hash validation: OK")


## CUDA enforcement


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. Select a Colab GPU runtime and rerun. "
        "Training will not fall back to CPU or MPS."
    )

props = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
major, minor = torch.cuda.get_device_capability(0)
selected_amp = "bfloat16" if (AMP_DTYPE == "auto" and major >= 8) else (
    "float16" if AMP_DTYPE == "auto" else AMP_DTYPE
)

print("GPU:", props.name)
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)
print("Total GPU memory bytes:", int(total_bytes))
print("Available GPU memory bytes:", int(free_bytes))
print("Selected AMP dtype:", selected_amp)


## Run experiment


In [ ]:
from src.vlm.ablation.config import AblationExperimentConfig
from src.vlm.ablation.runner import run_ablation_experiment

config = AblationExperimentConfig(
    run_mode=RUN_MODE,
    resume=RESUME,
    force_restart=FORCE_RESTART,
    run_primary_full_grid=RUN_PRIMARY_FULL_GRID,
    run_block_grid=RUN_BLOCK_GRID,
    seeds=list(SEEDS),
    primary_variants=list(PRIMARY_VARIANTS),
    block_variants=list(BLOCK_VARIANTS),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    evaluation_interval=EVALUATION_INTERVAL,
    max_epochs=MAX_EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    mixed_precision=MIXED_PRECISION,
    amp_dtype=AMP_DTYPE,
    vision_d_model=VISION_D_MODEL,
    vision_n_layers=VISION_N_LAYERS,
    vision_n_heads=VISION_N_HEADS,
    vision_d_ff=VISION_D_FF,
    decoder_d_model=DECODER_D_MODEL,
    decoder_n_layers=DECODER_N_LAYERS,
    decoder_n_heads=DECODER_N_HEADS,
    decoder_d_ff=DECODER_D_FF,
    num_blocks=NUM_BLOCKS,
    dropout=DROPOUT,
    project_root=str(PROJECT_ROOT),
    wandb_enabled=WANDB_ENABLED,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_mode=WANDB_MODE,
    wandb_resume=WANDB_RESUME,
)

summary = run_ablation_experiment(
    config,
    project_root=PROJECT_ROOT,
    source_root=SOURCE_ROOT,
)
summary


## Completion summary


In [ ]:
import json
from pathlib import Path

summary_path = PROJECT_ROOT / "summaries" / "experiment_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Mounted Drive root:", DRIVE_ROOT)
print("Experiment root:", PROJECT_ROOT)
print("GPU information:", json.dumps(summary.get("cuda", {}), indent=2))
print("Run mode:", summary.get("run_mode"))
print("W&B project:", summary.get("wandb_project"))
print("W&B entity:", summary.get("wandb_entity"))
print("W&B summary run:", summary.get("wandb_summary"))
print("Requested variants:", summary.get("requested_variants"))
print("Completed:", summary.get("completed"))
print("Resumed:", summary.get("resumed"))
print("Failed:", summary.get("failed"))
print("Best by variant:")
for variant, metrics in (summary.get("best_by_variant") or {}).items():
    print(
        f"  {variant}: val={metrics.get('validation_accuracy')} "
        f"test={metrics.get('test_accuracy')} "
        f"best_ckpt={metrics.get('checkpoint_best')}"
    )
print("Aggregate CSV paths:", summary.get("tables"))
print("Plot directory:", summary.get("plots_dir"))
print("All persistent artifacts under Google Drive:", summary.get("artifacts_under_drive"))
assert str(PROJECT_ROOT).startswith(str(DRIVE_ROOT)), "Project root escaped Drive"
print("Idempotent re-run safe: completed runs are skipped via completed.marker")
